# ARES-X · Rover autónomo de exploración planetaria

**Inteligencia Artificial Aplicada — Desafío integrador 2026**
Arquitectura híbrida: **red neuronal artificial + lógica difusa + algoritmo genético**,
arbitradas por un supervisor de seguridad.
Entorno: Python en **Google Colab gratuito con CPU**. Sin GPU, sin API de pago, sin claves privadas.

---

> ### Naturaleza del trabajo
> Simulación **ficticia** con parámetros astronómicos de referencia. No representa una misión
> real ni certifica condiciones operacionales de ingeniería espacial.
> Los valores de temperatura, presión, gravedad e irradiancia provienen de hojas de datos
> públicas de la NASA y están marcados `[NASA]`; **todos** los parámetros operacionales
> (consumo, desgaste, alcance, probabilidades de fallo) son hipótesis pedagógicas ajustables
> marcadas `[SUPUESTO]` y **nunca deben presentarse como especificaciones NASA**.
>
> El detector de biofirmas **sólo emite señales anómalas: nunca confirma vida**.
> Los eventos de "contacto alienígena" son narrativa explícitamente ficticia y sin valor científico.
> El oxígeno se trata como **composición ambiental**, no como consumible respiratorio:
> un rover no tripulado no respira.

---

## Cómo ejecutar este notebook

1. `Entorno de ejecución → Reiniciar y ejecutar todo` (unos 3–5 minutos en Colab gratuito).
2. La sección **12** abre el Centro de Control de Misión interactivo.
3. Las secciones **13–15** ejecutan las pruebas obligatorias, las pruebas de causalidad y
   el experimento con 20 semillas.

## Índice

| § | Contenido | Trazabilidad |
|---|---|---|
| 0 | Dependencias y entorno | RF-16 |
| 1 | Configuración y perfiles planetarios | RF-01, CA-07 |
| 2 | Mundo, mapas y mapa de creencias | RF-02 |
| 3 | Sensores, ruido y fallos | RF-03, RF-13 |
| 4 | Percepción neuronal (RNA) | RF-04, CA-06 |
| 5 | Controlador de lógica difusa | RF-05, RF-06, CA-05 |
| 6 | Planificador genético | RF-07, CA-04 |
| 7 | Supervisor de seguridad y bitácora | RF-08, RF-12 |
| 8 | Ciclo de misión y máquina de estados | RF-03, RF-13 |
| 9 | Experimentos, ablaciones y causalidad | RF-14, RF-15, CA-08 |
| 10 | Centro de Control (paneles A–E) | RF-09, RF-10, RF-11 |
| 11 | Laboratorios de inspección | RF-11 |
| 12 | **Interfaz interactiva** | RF-09 |
| 13 | Pruebas obligatorias (16 + 8) | §12, §13 del enunciado |
| 14 | Pruebas de causalidad | Condición de auditoría §4 |
| 15 | Experimento con 20 semillas | RF-14, RF-15 |
| 16 | Exportación y conclusiones | RF-12 |

## 0 · Dependencias y verificación del entorno

Sólo se usan paquetes gratuitos ya presentes en Colab: `numpy`, `pandas`, `matplotlib`,
`scikit-learn` e `ipywidgets`. `scikit-fuzzy` **no** es necesario: el motor difuso
(membresías, inferencia Mamdani min–max y desfusificación por centroide) está implementado
desde cero en la sección 5, precisamente para que cada etapa sea auditable.

In [ ]:
import sys, subprocess, importlib

REQUERIDOS = ["numpy", "pandas", "matplotlib", "sklearn", "ipywidgets"]
faltan = [p for p in REQUERIDOS if importlib.util.find_spec(p) is None]
if faltan:
    print("Instalando:", faltan)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *["scikit-learn" if p == "sklearn" else p for p in faltan]],
                   check=False)

import numpy, pandas, matplotlib, sklearn, ipywidgets
print(f"Python       {sys.version.split()[0]}")
for m in (numpy, pandas, matplotlib, sklearn, ipywidgets):
    print(f"{m.__name__:<12} {m.__version__}")

# Widgets interactivos en Colab
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
    print("\nGestor de widgets de Colab habilitado.")
except Exception:
    print("\nEntorno no-Colab: se usa el gestor de widgets estándar de Jupyter.")

import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.22,
                     "font.size": 9, "figure.autolayout": False})
import warnings; warnings.filterwarnings("ignore")
print("Entorno listo.")

## 1 · Configuración, perfiles planetarios y utilidades base

Aquí se define **qué es verdad astronómica y qué es supuesto de simulación**. Cada perfil
planetario guarda valores de referencia NASA (temperatura, presión, gravedad, irradiancia,
composición atmosférica) junto a parámetros inventados pero explícitos (límites térmicos del
hardware, factor de desgaste, probabilidad de eventos).

**Fidelidad planetaria (RF-01, CA-07).** `PlanetProfile.compatible()` es el mecanismo que
impide simular un rover rodante en Júpiter, Saturno, Urano o Neptuno: no existe una interfaz
sólido-atmósfera estable donde apoyar ruedas, de modo que la configuración se **rechaza con
una explicación científica** y se propone el modo sonda atmosférica.

`Config.validar()` rechaza además configuraciones imposibles o con NaN con un mensaje legible,
y `rng_de()` deriva flujos aleatorios independientes —mundo, sensores, IA, eventos— de una
única semilla. Usa CRC32 y no `hash()`, porque el hash de cadenas de Python está aleatorizado
por proceso y arruinaría la reproducibilidad entre sesiones de Colab (RF-12).

In [ ]:
"""
ARES-X · Módulo 1: Configuración, perfiles planetarios y utilidades base.

Naturaleza: simulación ficticia con parámetros astronómicos de referencia.
No representa una misión real ni certifica condiciones de ingeniería espacial.

Convención de trazabilidad de datos:
  [NASA]     valor de referencia tomado de hojas de datos públicas de NASA.
  [SUPUESTO] hipótesis pedagógica ajustable, inventada para que la simulación
             tenga dinámica; NUNCA debe presentarse como especificación NASA.
"""

from __future__ import annotations

import json
import math
import zlib
import time
import warnings
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional, Tuple

import numpy as np

warnings.filterwarnings("ignore", category=UserWarning)

VERSION = "1.0.0"
FECHA_BUILD = "2026-11-17"

# ----------------------------------------------------------------------------
# Registro central de supuestos de simulación (RF-16 / informe de límites)
# ----------------------------------------------------------------------------
SUPUESTOS_SIMULACION: Dict[str, str] = {
    "celda_m": "Cada celda de la cuadrícula equivale a 5 m simulados.",
    "paso_temporal_s": "Un paso de simulación equivale a 60 s simulados (1 'tick').",
    "consumo_base_pct": "Avanzar una celda consume 0.45 % de batería a velocidad 1.0.",
    "consumo_pendiente": "La pendiente añade consumo proporcional: +0.03 %/grado.",
    "consumo_idle_pct": "Mantener electrónica encendida cuesta 0.05 % por paso.",
    "recarga_pct": "Un refugio/estación recarga 6 % por paso si hay energía disponible.",
    "solar_pct_max": "La captación solar máxima es 1.2 % por paso a irradiancia terrestre.",
    "limite_termico": "Los límites térmicos del hardware son hipótesis pedagógicas.",
    "desgaste": "El desgaste mecánico acumulado es una escala 0-100 inventada.",
    "radiacion": "La dosis en mSv/h es una escala ilustrativa, no un modelo físico.",
    "prob_fallo_sensor": "Probabilidad base de fallo de sensor por paso: 0.8 %.",
    "ruido_sensores": "Las desviaciones estándar de ruido son valores didácticos.",
    "biofirma": "El detector sólo emite 'señal anómala'; nunca confirma vida.",
    "alienigena": "Evento narrativo opcional, explícitamente ficticio.",
    "deriva_odometria": "La odometría acumula un error de 0.4 % de la distancia.",
    "masa_instrumentos": "Masas y consumos de instrumentos son ficticios (mochila).",
}

FUENTES_NASA: List[str] = [
    "https://science.nasa.gov/resource/solar-system-temperatures/",
    "https://science.nasa.gov/venus/venus-facts/",
    "https://science.nasa.gov/solar-system/solar-system-facts/",
    "https://nssdc.gsfc.nasa.gov/planetary/factsheet/marsfact.html",
]

# Códigos de motivo del supervisor (RF-08: toda decisión tiene motivo reproducible)
CODIGOS_MOTIVO: Dict[str, str] = {
    "OK_AVANCE": "Acción permitida: avance dentro de límites.",
    "OK_RECARGA": "Acción permitida: recarga en refugio/estación.",
    "OK_MUESTREO": "Acción permitida: toma de muestra científica.",
    "OK_ESPERA": "Acción permitida: espera controlada.",
    "VETO_BARRANCO": "Veto duro: la celda destino es un precipicio confirmado.",
    "VETO_RNA_BARRANCO": "Veto preventivo: la RNA predice barranco con alta confianza.",
    "VETO_COLISION": "Veto duro: la celda destino es un obstáculo infranqueable.",
    "VETO_FUERA_MAPA": "Veto duro: la celda destino está fuera del mapa.",
    "VETO_TERMICO": "Veto duro: temperatura interna fuera de tolerancia del hardware.",
    "VETO_PENDIENTE": "Veto duro: pendiente superior al máximo de vuelco.",
    "VETO_ENERGIA": "Veto duro: la batería no permite alcanzar zona segura.",
    "VETO_SENSOR": "Veto duro: sensor crítico inválido (NaN/saturado).",
    "VETO_DESGASTE": "Veto duro: desgaste mecánico sobre el límite operacional.",
    "VETO_RUTA_INVIABLE": "La ruta propuesta por el AG cruza una restricción dura.",
    "MODO_SEGURO": "Degradación segura: subsistema aislado, rover detenido.",
    "REPLANIFICAR": "Se solicita nueva planificación al AG.",
    "SIN_RUTA": "No existe ruta factible hacia el objetivo.",
    "REFUGIO_INALCANZABLE": "El refugio no es alcanzable con la energía disponible.",
    "FIN_EXITO": "Objetivos cumplidos y retorno seguro.",
    "FIN_FRACASO": "Misión terminada por daño, colisión o energía agotada.",
}


# ----------------------------------------------------------------------------
# Perfiles planetarios
# ----------------------------------------------------------------------------
@dataclass
class PlanetProfile:
    """Perfil de un cuerpo del sistema solar.

    Los campos marcados [NASA] son aproximaciones de referencia; los demás son
    supuestos de simulación ajustables (ver SUPUESTOS_SIMULACION).
    """

    clave: str
    nombre: str
    tipo: str                      # 'rocoso' | 'gaseoso' | 'helado'
    permite_rover: bool            # [NASA] ¿hay superficie sólida de rodadura?
    temp_media_C: float            # [NASA]
    temp_min_C: float              # [NASA]
    temp_max_C: float              # [NASA]
    presion_bar: float             # [NASA]
    gravedad_ms2: float            # [NASA]
    irradiancia_rel: float         # [NASA] relativa a la Tierra (=1.0)
    atmosfera: Dict[str, float]    # [NASA] % volumen aproximado
    peligros: List[str]
    # --- Supuestos de simulación -------------------------------------------
    limite_termico_C: Tuple[float, float] = (-120.0, 85.0)   # [SUPUESTO]
    factor_desgaste: float = 1.0                             # [SUPUESTO]
    radiacion_base_mSv_h: float = 0.02                       # [SUPUESTO]
    prob_evento_base: float = 0.06                           # [SUPUESTO]
    rugosidad_terreno: float = 1.0                           # [SUPUESTO]
    nota: str = ""

    @property
    def modo_requerido(self) -> str:
        return "rover" if self.permite_rover else "sonda"

    def compatible(self, modo: str) -> Tuple[bool, str]:
        """RF-01: valida compatibilidad rover/sonda. CA-07."""
        if modo == "rover" and not self.permite_rover:
            return (
                False,
                f"{self.nombre} es un {self.tipo} sin una superficie sólida de "
                f"rodadura accesible: no existe 'suelo' donde apoyar ruedas. "
                f"La presión crece de forma continua con la profundidad y no hay "
                f"una interfaz sólido-atmósfera estable. Use el modo 'sonda' "
                f"(vehículo atmosférico/aerostático virtual).",
            )
        if modo == "sonda" and self.permite_rover and self.clave != "tierra":
            return (
                True,
                f"Advertencia: {self.nombre} admite rover; el modo sonda se "
                f"simula como vehículo aerostático sobre la superficie.",
            )
        return (True, "Configuración compatible.")


def _perfiles() -> Dict[str, PlanetProfile]:
    P = {}
    P["mercurio"] = PlanetProfile(
        clave="mercurio", nombre="Mercurio", tipo="rocoso", permite_rover=True,
        temp_media_C=167.0, temp_min_C=-180.0, temp_max_C=430.0,
        presion_bar=1e-14, gravedad_ms2=3.7, irradiancia_rel=6.67,
        atmosfera={"O2": 0.0, "CO2": 0.0, "SO2": 0.0},
        peligros=["radiacion_solar", "sobrecalentamiento", "frio_nocturno", "crateres"],
        limite_termico_C=(-150.0, 120.0), factor_desgaste=1.4,
        radiacion_base_mSv_h=0.45, prob_evento_base=0.07, rugosidad_terreno=1.25,
        nota="Vacío casi total; contraste térmico día/noche extremo. Rover hipotético.",
    )
    P["venus"] = PlanetProfile(
        clave="venus", nombre="Venus", tipo="rocoso", permite_rover=True,
        temp_media_C=464.0, temp_min_C=440.0, temp_max_C=480.0,
        presion_bar=92.0, gravedad_ms2=8.9, irradiancia_rel=1.91,
        atmosfera={"O2": 0.0, "CO2": 96.5, "SO2": 150e-4},
        peligros=["corrosion_acida", "temperatura", "presion"],
        limite_termico_C=(-20.0, 500.0), factor_desgaste=3.2,
        radiacion_base_mSv_h=0.05, prob_evento_base=0.12, rugosidad_terreno=1.0,
        nota="Rover especial hipotético y misión breve. 'Lluvia ácida en suelo' NO se presupone.",
    )
    P["tierra"] = PlanetProfile(
        clave="tierra", nombre="Tierra", tipo="rocoso", permite_rover=True,
        temp_media_C=15.0, temp_min_C=-40.0, temp_max_C=45.0,
        presion_bar=1.0, gravedad_ms2=9.8, irradiancia_rel=1.0,
        atmosfera={"O2": 20.9, "CO2": 0.04, "SO2": 0.0},
        peligros=["lluvia", "sismo", "deslizamiento"],
        limite_termico_C=(-60.0, 70.0), factor_desgaste=0.8,
        radiacion_base_mSv_h=0.0001, prob_evento_base=0.05, rugosidad_terreno=0.9,
        nota="Banco de calibración del simulador.",
    )
    P["marte"] = PlanetProfile(
        clave="marte", nombre="Marte", tipo="rocoso", permite_rover=True,
        temp_media_C=-65.0, temp_min_C=-125.0, temp_max_C=20.0,
        presion_bar=0.00636, gravedad_ms2=3.7, irradiancia_rel=0.43,
        atmosfera={"O2": 0.13, "CO2": 95.3, "SO2": 0.0},
        peligros=["barrancos", "tormenta_polvo", "hielo", "noche", "marsquake"],
        limite_termico_C=(-110.0, 60.0), factor_desgaste=1.0,
        radiacion_base_mSv_h=0.025, prob_evento_base=0.08, rugosidad_terreno=1.1,
        nota="Escenario principal del caso ARES-X.",
    )
    P["jupiter"] = PlanetProfile(
        clave="jupiter", nombre="Júpiter", tipo="gaseoso", permite_rover=False,
        temp_media_C=-110.0, temp_min_C=-145.0, temp_max_C=20.0,
        presion_bar=1.0, gravedad_ms2=24.8, irradiancia_rel=0.037,
        atmosfera={"O2": 0.0, "CO2": 0.0, "SO2": 0.0},
        peligros=["turbulencia", "presion_creciente", "radiacion"],
        limite_termico_C=(-160.0, 120.0), factor_desgaste=1.8,
        radiacion_base_mSv_h=1.2, prob_evento_base=0.16, rugosidad_terreno=0.0,
        nota="Gigante gaseoso sin suelo accesible estable: sólo sonda atmosférica.",
    )
    P["saturno"] = PlanetProfile(
        clave="saturno", nombre="Saturno", tipo="gaseoso", permite_rover=False,
        temp_media_C=-140.0, temp_min_C=-180.0, temp_max_C=-20.0,
        presion_bar=1.0, gravedad_ms2=10.4, irradiancia_rel=0.011,
        atmosfera={"O2": 0.0, "CO2": 0.0, "SO2": 0.0},
        peligros=["vientos", "presion", "latencia_comunicaciones"],
        limite_termico_C=(-190.0, 90.0), factor_desgaste=1.6,
        radiacion_base_mSv_h=0.35, prob_evento_base=0.15, rugosidad_terreno=0.0,
        nota="Gigante gaseoso sin suelo accesible estable: sólo sonda atmosférica.",
    )
    P["urano"] = PlanetProfile(
        clave="urano", nombre="Urano", tipo="helado", permite_rover=False,
        temp_media_C=-195.0, temp_min_C=-224.0, temp_max_C=-150.0,
        presion_bar=1.0, gravedad_ms2=8.7, irradiancia_rel=0.0027,
        atmosfera={"O2": 0.0, "CO2": 0.0, "SO2": 0.0},
        peligros=["frio_extremo", "energia_escasa", "vientos"],
        limite_termico_C=(-230.0, 60.0), factor_desgaste=1.7,
        radiacion_base_mSv_h=0.08, prob_evento_base=0.14, rugosidad_terreno=0.0,
        nota="Gigante helado sin suelo sólido de rodadura accesible.",
    )
    P["neptuno"] = PlanetProfile(
        clave="neptuno", nombre="Neptuno", tipo="helado", permite_rover=False,
        temp_media_C=-200.0, temp_min_C=-230.0, temp_max_C=-160.0,
        presion_bar=1.0, gravedad_ms2=11.0, irradiancia_rel=0.0011,
        atmosfera={"O2": 0.0, "CO2": 0.0, "SO2": 0.0},
        peligros=["frio_extremo", "vientos_supersonicos", "latencia_enlace"],
        limite_termico_C=(-240.0, 60.0), factor_desgaste=1.9,
        radiacion_base_mSv_h=0.06, prob_evento_base=0.17, rugosidad_terreno=0.0,
        nota="Gigante helado sin suelo sólido de rodadura accesible.",
    )
    return P


PERFILES: Dict[str, PlanetProfile] = _perfiles()


# ----------------------------------------------------------------------------
# Configuración de misión
# ----------------------------------------------------------------------------
@dataclass
class Config:
    """RF-01: planeta, modo, mapa, dificultad y semilla."""

    planeta: str = "marte"
    modo: str = "rover"                 # 'rover' | 'sonda'
    filas: int = 22
    columnas: int = 22
    dificultad: str = "media"           # 'facil' | 'media' | 'dificil'
    semilla: int = 42
    bateria_inicial: float = 55.0
    n_sitios_ciencia: int = 6
    n_refugios: int = 2
    max_pasos: int = 220
    duracion_paso_s: float = 60.0       # [SUPUESTO]
    hora_inicial: float = 8.0           # hora solar local (0-24)
    horas_por_paso: float = 0.12        # [SUPUESTO] ~7 min solares por paso
    # --- IA ---------------------------------------------------------------
    usar_rna: bool = True
    usar_difuso: bool = True
    usar_ag: bool = True
    ag_poblacion: int = 40
    ag_generaciones: int = 40
    ag_prob_cruce: float = 0.85
    ag_prob_mutacion: float = 0.20
    ag_elitismo: int = 2
    ag_torneo: int = 3
    ag_waypoints: int = 3
    ag_pesos: Dict[str, float] = field(default_factory=lambda: {
        "ciencia": 0.40, "distancia": 0.15, "energia": 0.15,
        "riesgo": 0.22, "tiempo": 0.08,
    })
    # --- Seguridad / física -----------------------------------------------
    pendiente_max_deg: float = 28.0     # [SUPUESTO] ángulo de vuelco
    desgaste_max: float = 100.0         # [SUPUESTO]
    umbral_rna_barranco: float = 0.55   # [SUPUESTO] veto preventivo
    umbral_riesgo_alerta: float = 0.55  # [SUPUESTO]
    reserva_energia_pct: float = 6.0    # [SUPUESTO] margen para refugio
    # --- Ciencia -----------------------------------------------------------
    prioridad_ciencia: float = 1.0      # multiplicador de valor científico
    masa_max_kg: float = 12.0           # [SUPUESTO] mochila de instrumentos
    potencia_max_W: float = 55.0        # [SUPUESTO] mochila de instrumentos
    # --- Eventos -----------------------------------------------------------
    eventos_aleatorios: bool = True
    prob_fallo_sensor: float = 0.008    # [SUPUESTO]
    # Baseline de control: planificar por distancia mínima IGNORANDO el riesgo
    # estimado (sólo se usa en los experimentos comparativos, RF-14).
    baseline_ruta_corta: bool = False

    def validar(self) -> Tuple[bool, List[str]]:
        """Rechaza configuraciones imposibles o NaN con mensaje legible."""
        errores: List[str] = []
        if self.planeta not in PERFILES:
            errores.append(f"Planeta desconocido: '{self.planeta}'.")
        if self.modo not in ("rover", "sonda"):
            errores.append(f"Modo desconocido: '{self.modo}'.")
        numericos = {
            "filas": self.filas, "columnas": self.columnas,
            "bateria_inicial": self.bateria_inicial, "semilla": self.semilla,
            "max_pasos": self.max_pasos, "n_sitios_ciencia": self.n_sitios_ciencia,
            "masa_max_kg": self.masa_max_kg, "potencia_max_W": self.potencia_max_W,
        }
        for k, v in numericos.items():
            if v is None or (isinstance(v, float) and not math.isfinite(v)):
                errores.append(f"Parámetro '{k}' no es un número finito (NaN/inf).")
        if self.filas < 8 or self.columnas < 8:
            errores.append("El mapa mínimo es de 8x8 celdas.")
        if self.filas > 60 or self.columnas > 60:
            errores.append("El mapa máximo soportado es de 60x60 celdas.")
        if not (0.0 <= self.bateria_inicial <= 100.0):
            errores.append("La batería inicial debe estar entre 0 y 100 %.")
        if self.n_sitios_ciencia < 0:
            errores.append("El número de sitios científicos no puede ser negativo.")
        if self.ag_poblacion < 4 or self.ag_generaciones < 1:
            errores.append("El AG requiere población >= 4 y generaciones >= 1.")
        if self.planeta in PERFILES:
            ok, msg = PERFILES[self.planeta].compatible(self.modo)
            if not ok:
                errores.append(msg)
        return (len(errores) == 0, errores)

    @property
    def perfil(self) -> PlanetProfile:
        return PERFILES[self.planeta]

    def to_json(self) -> str:
        d = asdict(self)
        d["_version"] = VERSION
        d["_timestamp"] = time.strftime("%Y-%m-%dT%H:%M:%S")
        return json.dumps(d, indent=2, ensure_ascii=False)

    @staticmethod
    def from_dict(d: dict) -> "Config":
        campos = {f for f in Config.__dataclass_fields__}
        return Config(**{k: v for k, v in d.items() if k in campos})


# ----------------------------------------------------------------------------
# Utilidades numéricas
# ----------------------------------------------------------------------------
def hash_estable(texto: str) -> int:
    """Hash determinista ENTRE PROCESOS.

    `hash()` de Python está aleatorizado por proceso (PYTHONHASHSEED), de modo
    que usarlo aquí haría que la misma semilla produjera trazas distintas en
    cada sesión de Colab. CRC32 es estable y suficiente para derivar flujos.
    """
    return zlib.crc32(texto.encode("utf-8")) & 0xFFFFFFFF


def rng_de(semilla: int, etiqueta: str = "") -> np.random.Generator:
    """Generador determinista derivado de (semilla, etiqueta).

    Permite que mundo, sensores, IA y eventos tengan flujos independientes pero
    reproducibles a partir de una única semilla (RF-12, condición de borde
    'se repite semilla y configuración').
    """
    mezcla = int(semilla) * 1_000_003 + (hash_estable(etiqueta) % 100_003
                                         if etiqueta else 0)
    return np.random.default_rng(mezcla % (2**32 - 1))


def clamp(x: float, lo: float, hi: float) -> float:
    if x is None or (isinstance(x, float) and not math.isfinite(x)):
        return lo
    return max(lo, min(hi, x))


def norm01(x: float, lo: float, hi: float) -> float:
    """Normaliza a [0,1] de forma segura ante lo==hi o NaN."""
    if hi - lo <= 1e-12:
        return 0.0
    return clamp((x - lo) / (hi - lo), 0.0, 1.0)


def es_valido(x) -> bool:
    """True si el valor es un número finito (detecta NaN, inf, None)."""
    try:
        return x is not None and math.isfinite(float(x))
    except (TypeError, ValueError):
        return False


def distancia_cheb(a: Tuple[int, int], b: Tuple[int, int]) -> int:
    """Distancia de Chebyshev (movimiento en 8 direcciones)."""
    return max(abs(a[0] - b[0]), abs(a[1] - b[1]))


def distancia_euclid(a: Tuple[int, int], b: Tuple[int, int]) -> float:
    return math.hypot(a[0] - b[0], a[1] - b[1])


DIRECCIONES_8: List[Tuple[int, int]] = [
    (-1, 0), (-1, 1), (0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1)
]
NOMBRES_DIR = ["N", "NE", "E", "SE", "S", "SO", "O", "NO"]

## 2 · Mundo, generación de mapas y mapa de creencias

Dos clases con una separación deliberada y central para todo el trabajo:

* **`World`** es la **verdad del terreno**: relieve fractal, obstáculos, barrancos (trazados
  como cañones contiguos y como celdas de gradiente extremo), refugios, sitios científicos y
  estado ambiental dinámico (hora solar, polvo, sismo, viento).
* **`BeliefMap`** es **lo que el rover cree**. El rover nunca consulta `World` para decidir:
  observa con sensores ruidosos y acumula creencias probabilísticas.

El umbral de bloqueo duro de una creencia es alto (0,85) y sólo las celdas **confirmadas** por
el sensor determinista de borde cierran el paso. Esto no es un detalle: con un umbral bajo, un
5 % de falsos positivos de la RNA fragmenta el mapa navegable y el rover se encierra solo.
La percepción influye sobre todo a través del **coste** de riesgo que evalúa el AG.

El punto de aterrizaje se sitúa *cerca* de un refugio pero no sobre él, para que los escenarios
de energía crítica y de "retorno seguro" no sean triviales.

In [ ]:
"""
ARES-X · Módulo 2: Mundo, generación de mapas y mapa de creencias.

Clases:
  World       - verdad del terreno (ground truth) del escenario.
  BeliefMap   - lo que el rover CREE del mundo (RF: "el rover no es omnisciente").

Clases de celda (verdad del terreno, usadas como etiqueta de la RNA):
  0 = libre        transitable
  1 = obstaculo    roca/bloque infranqueable
  2 = barranco     precipicio: celda NO transitable y peligro crítico
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Set, Tuple

import numpy as np


CELDA_M = 5.0          # [SUPUESTO] lado de celda en metros simulados
LIBRE, OBSTACULO, BARRANCO = 0, 1, 2
NOMBRE_CLASE = {LIBRE: "libre", OBSTACULO: "obstaculo", BARRANCO: "barranco"}

DIFICULTAD = {
    "facil":   dict(dens_obst=0.06, dens_barr=0.03, rugosidad=0.7, prob_evento=0.6),
    "media":   dict(dens_obst=0.11, dens_barr=0.06, rugosidad=1.0, prob_evento=1.0),
    "dificil": dict(dens_obst=0.17, dens_barr=0.10, rugosidad=1.4, prob_evento=1.5),
}


# ----------------------------------------------------------------------------
# Campo de elevación (ruido suavizado, sin dependencias externas)
# ----------------------------------------------------------------------------
def _suavizar(a: np.ndarray, pasadas: int = 2) -> np.ndarray:
    """Filtro de media 3x3 con bordes replicados (sustituye a scipy)."""
    out = a.astype(float).copy()
    for _ in range(pasadas):
        p = np.pad(out, 1, mode="edge")
        acc = np.zeros_like(out)
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                acc += p[1 + dy: 1 + dy + out.shape[0], 1 + dx: 1 + dx + out.shape[1]]
        out = acc / 9.0
    return out


def _campo_fractal(filas: int, cols: int, rng: np.random.Generator,
                   octavas: int = 4, rugosidad: float = 1.0) -> np.ndarray:
    """Suma de octavas de ruido suavizado -> relieve verosímil y reproducible."""
    campo = np.zeros((filas, cols), dtype=float)
    amplitud = 1.0
    for o in range(octavas):
        escala = max(2, int(min(filas, cols) / (2 ** o)))
        bajo = rng.normal(0.0, 1.0, size=(max(2, filas // escala + 2),
                                          max(2, cols // escala + 2)))
        # interpolación por repetición + suavizado (bilineal aproximada)
        rep = np.kron(bajo, np.ones((escala, escala)))[:filas, :cols]
        if rep.shape != (filas, cols):
            rep = np.pad(rep, ((0, filas - rep.shape[0]), (0, cols - rep.shape[1])),
                         mode="edge")
        campo += amplitud * _suavizar(rep, pasadas=1)
        amplitud *= 0.55
    campo = _suavizar(campo, pasadas=1)
    campo -= campo.min()
    if campo.max() > 1e-9:
        campo /= campo.max()
    campo = _suavizar(campo, pasadas=1)
    return campo * (11.0 * rugosidad)   # [SUPUESTO] relieve 0-11 m simulados


# ----------------------------------------------------------------------------
# Sitios científicos e instrumentos
# ----------------------------------------------------------------------------
@dataclass
class SitioCiencia:
    id: int
    pos: Tuple[int, int]
    valor: float            # valor científico 0-1
    masa_kg: float          # masa de la muestra [SUPUESTO]
    requiere: str           # instrumento requerido
    recolectado: bool = False
    prioritario: bool = False


@dataclass
class Instrumento:
    """Sub-problema tipo MOCHILA: seleccionar instrumentos bajo masa y potencia."""
    id: int
    nombre: str
    masa_kg: float
    potencia_W: float
    valor: float
    habilita: str


INSTRUMENTOS_BASE: List[Instrumento] = [
    Instrumento(0, "Espectrómetro APXS", 2.4, 8.0, 0.85, "mineral"),
    Instrumento(1, "Cámara multiespectral", 1.6, 6.5, 0.70, "imagen"),
    Instrumento(2, "Taladro de núcleo", 4.8, 22.0, 0.95, "subsuelo"),
    Instrumento(3, "Detector de biofirmas", 3.1, 14.0, 0.90, "biofirma"),
    Instrumento(4, "Radar de penetración", 3.9, 17.0, 0.65, "subsuelo"),
    Instrumento(5, "Estación meteorológica", 1.1, 3.5, 0.45, "ambiente"),
    Instrumento(6, "Magnetómetro", 0.9, 2.5, 0.40, "ambiente"),
    Instrumento(7, "Brazo de muestreo", 5.2, 19.0, 0.80, "mineral"),
]

TIPOS_MUESTRA = ["mineral", "imagen", "subsuelo", "biofirma", "ambiente"]


# ----------------------------------------------------------------------------
# World
# ----------------------------------------------------------------------------
class World:
    """Verdad del terreno. El rover NUNCA la lee directamente: la observa
    mediante SensorSuite (con ruido) y la estima con BeliefMap."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.perfil: PlanetProfile = cfg.perfil
        self.filas, self.cols = cfg.filas, cfg.columnas
        self.rng = rng_de(cfg.semilla, "mundo|" + cfg.planeta + "|" + cfg.modo)
        self.dif = DIFICULTAD.get(cfg.dificultad, DIFICULTAD["media"])
        self.modo = cfg.modo

        self.elev = np.zeros((self.filas, self.cols))
        self.clase = np.full((self.filas, self.cols), LIBRE, dtype=int)
        self.textura = np.zeros((self.filas, self.cols))
        self.reflectancia = np.zeros((self.filas, self.cols))
        self.sitios: List[SitioCiencia] = []
        self.refugios: List[Tuple[int, int]] = []
        self.zonas_riesgo: List[dict] = []     # eventos: meteorito, sismo...
        self.inicio: Tuple[int, int] = (0, 0)

        # Estado ambiental dinámico
        self.hora = float(cfg.hora_inicial)
        self.visibilidad = 1.0
        self.tormenta = 0.0          # 0-1 intensidad de tormenta de polvo
        self.sismo = 0.0             # 0-1 intensidad instantánea
        self.viento = 0.0            # 0-1 (modo sonda)
        self.anomalia_bio: Optional[Tuple[int, int]] = None
        self.alienigena: Optional[Tuple[int, int]] = None

        self._generar()

    # -- generación ---------------------------------------------------------
    def _generar(self) -> None:
        rug = self.dif["rugosidad"] * max(0.15, self.perfil.rugosidad_terreno)
        if self.modo == "sonda":
            self._generar_sonda(rug)
        else:
            self._generar_superficie(rug)
        self._colocar_refugios()
        self._colocar_ciencia()
        self._colocar_inicio()
        self.gradiente = self._calc_gradiente()

    def _generar_superficie(self, rug: float) -> None:
        self.elev = _campo_fractal(self.filas, self.cols, self.rng, 4, rug)

        # Barrancos: cañones trazados + celdas con gradiente extremo
        grad = self._calc_gradiente(self.elev)
        umbral = np.quantile(grad, 1.0 - self.dif["dens_barr"])
        self.clase[grad >= max(umbral, 4.0)] = BARRANCO
        n_canones = max(2, int(self.dif["dens_barr"] * 42))
        for _ in range(n_canones):
            self._trazar_canon()

        # Obstáculos: cúmulos de rocas
        n_rocas = int(self.filas * self.cols * self.dif["dens_obst"])
        colocadas = 0
        intentos = 0
        while colocadas < n_rocas and intentos < n_rocas * 25:
            intentos += 1
            f = int(self.rng.integers(0, self.filas))
            c = int(self.rng.integers(0, self.cols))
            if self.clase[f, c] != LIBRE:
                continue
            tam = int(self.rng.integers(1, 3))
            for df in range(-tam // 2, tam // 2 + 1):
                for dc in range(-tam // 2, tam // 2 + 1):
                    ff, cc = f + df, c + dc
                    if self.dentro((ff, cc)) and self.clase[ff, cc] == LIBRE:
                        self.clase[ff, cc] = OBSTACULO
                        colocadas += 1
            self.elev[f, c] += self.rng.uniform(0.5, 2.5)

        # Textura y reflectancia (alimentan la cámara multiespectral -> RNA)
        self.textura = _suavizar(self.rng.random((self.filas, self.cols)), 1)
        self.reflectancia = _suavizar(self.rng.random((self.filas, self.cols)), 2)
        self.textura[self.clase == OBSTACULO] = np.clip(
            self.textura[self.clase == OBSTACULO] + 0.35, 0, 1)
        self.reflectancia[self.clase == BARRANCO] = np.clip(
            self.reflectancia[self.clase == BARRANCO] - 0.30, 0, 1)

    def _trazar_canon(self) -> None:
        """Traza una grieta continua (barranco) sobre el mapa."""
        f = int(self.rng.integers(0, self.filas))
        c = int(self.rng.integers(0, self.cols))
        largo = int(self.rng.integers(self.filas // 3, self.filas))
        d = DIRECCIONES_8[int(self.rng.integers(0, 8))]
        for _ in range(largo):
            if not self.dentro((f, c)):
                break
            self.clase[f, c] = BARRANCO
            self.elev[f, c] -= 9.0
            if self.rng.random() < 0.25:
                d = DIRECCIONES_8[int(self.rng.integers(0, 8))]
            f, c = f + d[0], c + d[1]

    def _generar_sonda(self, rug: float) -> None:
        """Modo sonda atmosférica: corte vertical de la atmósfera.

        Filas = capas de altitud (0 = alta, N-1 = profunda / presión crítica).
        'obstaculo' = celda de turbulencia severa; 'barranco' = corriente
        descendente de presión crítica (destruye la sonda). No hay ruedas.
        """
        base = np.linspace(0.0, 18.0, self.filas).reshape(-1, 1)
        self.elev = base + _campo_fractal(self.filas, self.cols, self.rng, 3, 0.6)
        prof = np.linspace(0.2, 1.0, self.filas).reshape(-1, 1)
        campo = self.rng.random((self.filas, self.cols)) * prof
        self.clase[campo > (1.0 - self.dif["dens_obst"] * 3.2)] = OBSTACULO
        self.clase[campo > (1.0 - self.dif["dens_barr"] * 2.6)] = BARRANCO
        self.textura = _suavizar(self.rng.random((self.filas, self.cols)), 1)
        self.reflectancia = _suavizar(self.rng.random((self.filas, self.cols)), 2)
        self.viento = 0.35

    def _colocar_refugios(self) -> None:
        self.refugios = []
        objetivo = max(1, self.cfg.n_refugios)
        intentos = 0
        while len(self.refugios) < objetivo and intentos < 4000:
            intentos += 1
            if self.modo == "sonda":
                f = int(self.rng.integers(self.filas - 4, self.filas))
            else:
                f = int(self.rng.integers(0, self.filas))
            c = int(self.rng.integers(0, self.cols))
            if self.clase[f, c] == LIBRE and (f, c) not in self.refugios:
                if all(distancia_cheb((f, c), r) > 3 for r in self.refugios):
                    self.refugios.append((f, c))
        if not self.refugios:                    # degradación: libera una celda
            f, c = self.filas // 2, self.cols // 2
            self.clase[f, c] = LIBRE
            self.refugios.append((f, c))

    def _colocar_ciencia(self) -> None:
        self.sitios = []
        libres = [(f, c) for f in range(self.filas) for c in range(self.cols)
                  if self.clase[f, c] == LIBRE and (f, c) not in self.refugios]
        self.rng.shuffle(libres)
        n = min(self.cfg.n_sitios_ciencia, len(libres))
        for i in range(n):
            pos = libres[i]
            self.sitios.append(SitioCiencia(
                id=i, pos=(int(pos[0]), int(pos[1])),
                valor=float(self.rng.uniform(0.35, 1.0)),
                masa_kg=float(self.rng.uniform(0.3, 2.2)),
                requiere=TIPOS_MUESTRA[int(self.rng.integers(0, len(TIPOS_MUESTRA)))],
                prioritario=(i == 0),
            ))
        if self.sitios:
            self.sitios[0].valor = 1.0           # muestra prioritaria

    def _colocar_inicio(self) -> None:
        if self.modo == "sonda":
            candidatos = [(0, c) for c in range(self.cols) if self.clase[0, c] == LIBRE]
            self.inicio = candidatos[len(candidatos) // 2] if candidatos else (0, 0)
            if self.clase[self.inicio] != LIBRE:
                self.clase[self.inicio] = LIBRE
            return
        # El punto de aterrizaje queda CERCA de un refugio pero no sobre él:
        # si el rover empezara sobre la estación, recargaría de inmediato y los
        # escenarios de energía crítica y de "retorno seguro" serían triviales.
        if self.refugios:
            ref = self.refugios[0]
            candidatas = []
            for radio in (3, 4, 2, 5):
                for f in range(max(0, ref[0] - radio), min(self.filas, ref[0] + radio + 1)):
                    for c in range(max(0, ref[1] - radio), min(self.cols, ref[1] + radio + 1)):
                        if (f, c) in self.refugios:
                            continue
                        if self.clase[f, c] == LIBRE and \
                                distancia_cheb((f, c), ref) == radio:
                            candidatas.append((f, c))
                if candidatas:
                    break
            if candidatas:
                self.inicio = candidatas[int(self.rng.integers(0, len(candidatas)))]
                return
        for f in range(self.filas):
            for c in range(self.cols):
                if self.clase[f, c] == LIBRE:
                    self.inicio = (f, c)
                    return
        self.clase[0, 0] = LIBRE
        self.inicio = (0, 0)

    # -- consultas ----------------------------------------------------------
    def dentro(self, p: Tuple[int, int]) -> bool:
        return 0 <= p[0] < self.filas and 0 <= p[1] < self.cols

    def clase_de(self, p: Tuple[int, int]) -> int:
        return int(self.clase[p[0], p[1]]) if self.dentro(p) else OBSTACULO

    def transitable(self, p: Tuple[int, int]) -> bool:
        return self.dentro(p) and self.clase[p[0], p[1]] == LIBRE

    def _calc_gradiente(self, elev: Optional[np.ndarray] = None) -> np.ndarray:
        """Máxima diferencia de altura con los 8 vecinos (m)."""
        e = self.elev if elev is None else elev
        p = np.pad(e, 1, mode="edge")
        g = np.zeros_like(e)
        for dy, dx in DIRECCIONES_8:
            v = p[1 + dy: 1 + dy + e.shape[0], 1 + dx: 1 + dx + e.shape[1]]
            g = np.maximum(g, np.abs(v - e))
        return g

    def pendiente_deg(self, a: Tuple[int, int], b: Tuple[int, int]) -> float:
        """Pendiente entre dos celdas adyacentes (grados). Celda = CELDA_M."""
        if not (self.dentro(a) and self.dentro(b)):
            return 90.0
        dh = float(self.elev[b[0], b[1]] - self.elev[a[0], a[1]])
        dl = CELDA_M * max(1.0, distancia_cheb(a, b))
        return float(np.degrees(np.arctan2(dh, dl)))

    def pendiente_local(self, p: Tuple[int, int]) -> float:
        if not self.dentro(p):
            return 45.0
        return float(np.degrees(np.arctan2(self.gradiente[p[0], p[1]], CELDA_M)))

    def es_noche(self) -> bool:
        return not (6.0 <= (self.hora % 24.0) < 18.0)

    def irradiancia(self) -> float:
        """Fracción de potencia solar disponible (0-1) [SUPUESTO]."""
        if self.es_noche():
            return 0.0
        ang = np.sin(np.pi * ((self.hora % 24.0) - 6.0) / 12.0)
        base = max(0.0, float(ang)) * min(2.0, self.perfil.irradiancia_rel)
        return clamp(base * (1.0 - 0.85 * self.tormenta), 0.0, 2.0)

    def temperatura_superficie(self, p: Tuple[int, int]) -> float:
        """Temperatura ambiental simulada en una celda."""
        amp = (self.perfil.temp_max_C - self.perfil.temp_min_C) / 2.0
        media = self.perfil.temp_media_C
        ciclo = np.sin(np.pi * ((self.hora % 24.0) - 6.0) / 12.0)
        t = media + amp * float(ciclo) * 0.75
        if self.dentro(p):
            t -= 0.25 * float(self.elev[p[0], p[1]])     # gradiente con altura
        return float(t)

    def riesgo_real(self, p: Tuple[int, int]) -> float:
        """Riesgo verdadero de una celda (sólo para métricas, no para el AG)."""
        if not self.dentro(p):
            return 1.0
        cl = self.clase_de(p)
        if cl == BARRANCO:
            return 1.0
        r = 0.08 + 0.5 * norm01(self.gradiente[p[0], p[1]], 0.0, 8.0)
        if cl == OBSTACULO:
            r = max(r, 0.85)
        for z in self.zonas_riesgo:
            if distancia_cheb(p, z["centro"]) <= z["radio"]:
                r = max(r, z["intensidad"])
        return clamp(r, 0.0, 1.0)

    def avanzar_tiempo(self) -> None:
        self.hora = (self.hora + self.cfg.horas_por_paso) % 24.0
        self.sismo = max(0.0, self.sismo - 0.25)
        self.tormenta = max(0.0, self.tormenta - 0.012)
        self.visibilidad = clamp(1.0 - 0.75 * self.tormenta -
                                 (0.25 if self.es_noche() else 0.0), 0.05, 1.0)
        for z in list(self.zonas_riesgo):
            z["duracion"] -= 1
            if z["duracion"] <= 0:
                self.zonas_riesgo.remove(z)

    def resumen(self) -> dict:
        return {
            "planeta": self.perfil.nombre, "modo": self.modo,
            "dimensiones": f"{self.filas}x{self.cols}",
            "libres": int((self.clase == LIBRE).sum()),
            "obstaculos": int((self.clase == OBSTACULO).sum()),
            "barrancos": int((self.clase == BARRANCO).sum()),
            "sitios_ciencia": len(self.sitios), "refugios": len(self.refugios),
            "inicio": self.inicio,
        }


# ----------------------------------------------------------------------------
# BeliefMap: lo que el rover cree del mundo
# ----------------------------------------------------------------------------
class BeliefMap:
    """Creencias del rover sobre el entorno.

    - p_obst / p_barr: probabilidad estimada por la RNA (o por sensores
      deterministas) de que una celda sea obstáculo o barranco.
    - confirmado: celdas verificadas por sensor de borde o por colisión.
    - prohibido: zonas vetadas por el supervisor (RF: crear zona prohibida).
    - visitado / explorado: cobertura.

    El AG planifica SOBRE ESTE MAPA, nunca sobre la verdad del terreno.
    """

    def __init__(self, filas: int, cols: int):
        self.filas, self.cols = filas, cols
        self.p_obst = np.full((filas, cols), 0.12)   # prior neutro
        self.p_barr = np.full((filas, cols), 0.08)
        self.confirmado = np.zeros((filas, cols), dtype=bool)
        self.prohibido = np.zeros((filas, cols), dtype=bool)
        self.explorado = np.zeros((filas, cols), dtype=bool)
        self.visitado = np.zeros((filas, cols), dtype=bool)
        self.pendiente_est = np.zeros((filas, cols))

    def dentro(self, p) -> bool:
        return 0 <= p[0] < self.filas and 0 <= p[1] < self.cols

    def actualizar(self, p, p_libre: float, p_obst: float, p_barr: float,
                   pendiente: float, alfa: float = 0.55) -> None:
        """Fusión exponencial de la predicción de la RNA con la creencia previa."""
        if not self.dentro(p):
            return
        f, c = p
        if self.confirmado[f, c]:
            return
        self.p_obst[f, c] = (1 - alfa) * self.p_obst[f, c] + alfa * p_obst
        self.p_barr[f, c] = (1 - alfa) * self.p_barr[f, c] + alfa * p_barr
        self.pendiente_est[f, c] = pendiente
        self.explorado[f, c] = True

    def confirmar(self, p, clase: int) -> None:
        """Verificación determinista (sensor de borde / colisión / llegada)."""
        if not self.dentro(p):
            return
        f, c = p
        self.confirmado[f, c] = True
        self.explorado[f, c] = True
        self.p_obst[f, c] = 1.0 if clase == OBSTACULO else 0.0
        self.p_barr[f, c] = 1.0 if clase == BARRANCO else 0.0

    def prohibir(self, p, radio: int = 0) -> None:
        for df in range(-radio, radio + 1):
            for dc in range(-radio, radio + 1):
                q = (p[0] + df, p[1] + dc)
                if self.dentro(q):
                    self.prohibido[q[0], q[1]] = True

    def riesgo_estimado(self, p) -> float:
        """Riesgo que ve el planificador (0-1)."""
        if not self.dentro(p):
            return 1.0
        f, c = p
        if self.prohibido[f, c]:
            return 1.0
        r = 0.85 * float(self.p_barr[f, c]) + 0.55 * float(self.p_obst[f, c])
        r += 0.25 * norm01(float(self.pendiente_est[f, c]), 0.0, 35.0)
        if not self.explorado[f, c]:
            r += 0.10                     # penalización por incertidumbre
        return clamp(r, 0.0, 1.0)

    def bloqueada(self, p, umbral: float = 0.85) -> bool:
        """¿El rover CREE que esta celda es intransitable?

        El umbral es deliberadamente ALTO para las creencias no confirmadas: una
        predicción probabilística no debe cerrar el paso por sí sola, porque con
        un 5-8 % de falsos positivos el mapa navegable se fragmenta y el rover
        se encierra. La RNA influye sobre todo a través del COSTE de riesgo
        (blando) que evalúa el AG; sólo las celdas CONFIRMADAS por el sensor de
        borde o por colisión bloquean de forma dura.
        """
        if not self.dentro(p):
            return True
        f, c = p
        if self.prohibido[f, c]:
            return True
        if self.confirmado[f, c]:
            return (self.p_barr[f, c] >= 0.99) or (self.p_obst[f, c] >= 0.99)
        return (self.p_barr[f, c] >= umbral) or (self.p_obst[f, c] >= umbral)

    def mapa_riesgo(self) -> np.ndarray:
        r = 0.85 * self.p_barr + 0.55 * self.p_obst + \
            0.25 * np.clip(self.pendiente_est / 35.0, 0, 1)
        r = np.where(self.explorado, r, r + 0.10)
        r = np.where(self.prohibido, 1.0, r)
        return np.clip(r, 0.0, 1.0)

## 3 · Suite de sensores, ruido, deriva y fallos

Toda la información del entorno pasa por aquí con ruido que **crece cuando baja la visibilidad**
(polvo o noche), con deriva acumulada en la odometría y con fallos de sensor (NaN o saturación)
aleatorios o inyectados por el operador.

`extraer_rasgos()` es el **único** punto donde se construye el vector de entrada de la RNA, y se
usa tanto para generar el dataset como para inferir en línea: así no puede haber desajuste entre
entrenamiento y operación.

Los diez rasgos son magnitudes físicas observables, **ninguna es la etiqueta sin transformar**.
Dos son especialmente informativos y vale la pena entender por qué:

* **Proximidad frontal**: un obstáculo devuelve un eco cercano; un barranco es un hueco que
  *no* refleja y devuelve lectura larga. Separa obstáculo de barranco.
* **Dispersión LIDAR y contraste de borde**: altos en discontinuidades de relieve.

`confirmar_borde()` es el sensor determinista de corto alcance con 97 % de acierto: es el que
habilita el **veto de seguridad** y el que contrasta la predicción probabilística de la red.

In [ ]:
"""
ARES-X · Módulo 3: Suite de sensores, ruido, deriva y fallos.

Principio: el rover NO lee la verdad del terreno. Toda información del entorno
pasa por este módulo, que añade ruido, sesgo, deriva y fallos (NaN/saturación).

`extraer_rasgos()` es el ÚNICO punto donde se construye el vector de entrada de
la RNA, y se usa tanto para generar el dataset sintético como para inferir en
línea: así se garantiza que no haya desajuste entre entrenamiento y operación.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np


# Nombres de los rasgos de entrada de la RNA (orden fijo y auditable)
RASGOS: List[str] = [
    "prox_frontal_m",      # 0-20 m, saturado
    "delta_altura_m",      # altura(destino) - altura(actual)
    "pendiente_deg",       # 0-45
    "contraste_borde",     # discontinuidad de relieve (LIDAR)
    "vibracion",           # acelerómetro normalizado
    "textura",             # cámara: rugosidad aparente 0-1
    "reflectancia",        # cámara multiespectral 0-1
    "delta_temp_C",        # contraste térmico con el entorno
    "visibilidad",         # 0-1 (polvo/noche)
    "dispersion_lidar",    # varianza de retornos
]
N_RASGOS = len(RASGOS)

# Rangos para normalización min-max (fijos, parte del preprocesamiento)
RANGOS_RASGOS: Dict[str, Tuple[float, float]] = {
    "prox_frontal_m": (0.0, 20.0),
    "delta_altura_m": (-12.0, 12.0),
    "pendiente_deg": (0.0, 70.0),
    "contraste_borde": (0.0, 12.0),
    "vibracion": (0.0, 3.0),
    "textura": (0.0, 1.0),
    "reflectancia": (0.0, 1.0),
    "delta_temp_C": (-20.0, 20.0),
    "visibilidad": (0.0, 1.0),
    "dispersion_lidar": (0.0, 5.0),
}

SENSORES_CRITICOS = ["proximidad", "lidar", "inclinometro"]


def normalizar_rasgos(X: np.ndarray) -> np.ndarray:
    """Normaliza min-max a [0,1] con los rangos fijos. NaN -> 0.5 (neutro)."""
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(1, -1)
    Z = np.empty_like(X)
    for j, nombre in enumerate(RASGOS):
        lo, hi = RANGOS_RASGOS[nombre]
        Z[:, j] = (X[:, j] - lo) / (hi - lo)
    Z = np.clip(Z, -0.25, 1.25)
    return np.nan_to_num(Z, nan=0.5, posinf=1.25, neginf=-0.25)


# ----------------------------------------------------------------------------
# Extracción de rasgos de una celda objetivo
# ----------------------------------------------------------------------------
def extraer_rasgos(world: World, origen: Tuple[int, int], destino: Tuple[int, int],
                   rng: np.random.Generator, ruido: float = 1.0) -> np.ndarray:
    """Observación ruidosa de la celda `destino` vista desde `origen`.

    IMPORTANTE (RF-04): ninguna componente es la etiqueta de clase sin
    transformar. Todos los rasgos son magnitudes físicas observables, con ruido
    que depende de la visibilidad, la tormenta y la distancia.
    """
    if not world.dentro(destino):
        # Fuera del mapa: lectura de "pared" saturada
        v = np.array([0.0, 0.0, 45.0, 0.0, 0.1, 0.9, 0.2, 0.0,
                      world.visibilidad, 0.2])
        return v

    f, c = destino
    clase = world.clase_de(destino)
    vis = clamp(world.visibilidad, 0.05, 1.0)
    # El ruido crece cuando hay polvo o de noche
    k = ruido * (1.0 + 1.6 * (1.0 - vis))

    # 1) Proximidad frontal: un obstáculo devuelve eco cercano; un barranco NO
    #    refleja (hueco) y devuelve lectura larga -> rasgo discriminante.
    dist_celdas = max(1, abs(destino[0] - origen[0]), abs(destino[1] - origen[1]))
    if clase == OBSTACULO:
        prox = dist_celdas * CELDA_M + rng.normal(0, 0.9 * k)
    elif clase == BARRANCO:
        prox = 20.0 - rng.exponential(2.2 * k)
    else:
        prox = 20.0 - rng.exponential(1.1 * k)
        # eco espurio por polvo suspendido
        if rng.random() < 0.05 * k:
            prox = dist_celdas * CELDA_M + rng.normal(0, 2.0)
    prox = clamp(prox, 0.0, 20.0)

    # 2) Diferencia de altura (altimetría)
    dh = float(world.elev[f, c] - world.elev[origen[0], origen[1]])
    dh += rng.normal(0, 0.45 * k)

    # 3) Pendiente local medida por inclinómetro/LIDAR
    pend = world.pendiente_local(destino) + rng.normal(0, 3.2 * k)
    pend = clamp(pend, 0.0, 70.0)

    # 4) Contraste de borde: máxima discontinuidad entre destino y sus vecinos
    contraste = float(world.gradiente[f, c]) + rng.normal(0, 0.6 * k)
    contraste = clamp(contraste, 0.0, 12.0)

    # 5) Vibración del chasis (mayor sobre terreno rugoso o en sismo)
    vib = 0.18 + 0.55 * norm01(float(world.gradiente[f, c]), 0.0, 8.0)
    vib += 1.4 * world.sismo + rng.normal(0, 0.12 * k)
    vib = clamp(vib, 0.0, 3.0)

    # 6-7) Cámara: textura y reflectancia multiespectral (degradadas por polvo)
    tex = float(world.textura[f, c]) * vis + (1 - vis) * rng.random()
    tex = clamp(tex + rng.normal(0, 0.09 * k), 0.0, 1.0)
    refl = float(world.reflectancia[f, c]) * vis + (1 - vis) * rng.random()
    refl = clamp(refl + rng.normal(0, 0.09 * k), 0.0, 1.0)

    # 8) Contraste térmico: las sombras de barranco son más frías
    t_dest = world.temperatura_superficie(destino)
    t_orig = world.temperatura_superficie(origen)
    dt = (t_dest - t_orig) - (3.5 if clase == BARRANCO else 0.0)
    dt += rng.normal(0, 1.5 * k)
    dt = clamp(dt, -20.0, 20.0)

    # 9) Visibilidad reportada por el fotómetro
    visib = clamp(vis + rng.normal(0, 0.05 * k), 0.0, 1.0)

    # 10) Dispersión de retornos LIDAR (alta en bordes y huecos)
    disp = 0.3 + 0.35 * norm01(contraste, 0, 12) + \
        (1.1 if clase == BARRANCO else 0.0) + rng.normal(0, 0.35 * k)
    disp = clamp(disp, 0.0, 5.0)

    return np.array([prox, dh, pend, contraste, vib, tex, refl, dt, visib, disp],
                    dtype=float)


# ----------------------------------------------------------------------------
# Telemetría completa
# ----------------------------------------------------------------------------
@dataclass
class Lectura:
    """Telemetría de un paso (Panel B). `validos` indica qué sensores sirven."""
    prox_frontal_m: float = 20.0
    prox_lateral_izq_m: float = 20.0
    prox_lateral_der_m: float = 20.0
    altimetria_m: float = 0.0
    gradiente_m: float = 0.0
    inclinacion_deg: float = 0.0
    aceleracion: float = 0.0
    temp_exterior_C: float = 0.0
    temp_interior_C: float = 0.0
    radiacion_mSv_h: float = 0.0
    presion_bar: float = 0.0
    o2_pct: float = 0.0
    co2_pct: float = 0.0
    so2_ppm: float = 0.0
    particulas: float = 0.0
    visibilidad_m: float = 100.0
    bateria_pct: float = 100.0
    potencia_solar_W: float = 0.0
    odometria_m: float = 0.0
    rumbo_deg: float = 0.0
    pos_estimada: Tuple[int, int] = (0, 0)
    estado_mecanico: float = 100.0
    latencia_enlace_s: float = 0.4
    enlace_ok: bool = True
    validos: Dict[str, bool] = field(default_factory=dict)
    fallos: List[str] = field(default_factory=list)

    def sensor_critico_invalido(self) -> Optional[str]:
        for s in SENSORES_CRITICOS:
            if not self.validos.get(s, True):
                return s
        return None

    def to_dict(self) -> dict:
        d = {k: v for k, v in self.__dict__.items()
             if k not in ("validos", "fallos", "pos_estimada")}
        d["pos_estimada"] = str(self.pos_estimada)
        d["fallos"] = "|".join(self.fallos)
        return d


class SensorSuite:
    """Genera la telemetría del paso y los rasgos para la RNA."""

    def __init__(self, cfg: Config, semilla_extra: str = "sensores"):
        self.cfg = cfg
        self.rng = rng_de(cfg.semilla, semilla_extra)
        self.deriva_odom = 0.0
        self.fallos_forzados: Dict[str, str] = {}   # sensor -> 'nan'|'saturado'
        self.contador = 0

    # -- inyección manual de fallos (Panel E / RF-13) -----------------------
    def forzar_fallo(self, sensor: str, tipo: str = "nan") -> None:
        self.fallos_forzados[sensor] = tipo

    def reparar(self, sensor: Optional[str] = None) -> None:
        if sensor is None:
            self.fallos_forzados.clear()
        else:
            self.fallos_forzados.pop(sensor, None)

    def _aplicar_fallo(self, sensor: str, valor: float,
                       sat: float) -> Tuple[float, bool, Optional[str]]:
        tipo = self.fallos_forzados.get(sensor)
        if tipo is None and self.cfg.eventos_aleatorios:
            if self.rng.random() < self.cfg.prob_fallo_sensor:
                tipo = "nan" if self.rng.random() < 0.6 else "saturado"
        if tipo == "nan":
            return (float("nan"), False, f"{sensor}:NaN")
        if tipo == "saturado":
            return (sat, False, f"{sensor}:saturado")
        return (valor, True, None)

    # -- observación completa ----------------------------------------------
    def observar(self, world: World, estado) -> Lectura:
        """estado: objeto RoverState (módulo 8). Devuelve Lectura."""
        self.contador += 1
        L = Lectura()
        pos = estado.pos
        vis = clamp(world.visibilidad, 0.05, 1.0)
        k = 1.0 + 1.6 * (1.0 - vis)

        # Proximidad en 3 direcciones relativas al rumbo
        idx = estado.rumbo_idx
        for etiqueta, salto in (("prox_frontal_m", 0),
                                ("prox_lateral_izq_m", -2),
                                ("prox_lateral_der_m", 2)):
            d = DIRECCIONES_8[(idx + salto) % 8]
            valor = self._raycast(world, pos, d, vis)
            setattr(L, etiqueta, valor)
        L.prox_frontal_m, ok, fallo = self._aplicar_fallo(
            "proximidad", L.prox_frontal_m, 20.0)
        L.validos["proximidad"] = ok
        if fallo:
            L.fallos.append(fallo)

        # Altimetría y gradiente (LIDAR)
        L.altimetria_m = float(world.elev[pos[0], pos[1]]) + self.rng.normal(0, 0.3 * k)
        g = float(world.gradiente[pos[0], pos[1]]) + self.rng.normal(0, 0.4 * k)
        L.gradiente_m, ok, fallo = self._aplicar_fallo("lidar", clamp(g, 0, 12), 12.0)
        L.validos["lidar"] = ok
        if fallo:
            L.fallos.append(fallo)

        # Inclinómetro
        inc = world.pendiente_local(pos) + self.rng.normal(0, 2.0 * k)
        L.inclinacion_deg, ok, fallo = self._aplicar_fallo(
            "inclinometro", clamp(inc, 0, 70), 70.0)
        L.validos["inclinometro"] = ok
        if fallo:
            L.fallos.append(fallo)

        # Acelerómetro / sismómetro
        acc = 0.15 + 1.8 * world.sismo + 0.3 * estado.velocidad + \
            self.rng.normal(0, 0.1 * k)
        L.aceleracion, ok, fallo = self._aplicar_fallo("acelerometro",
                                                       clamp(acc, 0, 5), 5.0)
        L.validos["acelerometro"] = ok
        if fallo:
            L.fallos.append(fallo)

        # Térmicos: exterior (ambiente) vs interior (electrónica)
        L.temp_exterior_C = world.temperatura_superficie(pos) + self.rng.normal(0, 1.2)
        L.temp_interior_C = estado.temp_interna + self.rng.normal(0, 0.6)
        L.validos["termometro"] = True

        # Radiación, presión, gases: COMPOSICIÓN ambiental, no "respiración"
        p = world.perfil
        L.radiacion_mSv_h = max(0.0, p.radiacion_base_mSv_h *
                                (1.0 + 0.8 * world.irradiancia()) +
                                self.rng.normal(0, 0.01))
        L.presion_bar = p.presion_bar * (1.0 + self.rng.normal(0, 0.02))
        L.o2_pct = p.atmosfera.get("O2", 0.0) * (1 + self.rng.normal(0, 0.01))
        L.co2_pct = p.atmosfera.get("CO2", 0.0) * (1 + self.rng.normal(0, 0.01))
        L.so2_ppm = p.atmosfera.get("SO2", 0.0) * 1e4 * (1 + self.rng.normal(0, 0.05))

        # Partículas y visibilidad óptica
        L.particulas = clamp(world.tormenta + self.rng.normal(0, 0.03), 0, 1)
        L.visibilidad_m = clamp(vis * 250.0, 2.0, 250.0)
        L.validos["camara"] = vis > 0.12

        # Energía
        L.bateria_pct = clamp(estado.bateria, 0.0, 100.0)
        L.potencia_solar_W = 32.0 * world.irradiancia() * (1 - 0.5 * world.tormenta)

        # Odometría con deriva acumulada (localización imperfecta)
        self.deriva_odom += 0.004 * estado.distancia_recorrida_m * self.rng.random()
        L.odometria_m = estado.distancia_recorrida_m + self.deriva_odom
        L.rumbo_deg = (idx * 45.0 + self.rng.normal(0, 2.5)) % 360.0
        err = 1 if (self.deriva_odom > 12.0 and self.rng.random() < 0.3) else 0
        L.pos_estimada = (pos[0] + err * int(self.rng.integers(-1, 2)),
                          pos[1] + err * int(self.rng.integers(-1, 2)))

        # Estado mecánico y enlace
        L.estado_mecanico = clamp(100.0 - estado.desgaste, 0.0, 100.0)
        L.latencia_enlace_s = float(abs(self.rng.normal(0.4 + 2.0 * world.tormenta, 0.3)))
        L.enlace_ok = not (self.fallos_forzados.get("enlace") or
                           (self.cfg.eventos_aleatorios and self.rng.random() < 0.02))
        if not L.enlace_ok:
            L.fallos.append("enlace:degradado")
        L.validos["enlace"] = L.enlace_ok

        return L

    def _raycast(self, world: World, pos, d, vis) -> float:
        """Distancia al primer obstáculo en una dirección (saturada a 20 m)."""
        alcance = int(20.0 / CELDA_M)
        for i in range(1, alcance + 1):
            q = (pos[0] + d[0] * i, pos[1] + d[1] * i)
            if not world.dentro(q):
                return clamp(i * CELDA_M + self.rng.normal(0, 0.5), 0, 20)
            if world.clase_de(q) == OBSTACULO:
                return clamp(i * CELDA_M + self.rng.normal(0, 0.8 / vis), 0, 20)
        return clamp(20.0 - self.rng.exponential(1.0), 0, 20)

    # -- rasgos para la RNA --------------------------------------------------
    def rasgos_vecindario(self, world: World, pos: Tuple[int, int]
                          ) -> Dict[Tuple[int, int], np.ndarray]:
        """Rasgos observados de las 8 celdas vecinas (entrada de la RNA)."""
        out = {}
        for d in DIRECCIONES_8:
            q = (pos[0] + d[0], pos[1] + d[1])
            out[q] = extraer_rasgos(world, pos, q, self.rng)
        return out

    def confirmar_borde(self, world: World, destino: Tuple[int, int]) -> Optional[int]:
        """Sensor determinista de borde: a 1 celda confirma barranco/obstáculo.

        Es el "veto de seguridad" del caso: tiene 97 % de acierto y sirve para
        contrastar la predicción probabilística de la RNA.
        """
        if not world.dentro(destino):
            return OBSTACULO
        verdad = world.clase_de(destino)
        if self.rng.random() < 0.97:
            return verdad
        return int(self.rng.integers(0, 3))

## 4 · Percepción neuronal (RNA)

**Tarea**: clasificar la celda objetivo en `{libre, obstaculo, barranco}` desde lecturas ruidosas.

Tres decisiones metodológicas que condicionan la validez de las métricas:

1. **Partición por mapas, no por filas.** Las celdas de un mismo mapa están fuertemente
   correlacionadas; repartirlas al azar produce fuga de información y métricas infladas.
   Se generan 24 mapas independientes (4 planetas × 3 dificultades, con hora y polvo variables)
   y se asignan **enteros** a train (14), validación (4) y test (6).
   `verificar_sin_fuga()` comprueba que ningún mapa aparece en dos particiones.
2. **La verdad del terreno está definida aparte de los rasgos.** La etiqueta es la clase real de
   la celda; los rasgos son observaciones con ruido de esa celda.
3. **Baselines obligatorios.** Se compara contra un clasificador mayoritario y contra reglas fijas
   de umbral sobre dos rasgos. Sin baseline no se puede afirmar que la red aporta algo.

Arquitectura mínima exigida: **MLP de dos capas ocultas (32, 16)**, ReLU, softmax de 3 clases,
entrenado en CPU con `partial_fit` por épocas para poder registrar el historial y permitir la
inspección de iteraciones (RF-11).

Métricas reportadas: matriz de confusión, macro-F1, **recall de barrancos** (el error crítico),
falsos negativos barranco→libre y **error de calibración esperado (ECE)**, porque el supervisor
usa la confianza como señal y una confianza mal calibrada sería engañosa.

In [ ]:
"""
ARES-X · Módulo 4: Percepción neuronal (RNA).

Tarea: clasificar la celda objetivo en {libre, obstaculo, barranco} a partir de
lecturas de sensores con ruido.

Decisiones metodológicas exigidas por el caso (RF-04):
  * Los datos se generan con el MISMO extractor de rasgos que se usa en línea.
  * La verdad del terreno (etiqueta) es la clase real de la celda; NINGÚN rasgo
    es la etiqueta sin transformar.
  * La partición train/valid/test se hace POR MAPA, no por filas aleatorias:
    las celdas de un mismo mapa están fuertemente correlacionadas y dividirlas
    al azar produce fuga de información y métricas infladas.
  * Se reportan baseline, matriz de confusión, macro-F1, recall de barrancos y
    error de calibración esperado (ECE).
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np


CLASES = [LIBRE, OBSTACULO, BARRANCO]
NOMBRES_CLASES = [NOMBRE_CLASE[c] for c in CLASES]


# ----------------------------------------------------------------------------
# Generación del dataset sintético
# ----------------------------------------------------------------------------
@dataclass
class Dataset:
    X: np.ndarray
    y: np.ndarray
    mapa_id: np.ndarray
    particion: np.ndarray          # 'train' | 'valid' | 'test'

    def subset(self, nombre: str) -> Tuple[np.ndarray, np.ndarray]:
        m = self.particion == nombre
        return self.X[m], self.y[m]

    def resumen(self) -> Dict[str, dict]:
        out = {}
        for p in ("train", "valid", "test"):
            m = self.particion == p
            conteo = {NOMBRE_CLASE[c]: int((self.y[m] == c).sum()) for c in CLASES}
            out[p] = {"n": int(m.sum()), "mapas": int(len(np.unique(self.mapa_id[m]))),
                      **conteo}
        return out


def generar_dataset(cfg_base: Config, n_mapas: int = 24, muestras_por_mapa: int = 260,
                    semilla: int = 20260,
                    reparto: Tuple[int, int, int] = (14, 4, 6)) -> Dataset:
    """Crea el dataset etiquetado recorriendo mapas independientes.

    Cada muestra = (celda origen, celda destino adyacente) observada con ruido.
    Se equilibran las clases dentro de cada mapa para que los barrancos (clase
    minoritaria y crítica) tengan representación suficiente.
    """
    n_tr, n_va, n_te = reparto
    assert n_tr + n_va + n_te <= n_mapas, "El reparto excede el número de mapas"

    Xs, ys, ids, parts = [], [], [], []
    planetas = ["marte", "tierra", "mercurio", "venus"]
    dificultades = ["facil", "media", "dificil"]

    for m in range(n_mapas):
        rng = rng_de(semilla + m, "dataset")
        cfg = Config(
            planeta=planetas[m % len(planetas)],
            modo="rover",
            filas=cfg_base.filas, columnas=cfg_base.columnas,
            dificultad=dificultades[m % len(dificultades)],
            semilla=semilla + m * 101,
            n_sitios_ciencia=0, eventos_aleatorios=False,
        )
        w = World(cfg)
        # Condiciones ambientales variadas: día/noche y polvo
        w.hora = float(rng.uniform(0, 24))
        w.tormenta = float(rng.choice([0.0, 0.0, 0.25, 0.6]))
        w.visibilidad = clamp(1.0 - 0.75 * w.tormenta -
                              (0.25 if w.es_noche() else 0.0), 0.05, 1.0)

        # Índice de celdas por clase
        por_clase: Dict[int, List[Tuple[int, int]]] = {c: [] for c in CLASES}
        for f in range(w.filas):
            for c in range(w.cols):
                por_clase[int(w.clase[f, c])].append((f, c))

        cupo = max(1, muestras_por_mapa // 3)
        for clase in CLASES:
            celdas = por_clase[clase]
            if not celdas:
                continue
            idx = rng.integers(0, len(celdas), size=cupo)
            for i in idx:
                destino = celdas[int(i)]
                # Origen: vecino transitable (desde donde se observaría)
                vecinos = [(destino[0] + d[0], destino[1] + d[1])
                           for d in [(-1, 0), (1, 0), (0, -1), (0, 1),
                                     (-1, -1), (-1, 1), (1, -1), (1, 1)]]
                vecinos = [v for v in vecinos if w.dentro(v)]
                if not vecinos:
                    continue
                origen = vecinos[int(rng.integers(0, len(vecinos)))]
                x = extraer_rasgos(w, origen, destino, rng)
                Xs.append(x)
                ys.append(clase)
                ids.append(m)
                parts.append("train" if m < n_tr else
                             ("valid" if m < n_tr + n_va else "test"))

    return Dataset(
        X=np.array(Xs, dtype=float),
        y=np.array(ys, dtype=int),
        mapa_id=np.array(ids, dtype=int),
        particion=np.array(parts, dtype=object),
    )


def verificar_sin_fuga(ds: Dataset) -> Dict[str, object]:
    """Comprueba que ningún mapa aparece en dos particiones."""
    conjuntos = {p: set(ds.mapa_id[ds.particion == p].tolist())
                 for p in ("train", "valid", "test")}
    solapes = {}
    claves = list(conjuntos)
    for i in range(len(claves)):
        for j in range(i + 1, len(claves)):
            inter = conjuntos[claves[i]] & conjuntos[claves[j]]
            solapes[f"{claves[i]}∩{claves[j]}"] = sorted(inter)
    ok = all(len(v) == 0 for v in solapes.values())
    return {"sin_fuga": ok, "mapas_por_particion":
            {k: sorted(v) for k, v in conjuntos.items()}, "solapes": solapes}


# ----------------------------------------------------------------------------
# Métricas
# ----------------------------------------------------------------------------
def matriz_confusion(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    M = np.zeros((3, 3), dtype=int)
    for t, p in zip(y_true, y_pred):
        M[int(t), int(p)] += 1
    return M


def metricas_clasificacion(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    M = matriz_confusion(y_true, y_pred)
    out: Dict[str, float] = {}
    f1s = []
    for c in CLASES:
        tp = M[c, c]
        fp = M[:, c].sum() - tp
        fn = M[c, :].sum() - tp
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        out[f"precision_{NOMBRE_CLASE[c]}"] = float(prec)
        out[f"recall_{NOMBRE_CLASE[c]}"] = float(rec)
        out[f"f1_{NOMBRE_CLASE[c]}"] = float(f1)
        f1s.append(f1)
    out["exactitud"] = float((y_true == y_pred).mean()) if len(y_true) else 0.0
    out["macro_f1"] = float(np.mean(f1s))
    out["recall_barranco"] = out[f"recall_{NOMBRE_CLASE[BARRANCO]}"]
    # Coste crítico: barranco clasificado como libre (falso negativo peligroso)
    out["barranco_como_libre"] = int(M[BARRANCO, LIBRE])
    return out


def error_calibracion(y_true: np.ndarray, probas: np.ndarray,
                      bins: int = 10) -> Tuple[float, np.ndarray, np.ndarray, np.ndarray]:
    """ECE: |confianza media - exactitud| ponderado por bin."""
    conf = probas.max(axis=1)
    pred = probas.argmax(axis=1)
    acierto = (pred == y_true).astype(float)
    bordes = np.linspace(0.0, 1.0, bins + 1)
    ece, xs, accs, cnts = 0.0, [], [], []
    for i in range(bins):
        m = (conf > bordes[i]) & (conf <= bordes[i + 1])
        n = int(m.sum())
        cnts.append(n)
        if n == 0:
            xs.append((bordes[i] + bordes[i + 1]) / 2)
            accs.append(np.nan)
            continue
        a = float(acierto[m].mean())
        c = float(conf[m].mean())
        ece += (n / len(conf)) * abs(a - c)
        xs.append(c)
        accs.append(a)
    return float(ece), np.array(xs), np.array(accs), np.array(cnts)


# ----------------------------------------------------------------------------
# Baselines
# ----------------------------------------------------------------------------
def baseline_mayoritario(y_tr: np.ndarray, y_te: np.ndarray) -> Dict[str, float]:
    clase = int(np.bincount(y_tr, minlength=3).argmax())
    pred = np.full_like(y_te, clase)
    m = metricas_clasificacion(y_te, pred)
    m["_descripcion"] = f"Siempre predice '{NOMBRE_CLASE[clase]}'"
    return m


def baseline_reglas(X_te: np.ndarray, y_te: np.ndarray) -> Dict[str, float]:
    """Umbrales fijos sobre dos rasgos: referencia 'no neuronal'."""
    prox = X_te[:, RASGOS.index("prox_frontal_m")]
    dh = X_te[:, RASGOS.index("delta_altura_m")]
    pred = np.full(len(y_te), LIBRE)
    pred[prox < 9.0] = OBSTACULO
    pred[dh < -3.0] = BARRANCO
    m = metricas_clasificacion(y_te, pred)
    m["_descripcion"] = "Reglas fijas: prox<9 m -> obstáculo; Δh<-3 m -> barranco"
    return m


# ----------------------------------------------------------------------------
# NeuralPerception
# ----------------------------------------------------------------------------
class NeuralPerception:
    """MLP de dos capas ocultas (32, 16), ReLU, salida softmax de 3 clases.

    Se entrena por épocas con `partial_fit` para poder registrar el historial de
    pérdida y exactitud (RF-11) y permitir la inspección de iteraciones.
    """

    def __init__(self, capas: Tuple[int, ...] = (32, 16), semilla: int = 42,
                 lr: float = 0.012, epocas: int = 140, batch: int = 64):
        from sklearn.neural_network import MLPClassifier
        self.capas = capas
        self.epocas = epocas
        self.batch = batch
        self.semilla = semilla
        self.modelo = MLPClassifier(
            hidden_layer_sizes=capas, activation="relu", solver="adam",
            learning_rate_init=lr, alpha=1e-4, random_state=semilla,
            max_iter=1, warm_start=True, batch_size=min(batch, 200),
        )
        self.historial: Dict[str, List[float]] = {
            "epoca": [], "perdida": [], "acc_train": [], "acc_valid": [],
            "recall_barranco_valid": [],
        }
        self.entrenado = False
        self.metricas_test: Dict[str, float] = {}
        self.matriz_test: Optional[np.ndarray] = None
        self.ece_test: float = float("nan")
        self.baselines: Dict[str, dict] = {}
        self.dataset_resumen: Dict[str, dict] = {}
        self.n_parametros: int = 0

    # -- entrenamiento -------------------------------------------------------
    def entrenar(self, ds: Dataset, verbose: bool = True) -> "NeuralPerception":
        Xtr, ytr = ds.subset("train")
        Xva, yva = ds.subset("valid")
        Xte, yte = ds.subset("test")
        Ztr, Zva, Zte = (normalizar_rasgos(Xtr), normalizar_rasgos(Xva),
                         normalizar_rasgos(Xte))
        rng = np.random.default_rng(self.semilla)

        for ep in range(self.epocas):
            orden = rng.permutation(len(Ztr))
            self.modelo.partial_fit(Ztr[orden], ytr[orden], classes=np.array(CLASES))
            if ep % 2 == 0 or ep == self.epocas - 1:
                p_tr = self.modelo.predict(Ztr)
                p_va = self.modelo.predict(Zva)
                mv = metricas_clasificacion(yva, p_va)
                self.historial["epoca"].append(ep)
                self.historial["perdida"].append(float(self.modelo.loss_))
                self.historial["acc_train"].append(float((p_tr == ytr).mean()))
                self.historial["acc_valid"].append(float((p_va == yva).mean()))
                self.historial["recall_barranco_valid"].append(mv["recall_barranco"])

        self.entrenado = True
        self.n_parametros = int(sum(c.size for c in self.modelo.coefs_) +
                                sum(b.size for b in self.modelo.intercepts_))

        # Evaluación final SOBRE TEST (mapas nunca vistos)
        pte = self.modelo.predict(Zte)
        self.metricas_test = metricas_clasificacion(yte, pte)
        self.matriz_test = matriz_confusion(yte, pte)
        probas = self.modelo.predict_proba(Zte)
        self.ece_test = error_calibracion(yte, probas)[0]
        self.baselines = {
            "mayoritario": baseline_mayoritario(ytr, yte),
            "reglas_fijas": baseline_reglas(Xte, yte),
        }
        self.dataset_resumen = ds.resumen()
        if verbose:
            print(f"  RNA entrenada: capas={self.capas}  parámetros={self.n_parametros}")
            print(f"  TEST  exactitud={self.metricas_test['exactitud']:.3f}  "
                  f"macro-F1={self.metricas_test['macro_f1']:.3f}  "
                  f"recall_barranco={self.metricas_test['recall_barranco']:.3f}  "
                  f"ECE={self.ece_test:.3f}")
            print(f"  Baseline mayoritario macro-F1="
                  f"{self.baselines['mayoritario']['macro_f1']:.3f} | "
                  f"reglas fijas macro-F1="
                  f"{self.baselines['reglas_fijas']['macro_f1']:.3f}")
        return self

    # -- inferencia ----------------------------------------------------------
    def predecir(self, X: np.ndarray) -> np.ndarray:
        """Devuelve probabilidades [n,3] en el orden [libre, obstaculo, barranco]."""
        if not self.entrenado:
            n = 1 if np.asarray(X).ndim == 1 else len(X)
            return np.tile(np.array([0.34, 0.33, 0.33]), (n, 1))
        Z = normalizar_rasgos(X)
        return self.modelo.predict_proba(Z)

    def predecir_una(self, x: np.ndarray) -> Tuple[int, float, np.ndarray]:
        p = self.predecir(x.reshape(1, -1))[0]
        return int(p.argmax()), float(p.max()), p

    def explicar(self, x: np.ndarray) -> Dict[str, object]:
        """Trazabilidad RF-08: entrada cruda -> normalizada -> capas -> softmax."""
        z = normalizar_rasgos(x.reshape(1, -1))[0]
        info: Dict[str, object] = {
            "rasgos": {k: float(v) for k, v in zip(RASGOS, np.asarray(x).ravel())},
            "normalizado": {k: float(v) for k, v in zip(RASGOS, z)},
        }
        if self.entrenado:
            a = z.reshape(1, -1)
            activaciones = []
            for W, b in zip(self.modelo.coefs_[:-1], self.modelo.intercepts_[:-1]):
                a = np.maximum(0.0, a @ W + b)
                activaciones.append({
                    "n_neuronas": int(a.shape[1]),
                    "activas": int((a > 0).sum()),
                    "media": float(a.mean()), "max": float(a.max()),
                })
            info["capas_ocultas"] = activaciones
            p = self.predecir_una(np.asarray(x))[2]
            info["probabilidades"] = {NOMBRE_CLASE[c]: float(p[c]) for c in CLASES}
            info["prediccion"] = NOMBRE_CLASE[int(p.argmax())]
            info["confianza"] = float(p.max())
        return info

    def predecir_vecindario(self, rasgos: Dict[Tuple[int, int], np.ndarray]
                            ) -> Dict[Tuple[int, int], np.ndarray]:
        if not rasgos:
            return {}
        celdas = list(rasgos.keys())
        X = np.array([rasgos[c] for c in celdas])
        P = self.predecir(X)
        return {c: P[i] for i, c in enumerate(celdas)}

## 5 · Controlador de lógica difusa

Implementación **desde cero** de un controlador Mamdani: fusificación → activación de reglas →
implicación (min) → agregación (max) → desfusificación por **centroide**. Sin `scikit-fuzzy`,
para que cada etapa intermedia quede expuesta e inspeccionable.

**Cuatro variables de entrada × tres conjuntos lingüísticos** (RF-05), con funciones triangulares,
trapezoidales y gaussiana. Dos salidas: `velocidad` (parar/lenta/moderada/rápida) y `cautela`.

Sobre la variable térmica: se usa el **margen térmico en °C** (distancia de la temperatura interna
al límite más cercano del hardware) en lugar de la temperatura absoluta. La razón es que el
universo de discurso debe ser comparable entre planetas: 464 °C en Venus y −65 °C en Marte pueden
ser igualmente seguros o igualmente letales según el hardware simulado. La variable sigue estando
en grados Celsius, como pide el enunciado.

**Dieciséis reglas** justificadas una a una. Siete de ellas llevan el complemento difuso
`NO crítico` (negación estándar μ̄ = 1 − μ). Esto resuelve un conflicto real que apareció al
probar el sistema: con margen térmico crítico, la regla de avance rápido y la de parar se
disparaban ambas con grado 1, y el centroide devolvía 0,65 —una recomendación insegura—. Con la
guarda de inhibición, la misma entrada produce 0,07.

**Cobertura, verificada por salida y no por regla.** Un barrido que sólo cuente «reglas activas»
no basta: R11 se activa en muchas situaciones pero únicamente fija la cautela, de modo que la
*velocidad* puede quedar sin ninguna regla. Al probar el sistema apareció exactamente ese caso —
pendiente fuerte con el camino despejado— y el rover se congelaba en terreno simplemente
inclinado. R15 y R16 cierran el hueco; `cobertura_salidas()` comprueba que ninguna combinación
del espacio de entrada deja vacía el área agregada de velocidad o de cautela.

**Condición de borde**: si ninguna regla se activa, el área agregada es nula y el centroide no
está definido. La salida de respaldo documentada es **velocidad 0 y cautela 1** (detención
conservadora), nunca un valor indefinido. El laboratorio de la sección 11 permite desactivar
reglas para provocar y observar esa condición.

In [ ]:
"""
ARES-X · Módulo 5: Controlador de lógica difusa (Mamdani min-max + centroide).

Implementado desde cero (sin scikit-fuzzy) para que cada etapa sea auditable:
fusificación -> activación de reglas -> implicación -> agregación -> centroide.

Variables de entrada (4 variables x 3 conjuntos lingüísticos = RF-05):
  distancia_m        [0, 20]    cerca / media / lejos
  bateria_pct        [0, 100]   critica / baja / alta
  pendiente_deg      [0, 45]    plana / media / fuerte
  margen_termico_C   [-20, 80]  critico / ajustado / holgado
      (distancia en °C entre la temperatura interna y el límite más próximo
       del hardware; se usa el margen en lugar de la temperatura absoluta para
       que el universo de discurso sea comparable entre planetas: 464 °C en
       Venus y -65 °C en Marte pueden ser igualmente seguros o letales según
       el hardware simulado.)

Variables de salida:
  velocidad [0, 1]   parar / lenta / moderada / rapida
  cautela   [0, 1]   baja / media / alta
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np


# ----------------------------------------------------------------------------
# Funciones de membresía
# ----------------------------------------------------------------------------
def mf_triangular(x, a, b, c):
    x = np.asarray(x, dtype=float)
    izq = np.divide(x - a, max(b - a, 1e-9), out=np.zeros_like(x), where=True)
    der = np.divide(c - x, max(c - b, 1e-9), out=np.zeros_like(x), where=True)
    return np.clip(np.minimum(izq, der), 0.0, 1.0)


def mf_trapezoidal(x, a, b, c, d):
    x = np.asarray(x, dtype=float)
    izq = np.divide(x - a, max(b - a, 1e-9), out=np.ones_like(x), where=True)
    der = np.divide(d - x, max(d - c, 1e-9), out=np.ones_like(x), where=True)
    if b - a < 1e-9:
        izq = np.where(x >= a, 1.0, 0.0)
    if d - c < 1e-9:
        der = np.where(x <= d, 1.0, 0.0)
    return np.clip(np.minimum(np.minimum(izq, der), 1.0), 0.0, 1.0)


def mf_gaussiana(x, mu, sigma):
    x = np.asarray(x, dtype=float)
    return np.exp(-0.5 * ((x - mu) / max(sigma, 1e-9)) ** 2)


@dataclass
class ConjuntoDifuso:
    nombre: str
    tipo: str                     # 'tri' | 'trap' | 'gauss'
    params: Tuple[float, ...]

    def mu(self, x):
        if self.tipo == "tri":
            return mf_triangular(x, *self.params)
        if self.tipo == "trap":
            return mf_trapezoidal(x, *self.params)
        return mf_gaussiana(x, *self.params)


@dataclass
class VariableDifusa:
    nombre: str
    minimo: float
    maximo: float
    conjuntos: Dict[str, ConjuntoDifuso]
    unidad: str = ""

    def universo(self, n: int = 201) -> np.ndarray:
        return np.linspace(self.minimo, self.maximo, n)

    def fusificar(self, valor: float) -> Dict[str, float]:
        """Grados de pertenencia. Añade los complementos 'NO <conjunto>'
        (negación difusa estándar: mu_NO(x) = 1 - mu(x)), necesarios para
        redactar reglas de inhibición sin duplicar conjuntos."""
        v = clamp(float(valor), self.minimo, self.maximo)
        base = {k: float(c.mu(v)) for k, c in self.conjuntos.items()}
        base.update({f"NO {k}": 1.0 - m for k, m in list(base.items())})
        return base

    def curvas(self, n: int = 201) -> Tuple[np.ndarray, Dict[str, np.ndarray]]:
        u = self.universo(n)
        return u, {k: np.asarray(c.mu(u), dtype=float) for k, c in self.conjuntos.items()}


# ----------------------------------------------------------------------------
# Reglas
# ----------------------------------------------------------------------------
@dataclass
class Regla:
    id: int
    antecedentes: List[Tuple[str, str]]     # [(variable, conjunto), ...]
    operador: str                           # 'AND' (min) | 'OR' (max)
    consecuentes: List[Tuple[str, str]]     # [(salida, conjunto), ...]
    peso: float = 1.0
    justificacion: str = ""

    def texto(self) -> str:
        ant = f" {self.operador} ".join(
            (f"{v} NO ES {c[3:]}" if c.startswith("NO ") else f"{v} ES {c}")
            for v, c in self.antecedentes)
        con = " Y ".join(f"{v} ES {c}" for v, c in self.consecuentes)
        return f"R{self.id:02d}: SI {ant} ENTONCES {con}"


# ----------------------------------------------------------------------------
# Traza de inferencia
# ----------------------------------------------------------------------------
@dataclass
class TrazaDifusa:
    entradas: Dict[str, float] = field(default_factory=dict)
    membresias: Dict[str, Dict[str, float]] = field(default_factory=dict)
    activaciones: List[dict] = field(default_factory=list)
    salidas: Dict[str, float] = field(default_factory=dict)
    agregados: Dict[str, np.ndarray] = field(default_factory=dict)
    fallback: bool = False
    n_reglas_activas: int = 0

    def reglas_activas(self, umbral: float = 1e-3) -> List[dict]:
        return [a for a in self.activaciones if a["grado"] > umbral]

    def resumen_texto(self, top: int = 4) -> str:
        act = sorted(self.reglas_activas(), key=lambda a: -a["grado"])[:top]
        if not act:
            return "Sin reglas activas -> salida de respaldo (velocidad 0, cautela 1)."
        return " | ".join(f"R{a['id']:02d}({a['grado']:.2f})→{a['consecuente']}"
                          for a in act)


# ----------------------------------------------------------------------------
# Controlador
# ----------------------------------------------------------------------------
class FuzzyController:
    """Mamdani min-max con desfusificación por centroide."""

    def __init__(self, resolucion: int = 201):
        self.res = resolucion
        self.entradas = self._definir_entradas()
        self.salidas = self._definir_salidas()
        self.reglas = self._definir_reglas()
        self.ultima_traza: Optional[TrazaDifusa] = None
        # Permite desactivar reglas desde el laboratorio difuso para demostrar
        # la condición de borde "ninguna regla se activa" (salida de respaldo).
        self.reglas_habilitadas: Optional[set] = None

    # -- definición ----------------------------------------------------------
    @staticmethod
    def _definir_entradas() -> Dict[str, VariableDifusa]:
        return {
            "distancia_m": VariableDifusa(
                "distancia_m", 0.0, 20.0, {
                    "cerca": ConjuntoDifuso("cerca", "trap", (0.0, 0.0, 3.0, 7.5)),
                    "media": ConjuntoDifuso("media", "tri", (4.5, 10.0, 15.0)),
                    "lejos": ConjuntoDifuso("lejos", "trap", (12.0, 16.5, 20.0, 20.0)),
                }, unidad="m"),
            "bateria_pct": VariableDifusa(
                "bateria_pct", 0.0, 100.0, {
                    "critica": ConjuntoDifuso("critica", "trap", (0.0, 0.0, 8.0, 20.0)),
                    "baja": ConjuntoDifuso("baja", "tri", (12.0, 32.0, 55.0)),
                    "alta": ConjuntoDifuso("alta", "trap", (45.0, 70.0, 100.0, 100.0)),
                }, unidad="%"),
            "pendiente_deg": VariableDifusa(
                "pendiente_deg", 0.0, 45.0, {
                    "plana": ConjuntoDifuso("plana", "trap", (0.0, 0.0, 6.0, 14.0)),
                    "media": ConjuntoDifuso("media", "tri", (10.0, 20.0, 30.0)),
                    "fuerte": ConjuntoDifuso("fuerte", "trap", (25.0, 34.0, 45.0, 45.0)),
                }, unidad="°"),
            "margen_termico_C": VariableDifusa(
                "margen_termico_C", -20.0, 80.0, {
                    "critico": ConjuntoDifuso("critico", "trap", (-20.0, -20.0, 0.0, 12.0)),
                    "ajustado": ConjuntoDifuso("ajustado", "gauss", (22.0, 9.0)),
                    "holgado": ConjuntoDifuso("holgado", "trap", (30.0, 50.0, 80.0, 80.0)),
                }, unidad="°C"),
        }

    @staticmethod
    def _definir_salidas() -> Dict[str, VariableDifusa]:
        return {
            "velocidad": VariableDifusa(
                "velocidad", 0.0, 1.0, {
                    "parar": ConjuntoDifuso("parar", "trap", (0.0, 0.0, 0.04, 0.14)),
                    "lenta": ConjuntoDifuso("lenta", "tri", (0.06, 0.24, 0.42)),
                    "moderada": ConjuntoDifuso("moderada", "tri", (0.34, 0.55, 0.76)),
                    "rapida": ConjuntoDifuso("rapida", "trap", (0.66, 0.86, 1.0, 1.0)),
                }),
            "cautela": VariableDifusa(
                "cautela", 0.0, 1.0, {
                    "baja": ConjuntoDifuso("baja", "trap", (0.0, 0.0, 0.15, 0.35)),
                    "media": ConjuntoDifuso("media", "tri", (0.25, 0.50, 0.75)),
                    "alta": ConjuntoDifuso("alta", "trap", (0.65, 0.85, 1.0, 1.0)),
                }),
        }

    @staticmethod
    def _definir_reglas() -> List[Regla]:
        R = []
        a = R.append
        a(Regla(1, [("distancia_m", "cerca")], "AND",
                [("velocidad", "parar"), ("cautela", "alta")],
                justificacion="Riesgo inmediato al frente: detenerse domina cualquier "
                              "otra consideración (seguridad primero)."))
        a(Regla(2, [("bateria_pct", "critica")], "AND",
                [("velocidad", "lenta"), ("cautela", "alta")],
                justificacion="Con energía crítica se minimiza el consumo por celda y "
                              "se prioriza llegar al refugio."))
        a(Regla(3, [("pendiente_deg", "fuerte"), ("distancia_m", "media")], "AND",
                [("velocidad", "lenta"), ("cautela", "alta")],
                justificacion="Pendiente fuerte con obstáculo a media distancia: "
                              "riesgo de vuelco y de frenado insuficiente."))
        a(Regla(4, [("bateria_pct", "alta"), ("pendiente_deg", "plana"),
                    ("distancia_m", "lejos"), ("margen_termico_C", "NO critico")], "AND",
                [("velocidad", "rapida"), ("cautela", "baja")],
                justificacion="Condiciones óptimas: se aprovecha para cubrir distancia."))
        a(Regla(5, [("distancia_m", "media"), ("pendiente_deg", "plana"),
                    ("margen_termico_C", "NO critico")], "AND",
                [("velocidad", "moderada"), ("cautela", "media")],
                justificacion="Terreno despejado pero con obstáculo a la vista."))
        a(Regla(6, [("margen_termico_C", "critico")], "AND",
                [("velocidad", "parar"), ("cautela", "alta")],
                justificacion="Fuera de tolerancia térmica: mover el rover acelera el "
                              "daño; conviene detener y proteger la electrónica."))
        a(Regla(7, [("margen_termico_C", "ajustado"), ("bateria_pct", "alta")], "AND",
                [("velocidad", "moderada"), ("cautela", "media")],
                justificacion="Margen térmico ajustado: avanzar sin exigir el sistema."))
        a(Regla(8, [("pendiente_deg", "media"), ("bateria_pct", "baja")], "AND",
                [("velocidad", "lenta"), ("cautela", "media")],
                justificacion="Subida con poca energía: el consumo crece con la pendiente."))
        a(Regla(9, [("distancia_m", "lejos"), ("bateria_pct", "baja"),
                    ("pendiente_deg", "plana")], "AND",
                [("velocidad", "moderada"), ("cautela", "media")],
                justificacion="Camino despejado y llano: se puede avanzar pese a la "
                              "energía limitada."))
        a(Regla(10, [("pendiente_deg", "fuerte"), ("bateria_pct", "critica")], "AND",
                [("velocidad", "parar"), ("cautela", "alta")],
                justificacion="Peor combinación energética: una subida puede dejar al "
                              "rover sin reserva para volver."))
        a(Regla(11, [("distancia_m", "cerca"), ("pendiente_deg", "fuerte")], "OR",
                [("cautela", "alta")],
                justificacion="Cualquiera de los dos factores por sí solo justifica "
                              "máxima cautela (regla disyuntiva)."))
        a(Regla(12, [("distancia_m", "media"), ("bateria_pct", "alta"),
                     ("margen_termico_C", "holgado")], "AND",
                [("velocidad", "moderada"), ("cautela", "baja")],
                justificacion="Ventana operativa amplia con obstáculo lejano aún "
                              "visible: avance sostenido pero no máximo."))
        a(Regla(13, [("distancia_m", "media"), ("margen_termico_C", "critico")], "AND",
                [("velocidad", "parar"), ("cautela", "alta")],
                justificacion="Evita exponer el hardware mientras se maniobra."))
        a(Regla(14, [("pendiente_deg", "media"), ("distancia_m", "lejos"),
                     ("margen_termico_C", "holgado"),
                     ("bateria_pct", "NO critica")], "AND",
                [("velocidad", "moderada"), ("cautela", "media")],
                justificacion="Pendiente moderada sin obstáculos: avance controlado."))
        # R15 y R16 cierran un hueco de cobertura detectado al probar el sistema:
        # con pendiente fuerte o media y el camino despejado no se activaba
        # NINGUNA regla de velocidad. El área agregada quedaba vacía y el
        # controlador devolvía su salida de respaldo (velocidad 0), de modo que
        # el rover se congelaba en terreno simplemente inclinado. Un barrido que
        # sólo cuente "reglas activas" no detecta el problema, porque R11 se
        # activa pero únicamente fija la cautela: hay que verificar la cobertura
        # de CADA variable de salida por separado.
        a(Regla(15, [("pendiente_deg", "fuerte"), ("distancia_m", "lejos"),
                     ("margen_termico_C", "NO critico")], "AND",
                [("velocidad", "lenta"), ("cautela", "alta")],
                justificacion="Terreno muy inclinado pero despejado: se avanza, "
                              "despacio, por el riesgo de vuelco y deslizamiento."))
        a(Regla(16, [("pendiente_deg", "media"), ("distancia_m", "lejos"),
                     ("margen_termico_C", "NO critico")], "AND",
                [("velocidad", "moderada"), ("cautela", "media")],
                justificacion="Pendiente moderada y camino despejado: avance "
                              "sostenido sin exigir un margen térmico holgado."))
        return R

    # -- inferencia ----------------------------------------------------------
    def inferir(self, distancia_m: float, bateria_pct: float, pendiente_deg: float,
                margen_termico_C: float) -> TrazaDifusa:
        tz = TrazaDifusa()
        crudas = {
            "distancia_m": distancia_m, "bateria_pct": bateria_pct,
            "pendiente_deg": pendiente_deg, "margen_termico_C": margen_termico_C,
        }
        # Entradas inválidas (NaN) -> se asume el peor caso (seguridad)
        peor_caso = {"distancia_m": 0.0, "bateria_pct": 0.0,
                     "pendiente_deg": 45.0, "margen_termico_C": -20.0}
        for k, v in crudas.items():
            if not es_valido(v):
                crudas[k] = peor_caso[k]
                tz.fallback = True
        tz.entradas = {k: float(v) for k, v in crudas.items()}

        for nombre, var in self.entradas.items():
            tz.membresias[nombre] = var.fusificar(crudas[nombre])

        # Agregación por salida (máximo de las implicaciones recortadas)
        universos = {s: v.universo(self.res) for s, v in self.salidas.items()}
        agregados = {s: np.zeros(self.res) for s in self.salidas}

        for r in self.reglas:
            if self.reglas_habilitadas is not None and r.id not in self.reglas_habilitadas:
                tz.activaciones.append({
                    "id": r.id, "texto": r.texto(), "grado": 0.0,
                    "antecedentes": {}, "consecuente": "(regla desactivada)",
                    "justificacion": r.justificacion})
                continue
            grados = [tz.membresias[v][c] for v, c in r.antecedentes]
            grado = (min(grados) if r.operador == "AND" else max(grados)) * r.peso
            tz.activaciones.append({
                "id": r.id, "texto": r.texto(), "grado": float(grado),
                "antecedentes": {f"{v}.{c}": float(tz.membresias[v][c])
                                 for v, c in r.antecedentes},
                "consecuente": ", ".join(f"{v}={c}" for v, c in r.consecuentes),
                "justificacion": r.justificacion,
            })
            if grado <= 1e-9:
                continue
            for salida, conjunto in r.consecuentes:
                mu = np.asarray(self.salidas[salida].conjuntos[conjunto].mu(
                    universos[salida]), dtype=float)
                agregados[salida] = np.maximum(agregados[salida],
                                               np.minimum(grado, mu))

        tz.n_reglas_activas = len(tz.reglas_activas())

        # Desfusificación por centroide
        for salida in self.salidas:
            agr = agregados[salida]
            u = universos[salida]
            area = float(np.trapezoid(agr, u)) if hasattr(np, "trapezoid") \
                else float(np.trapz(agr, u))
            if area <= 1e-9:
                # Condición de borde: ninguna regla se activó.
                # Respaldo documentado y conservador: no moverse, máxima cautela.
                tz.salidas[salida] = 0.0 if salida == "velocidad" else 1.0
                tz.fallback = True
            else:
                num = float(np.trapezoid(agr * u, u)) if hasattr(np, "trapezoid") \
                    else float(np.trapz(agr * u, u))
                tz.salidas[salida] = float(num / area)
            tz.agregados[salida] = agr

        self.ultima_traza = tz
        return tz

    # -- utilidades ----------------------------------------------------------
    def superficie(self, var_x: str, var_y: str, salida: str = "velocidad",
                   fijos: Optional[Dict[str, float]] = None,
                   n: int = 25) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Superficie de control para dos entradas (las demás fijas)."""
        fijos = fijos or {"bateria_pct": 70.0, "margen_termico_C": 45.0,
                          "pendiente_deg": 8.0, "distancia_m": 15.0}
        xs = np.linspace(self.entradas[var_x].minimo, self.entradas[var_x].maximo, n)
        ys = np.linspace(self.entradas[var_y].minimo, self.entradas[var_y].maximo, n)
        Z = np.zeros((n, n))
        for i, yv in enumerate(ys):
            for j, xv in enumerate(xs):
                args = dict(fijos)
                args[var_x] = xv
                args[var_y] = yv
                Z[i, j] = self.inferir(**args).salidas[salida]
        return xs, ys, Z

    def cobertura_salidas(self, n: int = 9) -> Dict[str, int]:
        """Combinaciones del espacio de entrada en las que el área agregada de
        CADA salida queda vacía (y por tanto se aplica el respaldo).

        Contar sólo "reglas activas" es insuficiente: una regla que únicamente
        fija la cautela deja la velocidad sin cobertura.
        """
        import itertools
        rejillas = [np.linspace(v.minimo, v.maximo, n) for v in self.entradas.values()]
        vacios = {s: 0 for s in self.salidas}
        total = 0
        for combo in itertools.product(*rejillas):
            tz = self.inferir(*combo)
            total += 1
            for s, agr in tz.agregados.items():
                if float(np.max(agr)) <= 1e-9:
                    vacios[s] += 1
        vacios["_total"] = total
        return vacios

    def tabla_reglas(self) -> List[dict]:
        return [{"id": r.id, "regla": r.texto(), "operador": r.operador,
                 "justificacion": r.justificacion} for r in self.reglas]


def margen_termico(temp_interna_C: float, limites: Tuple[float, float]) -> float:
    """Distancia (°C) al límite térmico más cercano del hardware."""
    if not es_valido(temp_interna_C):
        return -20.0
    lo, hi = limites
    return clamp(min(temp_interna_C - lo, hi - temp_interna_C), -20.0, 80.0)

## 6 · Planificador por algoritmo genético

El cromosoma es **mixto** e integra las tres familias de subproblemas del curso (el enunciado
pide al menos dos):

| Gen | Contenido | Familia |
|---|---|---|
| `orden[]` | permutación de sitios científicos pendientes | **viajero** |
| `n_visitas` | cuántos se intentan en este plan | decisión de alcance |
| `instrumentos[]` | máscara binaria bajo límites de masa y potencia | **mochila** |
| `waypoints[]` | K celdas de guiado | **laberinto** (camino contiguo + reparación) |

**El decodificador es deliberadamente subóptimo.** Cada tramo se recorre con un avance voraz con
evitación local y tabú, no con un A\* óptimo. Si el decodificador fuera óptimo, la elección de
waypoints del AG sería irrelevante y el algoritmo genético quedaría decorativo: el A\* haría
todo el trabajo. Con un decodificador voraz, el orden de visitas y los waypoints cambian
realmente la calidad de la ruta.

**El AG evalúa siempre contra el mapa de riesgo ESTIMADO**, nunca contra la verdad del terreno:
planificar sobre el futuro conocido sería hacer trampa.

`fitness = w₁·ciencia − w₂·distancia − w₃·energía − w₄·riesgo − w₅·tiempo − P(inviabilidad)`
con todas las magnitudes normalizadas y los pesos configurables desde la interfaz.
La estimación energética está alineada con la física real del simulador (consumo ocioso +
tracción proporcional a velocidad y pendiente + instrumentos − recarga solar observada); cuando
no lo estaba, el AG consideraba inviable cualquier ruta larga y el rover se negaba a explorar.

**Restricciones duras**: atravesar precipicio confirmado, zona prohibida, salir del mapa,
energía insuficiente para la reserva y exceso de masa o potencia en la mochila.

Operadores: torneo, **OX** (preserva orden relativo) para la permutación, uniforme para la
mochila, mezcla aritmética para los waypoints, mutaciones por intercambio / bit-flip / jitter,
elitismo y parada temprana por falta de mejora.

In [ ]:
"""
ARES-X · Módulo 6: Planificador por Algoritmo Genético.

Cromosoma mixto que integra las TRES familias de subproblemas vistas en el curso
(RF-07 / 10.3):

  orden[]        permutación de los sitios científicos pendientes  -> VIAJERO
  n_visitas      cuántos de ellos se intentan en este plan         -> (decisión)
  instrumentos[] máscara binaria bajo límites de masa y potencia   -> MOCHILA
  waypoints[]    K celdas intermedias que moldean la trayectoria   -> LABERINTO
                 (la decodificación debe encontrar un camino contiguo y
                  reparar o penalizar los tramos inviables)

El AG evalúa SIEMPRE contra el mapa de riesgo ESTIMADO (BeliefMap), nunca contra
la verdad del terreno: planificar sobre el futuro conocido sería hacer trampa.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np


Celda = Tuple[int, int]


# ----------------------------------------------------------------------------
# Cromosoma y ruta decodificada
# ----------------------------------------------------------------------------
@dataclass
class Cromosoma:
    orden: List[int]                 # permutación de índices de sitios candidatos
    n_visitas: int                   # 0..len(orden)
    instrumentos: List[int]          # máscara binaria sobre INSTRUMENTOS_BASE
    waypoints: List[Celda]           # K celdas de guiado

    def copia(self) -> "Cromosoma":
        return Cromosoma(list(self.orden), int(self.n_visitas),
                         list(self.instrumentos), [tuple(w) for w in self.waypoints])

    def genes_texto(self) -> str:
        inst = "".join(str(b) for b in self.instrumentos)
        wp = ",".join(f"({f},{c})" for f, c in self.waypoints)
        return (f"orden={self.orden[:self.n_visitas]}|visitas={self.n_visitas}"
                f"|instr={inst}|wp=[{wp}]")


@dataclass
class RutaCandidata:
    cromosoma: Cromosoma
    camino: List[Celda]
    fitness: float = -1e9
    componentes: Dict[str, float] = field(default_factory=dict)
    factible: bool = True
    motivos: List[str] = field(default_factory=list)
    sitios_alcanzados: List[int] = field(default_factory=list)
    masa_usada: float = 0.0
    potencia_usada: float = 0.0
    energia_estimada: float = 0.0
    riesgo_max: float = 0.0

    def resumen(self) -> str:
        est = "FACTIBLE" if self.factible else "INVIABLE"
        return (f"fitness={self.fitness:+.3f} [{est}] celdas={len(self.camino)} "
                f"ciencia={self.componentes.get('ciencia', 0):.2f} "
                f"riesgo={self.componentes.get('riesgo', 0):.2f}")


@dataclass
class TrazaAG:
    """Historial completo para el Laboratorio Genético (RF-11, CA-04)."""
    mejor_por_gen: List[float] = field(default_factory=list)
    media_por_gen: List[float] = field(default_factory=list)
    peor_por_gen: List[float] = field(default_factory=list)
    diversidad_por_gen: List[float] = field(default_factory=list)
    snapshots: List[List[RutaCandidata]] = field(default_factory=list)  # top-3/gen
    eventos: List[dict] = field(default_factory=list)   # cruces y mutaciones
    torneos: List[dict] = field(default_factory=list)
    n_cruces: int = 0
    n_mutaciones: int = 0
    n_reparaciones: int = 0
    generaciones_ejecutadas: int = 0
    parada_temprana: bool = False
    segundos: float = 0.0


# ----------------------------------------------------------------------------
# Planificador
# ----------------------------------------------------------------------------
class GeneticPlanner:

    def __init__(self, cfg: Config, instrumentos: Optional[List[Instrumento]] = None):
        self.cfg = cfg
        self.instrumentos = instrumentos or INSTRUMENTOS_BASE
        self.traza: Optional[TrazaAG] = None
        self.ultimas_candidatas: List[RutaCandidata] = []
        self._cache: Dict[tuple, Optional[List[Celda]]] = {}
        self._llamadas = 0

    # ------------------------------------------------------------------ #
    # Decodificación: cromosoma -> camino contiguo
    # ------------------------------------------------------------------ #
    def _caminar(self, belief: BeliefMap, a: Celda, b: Celda,
                 max_pasos: int = 60) -> Optional[List[Celda]]:
        """Avance voraz con evitación local y tabú (decodificador de tramo).

        Devuelve la lista de celdas (sin incluir `a`) o None si no alcanza `b`.
        Es deliberadamente SUBÓPTIMO: así la elección de waypoints y del orden
        de visitas por parte del AG tiene un efecto real sobre la calidad de la
        ruta (si el decodificador fuera un A* óptimo, el AG sería decorativo).
        """
        clave = (a, b)
        if clave in self._cache:
            c = self._cache[clave]
            return list(c) if c is not None else None

        camino: List[Celda] = []
        actual = a
        recientes: Dict[Celda, int] = {}
        riesgo = belief.mapa_riesgo()
        for _ in range(max_pasos):
            if actual == b:
                break
            mejor, mejor_coste = None, float("inf")
            for d in DIRECCIONES_8:
                q = (actual[0] + d[0], actual[1] + d[1])
                if not belief.dentro(q):
                    continue
                if belief.bloqueada(q):
                    continue
                coste = (distancia_cheb(q, b)
                         + 4.0 * float(riesgo[q[0], q[1]])
                         + 2.5 * recientes.get(q, 0))
                if coste < mejor_coste:
                    mejor, mejor_coste = q, coste
            if mejor is None:
                self._cache[clave] = None
                return None
            recientes[mejor] = recientes.get(mejor, 0) + 1
            camino.append(mejor)
            actual = mejor
        resultado = camino if actual == b else None
        self._cache[clave] = list(resultado) if resultado is not None else None
        return list(resultado) if resultado is not None else None

    def decodificar(self, crom: Cromosoma, belief: BeliefMap, pos: Celda,
                    sitios: List[SitioCiencia], refugios: List[Celda]
                    ) -> Tuple[List[Celda], List[int], List[str], int]:
        """Construye el camino contiguo y repara/penaliza lo inviable."""
        camino: List[Celda] = [pos]
        alcanzados: List[int] = []
        motivos: List[str] = []
        reparaciones = 0
        actual = pos

        objetivos: List[Celda] = []
        for k, idx in enumerate(crom.orden[:crom.n_visitas]):
            if k < len(crom.waypoints):
                wp = crom.waypoints[k]
                # REPARACIÓN: un waypoint sobre celda creída bloqueada se
                # reubica al vecino libre más cercano en lugar de invalidar.
                if belief.bloqueada(wp):
                    alt = self._vecino_libre(belief, wp)
                    reparaciones += 1
                    if alt is not None:
                        wp = alt
                    else:
                        wp = None
                if wp is not None and wp != actual:
                    objetivos.append(("wp", wp))
            objetivos.append(("sitio", sitios[idx].pos, idx))

        # Destino final: refugio alcanzable más cercano al último objetivo
        if refugios:
            ref = min(refugios, key=lambda r: distancia_cheb(
                objetivos[-1][1] if objetivos else pos, r))
            objetivos.append(("refugio", ref))

        for obj in objetivos:
            destino = obj[1]
            tramo = self._caminar(belief, actual, destino)
            if tramo is None:
                motivos.append(f"tramo_inviable:{obj[0]}@{destino}")
                break
            camino.extend(tramo)
            actual = destino
            if obj[0] == "sitio":
                alcanzados.append(obj[2])
            if len(camino) > 160:          # tope de longitud de plan
                motivos.append("ruta_excede_longitud_maxima")
                break

        return camino, alcanzados, motivos, reparaciones

    @staticmethod
    def _vecino_libre(belief: BeliefMap, p: Celda) -> Optional[Celda]:
        for radio in (1, 2):
            for d in DIRECCIONES_8:
                q = (p[0] + d[0] * radio, p[1] + d[1] * radio)
                if belief.dentro(q) and not belief.bloqueada(q):
                    return q
        return None

    # ------------------------------------------------------------------ #
    # Evaluación
    # ------------------------------------------------------------------ #
    def evaluar(self, crom: Cromosoma, belief: BeliefMap, pos: Celda,
                sitios: List[SitioCiencia], refugios: List[Celda],
                bateria: float, velocidad_media: float = 0.55,
                recarga_por_celda: float = 0.30) -> RutaCandidata:
        pesos = self.cfg.ag_pesos
        camino, alcanzados, motivos, reps = self.decodificar(
            crom, belief, pos, sitios, refugios)
        ruta = RutaCandidata(cromosoma=crom, camino=camino, motivos=list(motivos))

        # --- MOCHILA: masa y potencia de los instrumentos seleccionados -----
        masa = sum(self.instrumentos[i].masa_kg
                   for i, b in enumerate(crom.instrumentos) if b)
        pot = sum(self.instrumentos[i].potencia_W
                  for i, b in enumerate(crom.instrumentos) if b)
        ruta.masa_usada, ruta.potencia_usada = masa, pot
        exceso = max(0.0, masa - self.cfg.masa_max_kg) / max(self.cfg.masa_max_kg, 1e-6) \
            + max(0.0, pot - self.cfg.potencia_max_W) / max(self.cfg.potencia_max_W, 1e-6)
        if exceso > 0:
            ruta.motivos.append("mochila_excede_limites")
        habilitadas = {self.instrumentos[i].habilita
                       for i, b in enumerate(crom.instrumentos) if b}

        # --- CIENCIA --------------------------------------------------------
        valor_total = sum(s.valor for s in sitios) or 1.0
        ciencia = 0.0
        for idx in alcanzados:
            s = sitios[idx]
            if s.requiere in habilitadas:
                ciencia += s.valor * (1.6 if s.prioritario else 1.0)
            else:
                ciencia += 0.12 * s.valor       # observación remota parcial
                ruta.motivos.append(f"sin_instrumento:{s.requiere}")
        ciencia = clamp(ciencia / valor_total, 0.0, 1.6) * self.cfg.prioridad_ciencia

        # --- DISTANCIA, RIESGO, ENERGÍA, TIEMPO -----------------------------
        n = max(1, len(camino) - 1)
        distancia = norm01(n, 0.0, 160.0)
        mapa_r = belief.mapa_riesgo()
        riesgos = [float(mapa_r[c[0], c[1]]) for c in camino if belief.dentro(c)]
        riesgo_med = float(np.mean(riesgos)) if riesgos else 1.0
        ruta.riesgo_max = float(np.max(riesgos)) if riesgos else 1.0
        riesgo = 0.65 * riesgo_med + 0.35 * ruta.riesgo_max

        pend = [float(belief.pendiente_est[c[0], c[1]]) for c in camino
                if belief.dentro(c)]
        pend_media = float(np.mean(pend)) if pend else 0.0
        # Estimación energética alineada con la física de actualizar_mundo():
        # consumo ocioso + tracción (proporcional a la velocidad y la pendiente)
        # + instrumentos, menos la recarga media esperada por celda.
        consumo_celda = (0.05 + (0.45 + 0.03 * pend_media) * velocidad_media
                         + 0.004 * pot)                       # [SUPUESTO]
        energia_kwh = n * max(0.05, consumo_celda - recarga_por_celda)
        ruta.energia_estimada = energia_kwh
        energia = norm01(energia_kwh, 0.0, 100.0)
        tiempo = norm01(n / max(velocidad_media, 0.05), 0.0, 400.0)

        # --- RESTRICCIONES DURAS --------------------------------------------
        factible = True
        for c in camino:
            if not belief.dentro(c):
                ruta.motivos.append("sale_del_mapa")
                factible = False
                break
            if belief.prohibido[c[0], c[1]]:
                ruta.motivos.append(f"zona_prohibida@{c}")
                factible = False
                break
            if belief.confirmado[c[0], c[1]] and belief.p_barr[c[0], c[1]] >= 0.99:
                ruta.motivos.append(f"precipicio_confirmado@{c}")
                factible = False
                break
        if any(m.startswith("tramo_inviable") for m in ruta.motivos):
            factible = False
        if energia_kwh > max(0.0, bateria - self.cfg.reserva_energia_pct):
            ruta.motivos.append("energia_insuficiente_para_reserva")
            factible = False
        if exceso > 0:
            factible = False
        ruta.factible = factible

        # --- FITNESS --------------------------------------------------------
        penalizacion = 0.0
        if not factible:
            penalizacion += 1.0
        penalizacion += 0.6 * exceso
        penalizacion += 0.25 * sum(1 for m in ruta.motivos
                                   if m.startswith("tramo_inviable"))
        if not alcanzados and crom.n_visitas > 0:
            penalizacion += 0.15

        fitness = (pesos["ciencia"] * ciencia
                   - pesos["distancia"] * distancia
                   - pesos["energia"] * energia
                   - pesos["riesgo"] * riesgo
                   - pesos["tiempo"] * tiempo
                   - penalizacion)

        ruta.componentes = {
            "ciencia": float(ciencia), "distancia": float(distancia),
            "energia": float(energia), "riesgo": float(riesgo),
            "tiempo": float(tiempo), "penalizacion": float(penalizacion),
            "riesgo_medio": riesgo_med, "riesgo_max": ruta.riesgo_max,
            "aporte_ciencia": float(pesos["ciencia"] * ciencia),
            "aporte_distancia": float(-pesos["distancia"] * distancia),
            "aporte_energia": float(-pesos["energia"] * energia),
            "aporte_riesgo": float(-pesos["riesgo"] * riesgo),
            "aporte_tiempo": float(-pesos["tiempo"] * tiempo),
        }
        ruta.fitness = float(fitness)
        ruta.sitios_alcanzados = alcanzados
        return ruta

    # ------------------------------------------------------------------ #
    # Operadores genéticos
    # ------------------------------------------------------------------ #
    def _individuo_aleatorio(self, rng, n_sitios: int, belief: BeliefMap,
                             pos: Celda) -> Cromosoma:
        orden = list(rng.permutation(n_sitios)) if n_sitios else []
        orden = [int(x) for x in orden]
        # Sesgo hacia planes con visitas: un plan vacío siempre es "seguro"
        # pero no cumple la misión; la población inicial debe explorar ambas.
        n_vis = int(rng.integers(1, n_sitios + 1)) if n_sitios else 0
        instr = [int(rng.random() < 0.45) for _ in self.instrumentos]
        wps = []
        for _ in range(self.cfg.ag_waypoints):
            f = int(clamp(pos[0] + rng.integers(-7, 8), 0, belief.filas - 1))
            c = int(clamp(pos[1] + rng.integers(-7, 8), 0, belief.cols - 1))
            wps.append((f, c))
        return Cromosoma(orden, n_vis, instr, wps)

    @staticmethod
    def _cruce_ox(p1: List[int], p2: List[int], rng) -> List[int]:
        """Order Crossover (OX): preserva el orden relativo (problema viajero)."""
        n = len(p1)
        if n < 2:
            return list(p1)
        i, j = sorted(rng.choice(n, size=2, replace=False))
        hijo = [None] * n
        hijo[i:j + 1] = p1[i:j + 1]
        resto = [g for g in p2 if g not in hijo[i:j + 1]]
        k = 0
        for t in range(n):
            if hijo[t] is None:
                hijo[t] = resto[k]
                k += 1
        return [int(x) for x in hijo]

    def cruzar(self, a: Cromosoma, b: Cromosoma, rng) -> Tuple[Cromosoma, dict]:
        hijo = a.copia()
        hijo.orden = self._cruce_ox(a.orden, b.orden, rng)
        # Uniforme para la mochila
        hijo.instrumentos = [a.instrumentos[i] if rng.random() < 0.5
                             else b.instrumentos[i]
                             for i in range(len(a.instrumentos))]
        # Mezcla aritmética entera para los waypoints
        hijo.waypoints = []
        for wa, wb in zip(a.waypoints, b.waypoints):
            lam = float(rng.random())
            hijo.waypoints.append((int(round(lam * wa[0] + (1 - lam) * wb[0])),
                                   int(round(lam * wa[1] + (1 - lam) * wb[1]))))
        hijo.n_visitas = int(a.n_visitas if rng.random() < 0.5 else b.n_visitas)
        hijo.n_visitas = int(clamp(hijo.n_visitas, 0, len(hijo.orden)))
        return hijo, {"tipo": "cruce", "corte": "OX+uniforme+aritmetico"}

    def mutar(self, c: Cromosoma, rng, belief: BeliefMap) -> Tuple[Cromosoma, List[str]]:
        m = c.copia()
        ops: List[str] = []
        if len(m.orden) >= 2 and rng.random() < 0.6:
            i, j = rng.choice(len(m.orden), size=2, replace=False)
            m.orden[i], m.orden[j] = m.orden[j], m.orden[i]
            ops.append(f"swap_orden({i},{j})")
        if rng.random() < 0.5 and m.instrumentos:
            k = int(rng.integers(0, len(m.instrumentos)))
            m.instrumentos[k] = 1 - m.instrumentos[k]
            ops.append(f"bitflip_instrumento({k})")
        if m.waypoints and rng.random() < 0.7:
            k = int(rng.integers(0, len(m.waypoints)))
            f = int(clamp(m.waypoints[k][0] + rng.integers(-3, 4), 0, belief.filas - 1))
            cc = int(clamp(m.waypoints[k][1] + rng.integers(-3, 4), 0, belief.cols - 1))
            m.waypoints[k] = (f, cc)
            ops.append(f"jitter_waypoint({k})")
        if rng.random() < 0.4:
            m.n_visitas = int(clamp(m.n_visitas + int(rng.integers(-1, 2)),
                                    0, len(m.orden)))
            ops.append("ajuste_n_visitas")
        return m, ops

    def _torneo(self, poblacion: List[RutaCandidata], rng, k: int
                ) -> Tuple[Cromosoma, dict]:
        idx = rng.choice(len(poblacion), size=min(k, len(poblacion)), replace=False)
        competidores = [(int(i), poblacion[int(i)].fitness) for i in idx]
        ganador = max(competidores, key=lambda t: t[1])
        return (poblacion[ganador[0]].cromosoma.copia(),
                {"competidores": competidores, "ganador": ganador[0]})

    # ------------------------------------------------------------------ #
    # Bucle evolutivo
    # ------------------------------------------------------------------ #
    def planificar(self, belief: BeliefMap, pos: Celda, sitios: List[SitioCiencia],
                   refugios: List[Celda], bateria: float, semilla: int,
                   velocidad_media: float = 0.55,
                   recarga_por_celda: float = 0.30,
                   paciencia: int = 10) -> Tuple[RutaCandidata, List[RutaCandidata], TrazaAG]:
        import time as _t
        t0 = _t.time()
        self._cache.clear()
        self._llamadas += 1
        rng = rng_de(semilla, f"ag{self._llamadas}")
        tz = TrazaAG()

        n_sitios = len(sitios)
        P = max(4, int(self.cfg.ag_poblacion))
        G = max(1, int(self.cfg.ag_generaciones))

        poblacion = [self._individuo_aleatorio(rng, n_sitios, belief, pos)
                     for _ in range(P)]
        evaluados = [self.evaluar(c, belief, pos, sitios, refugios, bateria,
                                  velocidad_media, recarga_por_celda)
                     for c in poblacion]
        mejor_global = max(evaluados, key=lambda r: r.fitness)
        sin_mejora = 0

        for g in range(G):
            evaluados.sort(key=lambda r: -r.fitness)
            tz.mejor_por_gen.append(evaluados[0].fitness)
            tz.media_por_gen.append(float(np.mean([r.fitness for r in evaluados])))
            tz.peor_por_gen.append(evaluados[-1].fitness)
            tz.diversidad_por_gen.append(float(np.std([r.fitness for r in evaluados])))
            tz.snapshots.append([r for r in evaluados[:3]])

            if evaluados[0].fitness > mejor_global.fitness + 1e-9:
                mejor_global = evaluados[0]
                sin_mejora = 0
            else:
                sin_mejora += 1
            if sin_mejora >= paciencia and g > 5:
                tz.parada_temprana = True
                tz.generaciones_ejecutadas = g + 1
                break

            # ELITISMO
            nueva: List[Cromosoma] = [evaluados[i].cromosoma.copia()
                                      for i in range(min(self.cfg.ag_elitismo, P))]
            while len(nueva) < P:
                padre, t1 = self._torneo(evaluados, rng, self.cfg.ag_torneo)
                madre, t2 = self._torneo(evaluados, rng, self.cfg.ag_torneo)
                if len(tz.torneos) < 60:
                    tz.torneos.append({"gen": g, "t1": t1, "t2": t2})
                if rng.random() < self.cfg.ag_prob_cruce:
                    hijo, info = self.cruzar(padre, madre, rng)
                    tz.n_cruces += 1
                    if len(tz.eventos) < 120:
                        tz.eventos.append({"gen": g, "op": "cruce",
                                           "padre": padre.genes_texto(),
                                           "madre": madre.genes_texto(),
                                           "hijo": hijo.genes_texto(), **info})
                else:
                    hijo = padre.copia()
                if rng.random() < self.cfg.ag_prob_mutacion:
                    antes = hijo.genes_texto()
                    hijo, ops = self.mutar(hijo, rng, belief)
                    tz.n_mutaciones += 1
                    if len(tz.eventos) < 120:
                        tz.eventos.append({"gen": g, "op": "mutacion", "antes": antes,
                                           "despues": hijo.genes_texto(),
                                           "operadores": ops})
                nueva.append(hijo)

            evaluados = [self.evaluar(c, belief, pos, sitios, refugios, bateria,
                                      velocidad_media, recarga_por_celda)
                         for c in nueva]
            tz.generaciones_ejecutadas = g + 1

        evaluados.sort(key=lambda r: -r.fitness)
        if evaluados[0].fitness > mejor_global.fitness:
            mejor_global = evaluados[0]
        tz.n_reparaciones = sum(1 for e in evaluados if "reparado" in e.motivos)
        tz.segundos = _t.time() - t0

        # Candidatas a inspeccionar (RF-07: al menos tres rutas)
        candidatas = self._diversificar(evaluados, mejor_global, k=3)
        self.ultimas_candidatas = candidatas
        self.traza = tz
        return mejor_global, candidatas, tz

    @staticmethod
    def _diversificar(evaluados: List[RutaCandidata], mejor: RutaCandidata,
                      k: int = 3) -> List[RutaCandidata]:
        """Selecciona k rutas distinguibles (no clones) para el panel."""
        elegidas = [mejor]
        for r in evaluados:
            if len(elegidas) >= k:
                break
            if all(set(r.camino) != set(e.camino) for e in elegidas):
                elegidas.append(r)
        i = 0
        while len(elegidas) < k and i < len(evaluados):
            if evaluados[i] not in elegidas:
                elegidas.append(evaluados[i])
            i += 1
        return elegidas[:k]


# ----------------------------------------------------------------------------
# Planificadores de referencia (baselines para las ablaciones)
# ----------------------------------------------------------------------------
def ruta_mas_corta(belief: BeliefMap, pos: Celda, objetivo: Celda,
                   ignorar_riesgo: bool = True, max_pasos: int = 200
                   ) -> Optional[List[Celda]]:
    """Baseline de control: A* por distancia, sin considerar riesgo estimado."""
    import heapq
    if not belief.dentro(objetivo):
        return None
    abierta = [(0.0, 0.0, pos, [pos])]
    vistos = {}
    while abierta:
        f, g, actual, camino = heapq.heappop(abierta)
        if actual == objetivo:
            return camino
        if len(camino) > max_pasos:
            continue
        if actual in vistos and vistos[actual] <= g:
            continue
        vistos[actual] = g
        for d in DIRECCIONES_8:
            q = (actual[0] + d[0], actual[1] + d[1])
            if not belief.dentro(q) or belief.bloqueada(q):
                continue
            coste = 1.0 if ignorar_riesgo else 1.0 + 4.0 * belief.riesgo_estimado(q)
            ng = g + coste
            heapq.heappush(abierta, (ng + distancia_cheb(q, objetivo), ng, q,
                                     camino + [q]))
    return None

## 7 · Supervisor híbrido de seguridad y bitácora

El supervisor es el **único** componente autorizado a decidir la acción final, y su orden de
evaluación es innegociable: **primero las prohibiciones físicas, después la fusión de
recomendaciones**. Cuando no veta, la acción y la velocidad provienen literalmente de las
salidas del AG y del controlador difuso: no existe ninguna política oculta.

Cada decisión emite un **código de motivo reproducible** (`VETO_BARRANCO`, `VETO_TERMICO`,
`REFUGIO_INALCANZABLE`, …) que queda en la bitácora.

Dos detalles de diseño que surgieron al probar el sistema:

* **El veto de pendiente evalúa la pendiente de la *transición* hacia el destino**, medida por
  altimetría, y no la inclinación del rover en su celda actual. Usar la segunda bloqueaba al
  rover de forma permanente en cuanto se detenía sobre una celda inclinada, porque esa lectura
  no dice nada sobre si el siguiente movimiento es seguro.
* **La reserva energética es dinámica**: la energía necesaria para alcanzar la zona segura más
  cercana más un margen del 50 %, no un porcentaje fijo. Un 12 % de batería puede ser holgado
  junto a una estación y letal a veinte celdas de distancia.

`MissionLogger` registra por paso: señales de todos los sensores, probabilidades de la RNA,
grados de membresía difusos, salida crisp, reglas disparadas, fitness y desglose del AG, genes
del cromosoma elegido, acción, motivo y explicación (RF-08). Exporta CSV y JSON con
configuración, semillas, versiones y supuestos (RF-12).

In [ ]:
"""
ARES-X · Módulo 7: Supervisor híbrido de seguridad y bitácora de misión.

El supervisor es el ÚNICO componente autorizado a decidir la acción final.
Su orden de evaluación es innegociable (sección 10.4 del caso):

  1. Prohibiciones físicas (vetos duros)   -> priman sobre cualquier fitness
  2. Fusión de recomendaciones de IA       -> RNA + difuso + AG
  3. Emisión de acción + código de motivo  -> trazable y reproducible

No existe ninguna "función secreta" que resuelva el problema por su cuenta:
cuando el supervisor no veta, la acción y la velocidad provienen literalmente
de las salidas del AG y del controlador difuso.
"""

from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd


Celda = Tuple[int, int]

ACCIONES = ["AVANZAR", "DETENER", "RECARGAR", "VOLVER", "REPLANIFICAR",
            "MUESTREAR", "MODO_SEGURO"]


@dataclass
class Decision:
    accion: str
    velocidad: float
    motivo: str                       # código de CODIGOS_MOTIVO
    detalle: str = ""
    destino: Optional[Celda] = None
    vetos: List[str] = field(default_factory=list)
    permitidos: List[str] = field(default_factory=list)
    replanificar: bool = False
    prohibir: Optional[Celda] = None

    def explicacion(self) -> str:
        base = CODIGOS_MOTIVO.get(self.motivo, self.motivo)
        extra = f" {self.detalle}" if self.detalle else ""
        return f"[{self.motivo}] {base}{extra}"


class SafetySupervisor:
    """Árbitro de restricciones duras y recomendaciones de IA."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.limites_termicos = cfg.perfil.limite_termico_C
        self.historial_vetos: List[dict] = []

    # ------------------------------------------------------------------ #
    def evaluar(self, estado, lectura, belief: BeliefMap, destino: Optional[Celda],
                velocidad_difusa: float, cautela: float,
                prob_rna: Optional[np.ndarray], confirmacion_borde: Optional[int],
                refugio_alcanzable: bool, hay_sitio_aqui: bool,
                ruta_factible: bool,
                pendiente_destino: float = 0.0,
                reserva_requerida: Optional[float] = None) -> Decision:
        vetos: List[str] = []
        permitidos: List[str] = []
        # Reserva energética DINÁMICA: la energía que hace falta para alcanzar
        # la zona segura más cercana, no un porcentaje fijo. Un 12 % puede ser
        # holgado junto a una estación y letal a veinte celdas de distancia.
        reserva = (self.cfg.reserva_energia_pct if reserva_requerida is None
                   else max(self.cfg.reserva_energia_pct, float(reserva_requerida)))

        # ---------------- 1. PROHIBICIONES FÍSICAS ------------------------
        # 1.1 Energía agotada
        if estado.bateria <= 0.0:
            return self._veto("DETENER", "FIN_FRACASO",
                              "Batería a 0 %: la misión no puede continuar.", vetos)

        # 1.2 Sensor crítico inválido -> degradación segura, nunca "a ciegas"
        sensor_malo = lectura.sensor_critico_invalido()
        if sensor_malo:
            d = self._veto("MODO_SEGURO", "VETO_SENSOR",
                           f"Sensor crítico '{sensor_malo}' inválido (NaN/saturado). "
                           f"Se detiene el avance y se registra la degradación.", vetos)
            d.replanificar = False
            return d

        # 1.3 Límite térmico del hardware
        t = estado.temp_interna
        lo, hi = self.limites_termicos
        if not es_valido(t) or t < lo or t > hi:
            return self._veto("MODO_SEGURO", "VETO_TERMICO",
                              f"Temperatura interna {t:.1f} °C fuera del rango "
                              f"[{lo:.0f}, {hi:.0f}] °C del hardware simulado.", vetos)

        # 1.4 Desgaste mecánico
        if estado.desgaste >= self.cfg.desgaste_max:
            return self._veto("DETENER", "VETO_DESGASTE",
                              f"Desgaste {estado.desgaste:.0f}/100: subsistema de "
                              f"tracción fuera de servicio.", vetos)

        # 1.5 Reserva energética para alcanzar zona segura
        if estado.bateria <= reserva and not refugio_alcanzable:
            return self._veto("DETENER", "REFUGIO_INALCANZABLE",
                              f"Batería {estado.bateria:.1f} % frente a una reserva "
                              f"requerida de {reserva:.1f} % y ningún refugio "
                              f"alcanzable: no se promete un rescate imposible.", vetos)

        # ---------------- 2. VALIDACIÓN DE LA CELDA DESTINO ----------------
        if destino is None:
            return Decision("MUESTREAR" if hay_sitio_aqui else "DETENER",
                            0.0, "OK_MUESTREO" if hay_sitio_aqui else "SIN_RUTA",
                            "No hay celda destino en el plan actual.",
                            vetos=vetos, permitidos=permitidos,
                            replanificar=not hay_sitio_aqui)

        if not belief.dentro(destino):
            d = self._veto("REPLANIFICAR", "VETO_FUERA_MAPA",
                           f"La celda {destino} está fuera del mapa.", vetos)
            d.replanificar = True
            return d

        # Confirmación determinista del sensor de borde (prima sobre la RNA)
        if confirmacion_borde is not None:
            if confirmacion_borde == BARRANCO:
                d = self._veto("REPLANIFICAR", "VETO_BARRANCO",
                               f"Sensor de borde confirma precipicio en {destino}. "
                               f"Se crea zona prohibida y se replanifica.", vetos)
                d.replanificar = True
                d.prohibir = destino
                return d
            if confirmacion_borde == OBSTACULO:
                d = self._veto("REPLANIFICAR", "VETO_COLISION",
                               f"Obstáculo infranqueable confirmado en {destino}.",
                               vetos)
                d.replanificar = True
                d.prohibir = destino
                return d
            permitidos.append("sensor_borde:libre")

        # Pendiente de vuelco: se evalúa la pendiente DE LA TRANSICIÓN hacia la
        # celda destino (medida por altimetría), no la inclinación del rover en
        # su celda actual: esta última no dice nada sobre si el siguiente
        # movimiento es seguro y bloquearía al rover de forma permanente.
        pend = abs(pendiente_destino)
        if es_valido(pend) and pend > self.cfg.pendiente_max_deg:
            d = self._veto("REPLANIFICAR", "VETO_PENDIENTE",
                           f"Pendiente de transición {pend:.1f}° hacia {destino} "
                           f"supera el máximo de vuelco "
                           f"({self.cfg.pendiente_max_deg:.0f}°).", vetos)
            d.replanificar = True
            d.prohibir = destino
            return d

        # Zona prohibida previamente declarada
        if belief.prohibido[destino[0], destino[1]]:
            d = self._veto("REPLANIFICAR", "VETO_RUTA_INVIABLE",
                           f"La celda {destino} está en una zona prohibida activa.",
                           vetos)
            d.replanificar = True
            return d

        # Veto preventivo basado en la RNA (probabilístico, no determinista)
        if prob_rna is not None and len(prob_rna) == 3:
            p_barr = float(prob_rna[BARRANCO])
            if p_barr >= self.cfg.umbral_rna_barranco and confirmacion_borde is None:
                d = self._veto("REPLANIFICAR", "VETO_RNA_BARRANCO",
                               f"La RNA predice barranco con p={p_barr:.2f} "
                               f"(umbral {self.cfg.umbral_rna_barranco:.2f}).", vetos)
                d.replanificar = True
                d.prohibir = destino
                return d
            permitidos.append(f"rna:p_barranco={p_barr:.2f}")

        # Ruta global marcada como inviable por el AG
        if not ruta_factible:
            d = Decision("REPLANIFICAR", 0.0, "VETO_RUTA_INVIABLE",
                         "El plan vigente dejó de ser factible.",
                         destino=destino, vetos=vetos, permitidos=permitidos,
                         replanificar=True)
            self.historial_vetos.append({"motivo": d.motivo, "detalle": d.detalle})
            return d

        # ---------------- 3. FUSIÓN DE RECOMENDACIONES ---------------------
        v = clamp(velocidad_difusa, 0.0, 1.0)
        # La cautela alta recorta la velocidad propuesta (acoplamiento difuso)
        v *= (1.0 - 0.35 * clamp(cautela, 0.0, 1.0))
        # Baja confianza de la RNA -> modo conservador (fusión con sensor)
        if prob_rna is not None and float(np.max(prob_rna)) < 0.55:
            v *= 0.6
            permitidos.append("rna_baja_confianza:modo_conservador")

        if estado.bateria <= reserva and refugio_alcanzable:
            return Decision("VOLVER", max(0.12, v * 0.8), "OK_AVANCE",
                            f"Batería {estado.bateria:.1f} % por debajo de la reserva "
                            f"requerida ({reserva:.1f} %): rumbo al refugio "
                            f"alcanzable.", destino=destino,
                            vetos=vetos, permitidos=permitidos)

        # Una velocidad difusa muy baja significa MARCHA LENTA, no parálisis.
        # Sólo la salida de respaldo del controlador (exactamente 0, cuando no
        # se activa ninguna regla) ordena una detención efectiva. Convertir todo
        # "muy lento" en "detenido" castigaba al propio control difuso: el rover
        # acumulaba esperas estériles hasta declararse sin ruta.
        if velocidad_difusa <= 1e-6:
            return Decision("DETENER", 0.0, "OK_ESPERA",
                            "Salida de respaldo del controlador difuso "
                            "(ninguna regla activa): detención conservadora.",
                            destino=destino, vetos=vetos, permitidos=permitidos)
        VELOCIDAD_MINIMA = 0.08          # [SUPUESTO] marcha de aproximación
        if v < VELOCIDAD_MINIMA:
            v = VELOCIDAD_MINIMA
            permitidos.append("marcha_lenta_minima")

        if hay_sitio_aqui:
            return Decision("MUESTREAR", 0.0, "OK_MUESTREO",
                            "Sitio científico en la celda actual.",
                            destino=destino, vetos=vetos, permitidos=permitidos)

        return Decision("AVANZAR", v, "OK_AVANCE",
                        f"Velocidad difusa {velocidad_difusa:.2f} ajustada por "
                        f"cautela {cautela:.2f} -> {v:.2f}.",
                        destino=destino, vetos=vetos, permitidos=permitidos)

    def _veto(self, accion: str, motivo: str, detalle: str,
              vetos: List[str]) -> Decision:
        vetos.append(motivo)
        self.historial_vetos.append({"motivo": motivo, "detalle": detalle})
        return Decision(accion, 0.0, motivo, detalle, vetos=vetos)

    # ------------------------------------------------------------------ #
    def validar_ruta(self, camino: List[Celda], belief: BeliefMap,
                     bateria: float) -> Tuple[bool, List[str]]:
        """Chequeo duro de una ruta completa propuesta por el AG."""
        motivos: List[str] = []
        for c in camino:
            if not belief.dentro(c):
                motivos.append(f"{c}:fuera_del_mapa")
                break
            if belief.prohibido[c[0], c[1]]:
                motivos.append(f"{c}:zona_prohibida")
                break
            if belief.confirmado[c[0], c[1]] and belief.p_barr[c[0], c[1]] >= 0.99:
                motivos.append(f"{c}:precipicio_confirmado")
                break
        return (len(motivos) == 0, motivos)


# ----------------------------------------------------------------------------
# MissionLogger
# ----------------------------------------------------------------------------
class MissionLogger:
    """RF-08 / RF-12: registra por paso señales, probabilidades RNA, grados
    difusos, salida crisp, fitness, ruta, acción y explicación."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.filas: List[dict] = []
        self.eventos: List[dict] = []
        self.resultados: Dict[str, object] = {}

    def registrar(self, paso: int, estado, lectura, decision: Decision,
                  prob_rna: Optional[np.ndarray], traza_difusa, ruta,
                  world, extra: Optional[dict] = None) -> None:
        fila: Dict[str, object] = {
            "paso": paso,
            "hora_local": round(world.hora, 3),
            "es_noche": world.es_noche(),
            "estado_mision": estado.estado,
            "pos_f": estado.pos[0], "pos_c": estado.pos[1],
            "rumbo": estado.rumbo_idx,
            "bateria_pct": round(estado.bateria, 3),
            "temp_interna_C": round(estado.temp_interna, 2),
            "desgaste": round(estado.desgaste, 2),
            "distancia_m": round(estado.distancia_recorrida_m, 1),
            "accion": decision.accion,
            "velocidad": round(decision.velocidad, 4),
            "motivo": decision.motivo,
            "explicacion": decision.explicacion(),
            "vetos": "|".join(decision.vetos),
            "permitidos": "|".join(decision.permitidos),
        }
        # Señales de sensores
        for k, v in lectura.to_dict().items():
            fila[f"sensor_{k}"] = (round(v, 4) if isinstance(v, (int, float))
                                   and not isinstance(v, bool) else v)
        # RNA
        if prob_rna is not None and len(prob_rna) == 3:
            fila["rna_p_libre"] = round(float(prob_rna[LIBRE]), 4)
            fila["rna_p_obstaculo"] = round(float(prob_rna[OBSTACULO]), 4)
            fila["rna_p_barranco"] = round(float(prob_rna[BARRANCO]), 4)
            fila["rna_prediccion"] = NOMBRE_CLASE[int(np.argmax(prob_rna))]
            fila["rna_confianza"] = round(float(np.max(prob_rna)), 4)
        # Difuso
        if traza_difusa is not None:
            for var, grados in traza_difusa.membresias.items():
                for conj, g in grados.items():
                    if conj.startswith("NO "):
                        continue
                    fila[f"mu_{var}_{conj}"] = round(float(g), 4)
            fila["difuso_velocidad"] = round(traza_difusa.salidas.get("velocidad", 0), 4)
            fila["difuso_cautela"] = round(traza_difusa.salidas.get("cautela", 0), 4)
            fila["difuso_reglas_activas"] = traza_difusa.n_reglas_activas
            fila["difuso_reglas"] = traza_difusa.resumen_texto(3)
            fila["difuso_fallback"] = traza_difusa.fallback
        # AG
        if ruta is not None:
            fila["ag_fitness"] = round(ruta.fitness, 4)
            fila["ag_factible"] = ruta.factible
            fila["ag_largo_ruta"] = len(ruta.camino)
            fila["ag_ciencia"] = round(ruta.componentes.get("ciencia", 0), 4)
            fila["ag_riesgo"] = round(ruta.componentes.get("riesgo", 0), 4)
            fila["ag_energia_est"] = round(ruta.energia_estimada, 2)
            fila["ag_genes"] = ruta.cromosoma.genes_texto()
        if extra:
            fila.update(extra)
        self.filas.append(fila)

    def evento(self, paso: int, tipo: str, detalle: str, datos: Optional[dict] = None):
        self.eventos.append({"paso": paso, "tipo": tipo, "detalle": detalle,
                             **(datos or {})})

    # -- exportación ---------------------------------------------------------
    def df(self) -> pd.DataFrame:
        return pd.DataFrame(self.filas)

    def df_eventos(self) -> pd.DataFrame:
        return pd.DataFrame(self.eventos) if self.eventos else pd.DataFrame(
            columns=["paso", "tipo", "detalle"])

    def exportar_csv(self, ruta: str = "ares_x_traza.csv") -> str:
        self.df().to_csv(ruta, index=False, encoding="utf-8")
        return ruta

    def exportar_json(self, ruta: str = "ares_x_mision.json",
                      extra: Optional[dict] = None) -> str:
        paquete = {
            "version_software": VERSION,
            "configuracion": json.loads(self.cfg.to_json()),
            "resultados": self.resultados,
            "eventos": self.eventos,
            "n_pasos": len(self.filas),
            "supuestos_simulacion": SUPUESTOS_SIMULACION,
            "fuentes_nasa": FUENTES_NASA,
            "aviso": ("Simulación ficticia. Los parámetros operacionales son "
                      "hipótesis pedagógicas y no especificaciones NASA."),
        }
        if extra:
            paquete.update(extra)
        with open(ruta, "w", encoding="utf-8") as fh:
            json.dump(paquete, fh, indent=2, ensure_ascii=False, default=str)
        return ruta

## 8 · Estado del rover, eventos y ciclo de misión

Implementa literalmente el pseudociclo del enunciado:

`observar() → validar_sensores() → predecir_riesgo_rna() → inferir_control_difuso() →
replanificar_ag_si_corresponde() → aplicar_restricciones() → ejecutar_accion() →
actualizar_mundo() → registrar_traza() → actualizar_dashboard()`

El AG se recalcula al llegar a un objetivo, al detectar un bloqueo o ante un evento grave;
nunca en cada cuadro.

**Física simplificada con acoplamientos que importan.** Tres decisiones dan sentido operacional
al control difuso, que de otro modo sería decorativo:

1. **La velocidad es avance por paso** (celdas/paso): recorrer una celda a 0,25 toma cuatro
   pasos; a 1,0, uno. Así ir rápido tiene un beneficio real —cubrir terreno dentro del
   presupuesto de pasos— y no sólo un costo.
2. **Frenado insuficiente**: por encima de 0,60 de velocidad, detectar un peligro a una celda
   puede no bastar para detenerse. Es el riesgo que justifica la regla R03.
3. **Deslizamiento en pendiente** proporcional a velocidad y pendiente, con posibilidad de
   frenar en el borde.

Hay además control térmico acoplado a la energía (calefactor y refrigerador consumen batería),
recarga solar dependiente de la irradiancia planetaria y del polvo, desgaste por vibración y
corrosión, y deriva lateral por viento en modo sonda.

**Eventos** (aleatorios o inyectados): sismo que altera el relieve, impacto de meteorito que
crea zona de riesgo, tormenta de polvo, noche forzada, fallo de sensor, pérdida de enlace,
señal anómala —que **no** afirma vida—, contacto ficticio, corrosión ácida y ráfagas.

Las banderas `usar_rna`, `usar_difuso` y `usar_ag` implementan las **ablaciones** de RF-14
desactivando una técnica y dejando el resto intacto.

In [ ]:
"""
ARES-X · Módulo 8: Estado del rover, eventos y ciclo de misión.

Pseudociclo implementado literalmente (sección 11 del caso):
  observar() -> validar_sensores() -> predecir_riesgo_rna() ->
  inferir_control_difuso() -> replanificar_ag_si_corresponde() ->
  aplicar_restricciones() -> ejecutar_accion() -> actualizar_mundo() ->
  registrar_traza() -> actualizar_dashboard()

Las banderas cfg.usar_rna / usar_difuso / usar_ag permiten las ABLACIONES
exigidas por RF-14 desactivando una técnica y dejando el resto intacto.
"""

from __future__ import annotations

import time
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np


Celda = Tuple[int, int]

ESTADOS = ["CONFIGURACION", "EXPLORACION", "ALERTA", "REPLANIFICACION",
           "REFUGIO", "MODO_SEGURO", "EXITO", "FRACASO"]


# ----------------------------------------------------------------------------
@dataclass
class RoverState:
    pos: Celda = (0, 0)
    rumbo_idx: int = 2
    bateria: float = 60.0
    temp_interna: float = 20.0
    desgaste: float = 0.0
    velocidad: float = 0.0
    distancia_recorrida_m: float = 0.0
    estado: str = "CONFIGURACION"
    muestras: List[int] = field(default_factory=list)
    valor_ciencia: float = 0.0
    instrumentos: List[int] = field(default_factory=list)
    pasos_modo_seguro: int = 0
    velocidad_previa: float = 0.0
    progreso: float = 0.0        # avance acumulado dentro de la celda actual
    calefactor: bool = False
    refrigerador: bool = False

    def copia_ligera(self) -> dict:
        return {"pos": self.pos, "bateria": round(self.bateria, 2),
                "estado": self.estado, "muestras": len(self.muestras)}


# ----------------------------------------------------------------------------
class EventManager:
    """Eventos ambientales: aleatorios o inyectados por el operador (RF-09)."""

    CATALOGO = ["sismo", "meteorito", "tormenta_polvo", "noche", "fallo_sensor",
                "perdida_enlace", "biofirma", "alienigena", "corrosion", "viento"]

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.rng = rng_de(cfg.semilla, "eventos")
        self.cola: List[str] = []
        self.historial: List[dict] = []

    def inyectar(self, tipo: str) -> None:
        if tipo in self.CATALOGO:
            self.cola.append(tipo)

    def paso(self, world: World, estado: RoverState, sensores: SensorSuite,
            belief: BeliefMap, paso: int) -> List[dict]:
        ocurridos: List[dict] = []
        pendientes = list(self.cola)
        self.cola.clear()

        if self.cfg.eventos_aleatorios:
            p = world.perfil.prob_evento_base * \
                {"facil": 0.6, "media": 1.0, "dificil": 1.5}.get(self.cfg.dificultad, 1.0)
            if self.rng.random() < p * 0.35:
                candidatos = self._candidatos_planeta(world)
                pendientes.append(candidatos[int(self.rng.integers(0, len(candidatos)))])

        for tipo in pendientes:
            det = self._aplicar(tipo, world, estado, sensores, belief)
            reg = {"paso": paso, "tipo": tipo, "detalle": det}
            ocurridos.append(reg)
            self.historial.append(reg)
        return ocurridos

    def _candidatos_planeta(self, world: World) -> List[str]:
        p = world.perfil.clave
        if world.modo == "sonda":
            return ["viento", "turbulencia_fallo", "perdida_enlace", "fallo_sensor"] \
                if False else ["viento", "perdida_enlace", "fallo_sensor"]
        base = ["fallo_sensor", "perdida_enlace", "meteorito"]
        if p == "marte":
            base += ["tormenta_polvo", "sismo", "biofirma"]
        elif p == "tierra":
            base += ["sismo", "tormenta_polvo"]
        elif p == "mercurio":
            base += ["noche", "meteorito"]
        elif p == "venus":
            base += ["corrosion", "corrosion"]
        return base

    def _aplicar(self, tipo: str, world: World, estado: RoverState,
                 sensores: SensorSuite, belief: BeliefMap) -> str:
        if tipo == "sismo":
            world.sismo = 1.0
            # El relieve cambia: algunas celdas pueden volverse barranco
            cambiadas = 0
            for _ in range(6):
                f = int(self.rng.integers(0, world.filas))
                c = int(self.rng.integers(0, world.cols))
                world.elev[f, c] -= self.rng.uniform(1.0, 5.0)
                if world.clase[f, c] == LIBRE and self.rng.random() < 0.35:
                    world.clase[f, c] = BARRANCO
                    belief.confirmado[f, c] = False     # el rover aún no lo sabe
                    cambiadas += 1
            world.gradiente = world._calc_gradiente()
            return f"Sismo: picos de aceleración; {cambiadas} celdas alteradas."
        if tipo == "meteorito":
            f = int(clamp(estado.pos[0] + self.rng.integers(-5, 6), 0, world.filas - 1))
            c = int(clamp(estado.pos[1] + self.rng.integers(-5, 6), 0, world.cols - 1))
            world.zonas_riesgo.append({"centro": (f, c), "radio": 2,
                                       "intensidad": 0.9, "duracion": 40,
                                       "tipo": "meteorito"})
            world.tormenta = clamp(world.tormenta + 0.25, 0, 1)
            world.sismo = max(world.sismo, 0.6)
            belief.prohibir((f, c), radio=1)
            return f"Impacto próximo en {(f, c)}: nueva zona de riesgo y polvo."
        if tipo == "tormenta_polvo":
            world.tormenta = clamp(world.tormenta + 0.75, 0, 1)
            return "Tormenta de polvo: visibilidad y producción solar reducidas."
        if tipo == "noche":
            world.hora = 21.5
            return "Se fuerza el ciclo nocturno: sin captación solar."
        if tipo == "fallo_sensor":
            s = ["proximidad", "lidar", "inclinometro", "acelerometro"][
                int(self.rng.integers(0, 4))]
            sensores.forzar_fallo(s, "nan" if self.rng.random() < 0.6 else "saturado")
            return f"Fallo inyectado en el sensor '{s}'."
        if tipo == "perdida_enlace":
            sensores.forzar_fallo("enlace", "nan")
            return "Enlace degradado: el rover opera en autonomía local."
        if tipo == "biofirma":
            f = int(clamp(estado.pos[0] + self.rng.integers(-3, 4), 0, world.filas - 1))
            c = int(clamp(estado.pos[1] + self.rng.integers(-3, 4), 0, world.cols - 1))
            world.anomalia_bio = (f, c)
            return ("Señal ANÓMALA simulada en espectro orgánico. NO constituye "
                    "detección de vida: requiere análisis independiente.")
        if tipo == "alienigena":
            f = int(clamp(estado.pos[0] + self.rng.integers(-4, 5), 0, world.filas - 1))
            c = int(clamp(estado.pos[1] + self.rng.integers(-4, 5), 0, world.cols - 1))
            world.alienigena = (f, c)
            world.zonas_riesgo.append({"centro": (f, c), "radio": 1,
                                       "intensidad": 0.8, "duracion": 25,
                                       "tipo": "contacto_ficticio"})
            belief.prohibir((f, c), radio=1)
            return ("EVENTO NARRATIVO FICTICIO: objeto no identificado tratado "
                    "como obstáculo dinámico. Sin valor científico.")
        if tipo == "corrosion":
            estado.desgaste = clamp(estado.desgaste + 9.0, 0, 200)
            return "Exposición ácida: aumento del desgaste estructural."
        if tipo == "viento":
            world.viento = clamp(world.viento + 0.5, 0, 1)
            return "Ráfaga: desviación lateral de la trayectoria de la sonda."
        return "Evento sin efecto."


# ----------------------------------------------------------------------------
class Mission:
    """Orquestador del ciclo observación → decisión → acción."""

    def __init__(self, cfg: Config, rna: Optional[NeuralPerception] = None,
                 verbose: bool = False):
        ok, errores = cfg.validar()
        if not ok:
            raise ValueError("Configuración inválida:\n - " + "\n - ".join(errores))
        self.cfg = cfg
        self.verbose = verbose
        self.world = World(cfg)
        self.belief = BeliefMap(cfg.filas, cfg.columnas)
        self.sensores = SensorSuite(cfg)
        self.rna = rna
        self.difuso = FuzzyController()
        self.ag = GeneticPlanner(cfg)
        self.supervisor = SafetySupervisor(cfg)
        self.logger = MissionLogger(cfg)
        self.eventos = EventManager(cfg)
        self.rng = rng_de(cfg.semilla, "mision")

        self.estado = RoverState(
            pos=self.world.inicio,
            bateria=float(cfg.bateria_inicial),
            temp_interna=clamp(self.world.temperatura_superficie(self.world.inicio),
                               cfg.perfil.limite_termico_C[0] + 15,
                               cfg.perfil.limite_termico_C[1] - 15),
            estado="CONFIGURACION",
        )
        self.belief.confirmar(self.estado.pos, LIBRE)
        self.belief.visitado[self.estado.pos] = True

        self.paso_actual = 0
        self.ruta_planificada: List[Celda] = []
        self.ruta_ejecutada: List[Celda] = [self.estado.pos]
        self.plan: Optional[RutaCandidata] = None
        self.candidatas: List[RutaCandidata] = []
        self.traza_ag = None
        self.traza_difusa = None
        self.prob_rna: Optional[np.ndarray] = None
        self.ultima_decision: Optional[Decision] = None
        self.indice_ruta = 0
        self.terminada = False
        self.motivo_fin = ""
        self._replanificado_este_paso = False
        self._pasos_sin_progreso = 0
        self._en_transito = False
        self.recarga_media_por_celda = 0.30
        # Media móvil de la velocidad realmente ejecutada. El AG debe estimar el
        # tiempo con una velocidad REPRESENTATIVA: si usara la instantánea, una
        # frenada puntual por cautela le haría concluir que todos los objetivos
        # quedan demasiado lejos y abandonaría la ciencia.
        self.velocidad_media_obs = 0.45
        self.metricas: Dict[str, float] = {
            "colisiones": 0, "barrancos_evitados": 0, "replanificaciones": 0,
            "violaciones_seguridad": 0, "pasos_modo_seguro": 0,
            "vetos_duros": 0, "muestras": 0, "tiempo_decision_s": 0.0,
            "pasos_sin_avance": 0,
        }
        self.valor_ciencia_total = sum(s.valor for s in self.world.sitios) or 1.0
        self.estado.estado = "EXPLORACION"

    # ================================================================== #
    # 1. observar()
    # ================================================================== #
    def observar(self):
        return self.sensores.observar(self.world, self.estado)

    # 2. validar_sensores()
    @staticmethod
    def validar_sensores(lectura) -> Tuple[bool, List[str]]:
        invalidos = [k for k, v in lectura.validos.items() if not v]
        return (len(invalidos) == 0, invalidos)

    # 3. predecir_riesgo_rna()
    def predecir_riesgo_rna(self, destino: Optional[Celda]) -> Optional[np.ndarray]:
        """Actualiza el mapa de creencias con la RNA y devuelve p(destino)."""
        if not self.cfg.usar_rna or self.rna is None or not self.rna.entrenado:
            # ABLACIÓN sin RNA: sólo creencia determinista del sensor de borde
            # a una celda; el rover queda más miope.
            for d in DIRECCIONES_8:
                q = (self.estado.pos[0] + d[0], self.estado.pos[1] + d[1])
                if self.belief.dentro(q) and not self.belief.explorado[q[0], q[1]]:
                    self.belief.actualizar(q, 0.8, 0.1, 0.1,
                                           self.world.pendiente_local(q), alfa=0.3)
            return None

        rasgos = self.sensores.rasgos_vecindario(self.world, self.estado.pos)
        probas = self.rna.predecir_vecindario(rasgos)
        for celda, p in probas.items():
            if self.belief.dentro(celda):
                self.belief.actualizar(celda, float(p[LIBRE]), float(p[OBSTACULO]),
                                       float(p[BARRANCO]),
                                       self.world.pendiente_local(celda))
        self.rasgos_actuales = rasgos
        if destino is not None and destino in probas:
            return probas[destino]
        return None

    # 4. inferir_control_difuso()
    def inferir_control_difuso(self, lectura, destino: Optional[Celda]):
        if not self.cfg.usar_difuso:
            # ABLACIÓN sin difuso: velocidad FIJA calibrada a la velocidad media
            # que produce el controlador difuso (0.40 medida sobre 8 misiones).
            # Igualar la media es lo que hace honesta la ablación: lo que se
            # elimina es la ASIGNACIÓN inteligente de velocidad (rápido donde es
            # seguro, lento donde no), no el presupuesto de velocidad.
            class _Fija:
                salidas = {"velocidad": 0.40, "cautela": 0.30}
                membresias: Dict[str, Dict[str, float]] = {}
                activaciones: List[dict] = []
                n_reglas_activas = 0
                fallback = False
                entradas: Dict[str, float] = {}
                agregados: Dict[str, np.ndarray] = {}

                @staticmethod
                def resumen_texto(top=3):
                    return ("ABLACIÓN: controlador difuso desactivado "
                            "(velocidad fija 0.40 = media del difuso).")

                @staticmethod
                def reglas_activas(umbral=1e-3):
                    return []
            return _Fija()

        dist = lectura.prox_frontal_m
        if not es_valido(dist):
            dist = 0.0
        mt = margen_termico(self.estado.temp_interna, self.cfg.perfil.limite_termico_C)
        pend = lectura.inclinacion_deg if es_valido(lectura.inclinacion_deg) else 45.0
        # Riesgo creído del destino -> acorta la distancia efectiva percibida
        if destino is not None and self.belief.dentro(destino):
            r = self.belief.riesgo_estimado(destino)
            dist = min(dist, 20.0 * (1.0 - 0.60 * r))
        return self.difuso.inferir(distancia_m=clamp(dist, 0, 20),
                                   bateria_pct=clamp(self.estado.bateria, 0, 100),
                                   pendiente_deg=clamp(pend, 0, 45),
                                   margen_termico_C=mt)

    # 5. replanificar_ag_si_corresponde()
    def necesita_replanificar(self) -> bool:
        if self.plan is None or not self.ruta_planificada:
            return True
        if self.indice_ruta >= len(self.ruta_planificada):
            return True
        ok, _ = self.supervisor.validar_ruta(
            self.ruta_planificada[self.indice_ruta:], self.belief, self.estado.bateria)
        return not ok

    def replanificar(self, motivo: str = "") -> None:
        if self._replanificado_este_paso:
            return
        self._replanificado_este_paso = True
        t0 = time.time()
        pendientes = [s for s in self.world.sitios if not s.recolectado]
        if self.cfg.usar_ag:
            plan, cands, tz = self.ag.planificar(
                self.belief, self.estado.pos, pendientes, self.world.refugios,
                self.estado.bateria, semilla=self.cfg.semilla + self.paso_actual,
                velocidad_media=max(0.15, self.velocidad_media_obs),
                recarga_por_celda=self.recarga_media_por_celda)
            self.plan, self.candidatas, self.traza_ag = plan, cands, tz
            # El supervisor puede VETAR la mejor ruta y tomar la siguiente
            ok, motivos = self.supervisor.validar_ruta(plan.camino, self.belief,
                                                       self.estado.bateria)
            if not ok or not plan.factible:
                alternativa = next((c for c in cands
                                    if c is not plan and c.factible and
                                    self.supervisor.validar_ruta(
                                        c.camino, self.belief,
                                        self.estado.bateria)[0]), None)
                self.logger.evento(self.paso_actual, "veto_ruta",
                                   f"Ruta de mejor fitness rechazada ({motivos or plan.motivos}); "
                                   f"{'se usa alternativa' if alternativa else 'sin alternativa factible'}.")
                self.metricas["vetos_duros"] += 1
                if alternativa is not None:
                    self.plan = alternativa
            self.ruta_planificada = list(self.plan.camino)
        else:
            # ABLACIÓN sin AG: A* por distancia al objetivo más cercano
            objetivo = None
            if pendientes:
                objetivo = min(pendientes,
                               key=lambda s: distancia_cheb(self.estado.pos, s.pos)).pos
            elif self.world.refugios:
                objetivo = min(self.world.refugios,
                               key=lambda r: distancia_cheb(self.estado.pos, r))
            camino = ruta_mas_corta(
                self.belief, self.estado.pos, objetivo,
                ignorar_riesgo=self.cfg.baseline_ruta_corta) if objetivo else None
            self.ruta_planificada = camino or [self.estado.pos]
            self.plan = RutaCandidata(
                cromosoma=type("C", (), {"genes_texto": lambda self: "ABLACIÓN_sin_AG"})(),
                camino=self.ruta_planificada, fitness=0.0,
                componentes={"ciencia": 0.0, "riesgo": 0.0}, factible=camino is not None)
            self.candidatas = [self.plan]
            self.traza_ag = None
        self.indice_ruta = 1 if len(self.ruta_planificada) > 1 else 0
        self.metricas["replanificaciones"] += 1
        self.metricas["tiempo_decision_s"] += time.time() - t0
        if motivo:
            self.logger.evento(self.paso_actual, "replanificacion", motivo)

    # 6-7. aplicar_restricciones() + ejecutar_accion()
    def _destino_actual(self) -> Optional[Celda]:
        if not self.ruta_planificada or self.indice_ruta >= len(self.ruta_planificada):
            return None
        d = self.ruta_planificada[self.indice_ruta]
        if distancia_cheb(d, self.estado.pos) > 1:
            return None            # el plan quedó desincronizado -> replanificar
        return d

    def ejecutar_accion(self, decision: Decision) -> None:
        e, w = self.estado, self.world
        e.velocidad_previa = e.velocidad
        e.velocidad = decision.velocidad

        if decision.prohibir is not None:
            self.belief.prohibir(decision.prohibir, radio=0)
            if decision.motivo == "VETO_BARRANCO":
                # Confirmado por el sensor determinista de borde.
                self.belief.confirmar(decision.prohibir, BARRANCO)
                self.metricas["barrancos_evitados"] += 1
            elif decision.motivo == "VETO_COLISION":
                self.belief.confirmar(decision.prohibir, OBSTACULO)
            elif decision.motivo == "VETO_RNA_BARRANCO":
                # Sospecha probabilística: se eleva la creencia pero NO se
                # confirma. Queda sujeta a revisión por el sensor de borde.
                f, c = decision.prohibir
                self.belief.p_barr[f, c] = max(float(self.belief.p_barr[f, c]), 0.9)
                self.metricas["barrancos_evitados"] += 1

        # --- FÍSICA: frenado insuficiente --------------------------------
        if decision.prohibir is not None and decision.motivo in (
                "VETO_BARRANCO", "VETO_COLISION", "VETO_PENDIENTE"):
            v = e.velocidad_previa
            if v > 0.60:
                p_fallo = 0.30 * (v - 0.60) / 0.40          # [SUPUESTO]
                if self.rng.random() < p_fallo:
                    destino = decision.prohibir
                    clase_real = w.clase_de(destino)
                    e.pos = destino
                    self.ruta_ejecutada.append(destino)
                    self.belief.visitado[destino] = True
                    self.metricas["violaciones_seguridad"] += 1
                    self.logger.evento(
                        self.paso_actual, "frenado_insuficiente",
                        f"Velocidad previa {v:.2f}: el rover no alcanzó a detenerse "
                        f"y entró en {destino} ({NOMBRE_CLASE[clase_real]}).")
                    if clase_real == BARRANCO:
                        self.metricas["colisiones"] += 1
                        e.desgaste = 200.0
                        self.terminada = True
                        self.motivo_fin = ("Frenado insuficiente a alta velocidad: "
                                           "caída en precipicio.")
                        e.estado = "FRACASO"
                    else:
                        self.metricas["colisiones"] += 1
                        e.desgaste = clamp(e.desgaste + 22.0, 0, 250)
                    return

        if decision.accion == "MODO_SEGURO":
            e.estado = "MODO_SEGURO"
            e.pasos_modo_seguro += 1
            self.metricas["pasos_modo_seguro"] += 1
            # Recuperación: se intenta reparar el sensor tras 3 pasos
            if e.pasos_modo_seguro % 3 == 0:
                self.sensores.reparar()
                self.logger.evento(self.paso_actual, "recuperacion",
                                   "Reinicio de subsistema de sensores tras modo seguro.")
            return

        e.pasos_modo_seguro = 0

        if decision.accion == "MUESTREAR":
            self._muestrear()
            e.estado = "EXPLORACION"
            return

        if decision.accion == "RECARGAR":
            e.estado = "REFUGIO"
            return

        if decision.accion in ("DETENER",):
            self.metricas["pasos_sin_avance"] += 1
            e.estado = "ALERTA"
            return

        if decision.accion == "REPLANIFICAR":
            e.estado = "REPLANIFICACION"
            self.replanificar(decision.explicacion())
            return

        if decision.accion in ("AVANZAR", "VOLVER"):
            destino = decision.destino
            if destino is None or not w.dentro(destino):
                self.replanificar("Destino inválido")
                return

            # --- FÍSICA: la velocidad es AVANCE POR PASO (celdas/paso) -----
            # Recorrer una celda a velocidad 0.25 toma cuatro pasos; a 1.0, uno.
            # Así la velocidad tiene un beneficio real (cubrir terreno) además
            # de un costo (riesgo de frenado y deslizamiento), y el controlador
            # difuso resuelve ese compromiso en vez de ser decorativo.
            # El estado mecánico limita la velocidad efectiva [SUPUESTO]:
            # un rover dañado avanza más lento, así que chocar se paga en
            # terreno cubierto y no sólo en un contador de colisiones.
            factor_mecanico = clamp(1.0 - 0.5 * e.desgaste / 100.0, 0.35, 1.0)
            v_efectiva = max(0.05, decision.velocidad * factor_mecanico)
            e.progreso += v_efectiva
            e.distancia_recorrida_m += CELDA_M * v_efectiva
            if e.progreso < 1.0:
                self._en_transito = True
                e.estado = "EXPLORACION"
                return
            e.progreso -= 1.0
            self._en_transito = False

            clase_real = w.clase_de(destino)
            # Movimiento real (la física no conoce la creencia)
            e.rumbo_idx = self._rumbo(e.pos, destino)
            e.pos = destino
            self.ruta_ejecutada.append(destino)
            self.belief.visitado[destino] = True
            self.belief.confirmar(destino, clase_real)
            self.indice_ruta += 1
            e.estado = "VOLVER" if decision.accion == "VOLVER" else "EXPLORACION"
            if e.estado == "VOLVER":
                e.estado = "REFUGIO"

            # --- FÍSICA: deslizamiento en pendiente ----------------------
            pend_real = abs(w.pendiente_deg(self.ruta_ejecutada[-2]
                                            if len(self.ruta_ejecutada) > 1 else destino,
                                            destino))
            p_desliz = 0.18 * decision.velocidad * clamp(
                (pend_real - 12.0) / 18.0, 0.0, 1.0)                # [SUPUESTO]
            if clase_real == LIBRE and self.rng.random() < p_desliz:
                # El rover resbala una celda en el sentido de la máxima pendiente
                peor, h = None, 0.0
                for d in DIRECCIONES_8:
                    q = (destino[0] + d[0], destino[1] + d[1])
                    if not w.dentro(q):
                        continue
                    caida = float(w.elev[destino] - w.elev[q])
                    if caida > h:
                        peor, h = q, caida
                if peor is not None and h > 2.0 and \
                        w.clase_de(peor) == BARRANCO and self.rng.random() < 0.60:
                    # El rover alcanza a frenar en el borde: daño, no caída.
                    e.desgaste = clamp(e.desgaste + 12.0, 0, 250)
                    self.belief.confirmar(peor, BARRANCO)
                    self.belief.prohibir(peor)
                    self.metricas["barrancos_evitados"] += 1
                    self.logger.evento(
                        self.paso_actual, "frenado_en_borde",
                        f"Deslizamiento detenido en el borde de {peor}: daño "
                        f"estructural pero sin caída.")
                    peor = None
                if peor is not None and h > 2.0:
                    e.pos = peor
                    self.ruta_ejecutada.append(peor)
                    self.belief.visitado[peor] = True
                    clase_real = w.clase_de(peor)
                    self.belief.confirmar(peor, clase_real)
                    e.desgaste = clamp(e.desgaste + 6.0, 0, 250)
                    self.logger.evento(
                        self.paso_actual, "deslizamiento",
                        f"Deslizamiento a {decision.velocidad:.2f} de velocidad sobre "
                        f"pendiente {pend_real:.0f}°: el rover resbala a {peor} "
                        f"({NOMBRE_CLASE[clase_real]}).")
                    destino = peor

            # ¿Violación de seguridad? (sólo medición, el rover ya se movió)
            if clase_real == BARRANCO:
                self.metricas["violaciones_seguridad"] += 1
                self.metricas["colisiones"] += 1
                e.desgaste = 200.0
                self.terminada = True
                self.motivo_fin = "Caída en precipicio: pérdida del vehículo."
                e.estado = "FRACASO"
            elif clase_real == OBSTACULO:
                # El rover NO entra: rebota a la celda anterior, pierde el paso
                # y acumula daño. Ocupar una celda infranqueable sería
                # físicamente imposible y abarataría artificialmente el error
                # de percepción.
                self.metricas["colisiones"] += 1
                e.desgaste = clamp(e.desgaste + 18.0, 0, 250)
                self.ruta_ejecutada.pop()
                e.pos = self.ruta_ejecutada[-1]
                e.progreso = 0.0
                self.indice_ruta -= 1
                self.belief.confirmar(destino, OBSTACULO)
                self.belief.prohibir(destino)
                self._replanificado_este_paso = False
                self.replanificar(f"Colisión con obstáculo en {destino}")
                self.logger.evento(
                    self.paso_actual, "colision",
                    f"Colisión con obstáculo en {destino}: el rover rebota a "
                    f"{e.pos}, pierde el paso y suma 18 de desgaste.")

    def _muestrear(self) -> None:
        for s in self.world.sitios:
            if s.pos == self.estado.pos and not s.recolectado:
                habilita = {INSTRUMENTOS_BASE[i].habilita
                            for i in self.estado.instrumentos}
                if s.requiere in habilita or not self.estado.instrumentos:
                    s.recolectado = True
                    self.estado.muestras.append(s.id)
                    self.estado.valor_ciencia += s.valor
                    self.metricas["muestras"] += 1
                    self.estado.bateria -= 0.6      # [SUPUESTO] coste de muestreo
                    self.logger.evento(self.paso_actual, "muestra",
                                       f"Muestra {s.id} recolectada en {s.pos} "
                                       f"(valor {s.valor:.2f}, tipo {s.requiere}).")
                else:
                    self.logger.evento(self.paso_actual, "muestra_fallida",
                                       f"Sitio {s.id} requiere '{s.requiere}': "
                                       f"instrumento no embarcado.")

    @staticmethod
    def _rumbo(a: Celda, b: Celda) -> int:
        d = (int(np.sign(b[0] - a[0])), int(np.sign(b[1] - a[1])))
        return DIRECCIONES_8.index(d) if d in DIRECCIONES_8 else 0

    # 8. actualizar_mundo() — física simplificada
    def actualizar_mundo(self, decision: Decision) -> None:
        e, w, cfg = self.estado, self.world, self.cfg
        lo, hi = cfg.perfil.limite_termico_C

        # --- térmica: la electrónica tiende al ambiente, el motor calienta ---
        t_ext = w.temperatura_superficie(e.pos)
        e.temp_interna += 0.10 * (t_ext - e.temp_interna) + 1.8 * e.velocidad
        consumo_termico = 0.0
        e.calefactor = e.refrigerador = False
        if e.temp_interna < lo + 15:
            e.temp_interna += 3.0
            consumo_termico += 0.30                 # [SUPUESTO]
            e.calefactor = True
        elif e.temp_interna > hi - 15:
            e.temp_interna -= 3.0
            consumo_termico += 0.35                 # [SUPUESTO]
            e.refrigerador = True

        # --- energía ---------------------------------------------------------
        pend = w.pendiente_local(e.pos)
        consumo = 0.05 + consumo_termico            # idle + control térmico
        if decision.accion in ("AVANZAR", "VOLVER"):
            consumo += (0.45 + 0.03 * max(0.0, pend)) * max(0.25, e.velocidad)
        consumo += 0.004 * sum(INSTRUMENTOS_BASE[i].potencia_W
                               for i in e.instrumentos)
        recarga = 1.2 * w.irradiancia()             # [SUPUESTO] solar
        self.recarga_media_por_celda = clamp(
            0.85 * self.recarga_media_por_celda + 0.15 * recarga, 0.0, 3.0)
        if decision.accion in ("AVANZAR", "VOLVER"):
            self.velocidad_media_obs = clamp(
                0.90 * self.velocidad_media_obs + 0.10 * e.velocidad, 0.05, 1.0)
        if e.pos in w.refugios:
            recarga += 6.0                          # [SUPUESTO] estación
        e.bateria = clamp(e.bateria - consumo + recarga, 0.0, 100.0)

        # --- desgaste ---------------------------------------------------------
        e.desgaste = clamp(
            e.desgaste + 0.05 * e.velocidad * cfg.perfil.factor_desgaste
            + 0.5 * w.sismo + (0.12 if cfg.planeta == "venus" else 0.0), 0, 250)

        # --- viento lateral en modo sonda -------------------------------------
        if w.modo == "sonda" and w.viento > 0.05:
            if self.rng.random() < w.viento * 0.5:
                dc = int(np.sign(self.rng.normal()))
                nueva = (e.pos[0], int(clamp(e.pos[1] + dc, 0, w.cols - 1)))
                if w.transitable(nueva):
                    e.pos = nueva
                    self.ruta_ejecutada.append(nueva)
                    self.logger.evento(self.paso_actual, "deriva_viento",
                                       f"Ráfaga desvía la sonda a {nueva}.")
            w.viento = max(0.0, w.viento - 0.08)

        w.avanzar_tiempo()

    # ================================================================== #
    # Ciclo principal
    # ================================================================== #
    def paso(self) -> dict:
        """Ejecuta un ciclo completo. Devuelve un resumen del paso."""
        if self.terminada:
            return {"terminada": True, "motivo": self.motivo_fin}
        self.paso_actual += 1
        self._replanificado_este_paso = False
        self._en_transito = False
        pos_previa = self.estado.pos
        t0 = time.time()

        eventos = self.eventos.paso(self.world, self.estado, self.sensores,
                                    self.belief, self.paso_actual)
        for ev in eventos:
            self.logger.evento(self.paso_actual, ev["tipo"], ev["detalle"])

        lectura = self.observar()
        sensores_ok, invalidos = self.validar_sensores(lectura)

        if self.necesita_replanificar():
            self.replanificar("Plan ausente, agotado o inviable")

        destino = self._destino_actual()
        self.prob_rna = self.predecir_riesgo_rna(destino)
        self.traza_difusa = self.inferir_control_difuso(lectura, destino)

        # Confirmación determinista del borde cuando el destino es sospechoso
        confirmacion = None
        if destino is not None:
            riesgo_destino = self.belief.riesgo_estimado(destino)
            sospecha_rna = (self.prob_rna is not None and
                            float(self.prob_rna[BARRANCO]) >= 0.30)
            if riesgo_destino > 0.35 or sospecha_rna:
                confirmacion = self.sensores.confirmar_borde(self.world, destino)
                if confirmacion is not None:
                    self.belief.confirmar(destino, confirmacion)

        # Sin destino y sobre un refugio: recargar en vez de replanificar en vacío
        if destino is None and self.estado.pos in self.world.refugios \
                and self.estado.bateria < 98.0:
            pendientes_r = [s for s in self.world.sitios if not s.recolectado]
            if not pendientes_r or self.estado.bateria < 35.0:
                decision_r = Decision("RECARGAR", 0.0, "OK_RECARGA",
                                      f"En refugio {self.estado.pos}: recarga antes "
                                      f"de continuar o cerrar la misión.")
                self.ejecutar_accion(decision_r)
                self.actualizar_mundo(decision_r)
                self.ultima_decision = decision_r
                self.logger.registrar(self.paso_actual, self.estado, lectura,
                                      decision_r, self.prob_rna, self.traza_difusa,
                                      self.plan, self.world,
                                      extra={"eventos": "|".join(e["tipo"] for e in eventos),
                                             "ciencia_acumulada": round(self.estado.valor_ciencia, 3),
                                             "sensores_invalidos": "|".join(invalidos),
                                             "riesgo_destino": 0.0})
                self._evaluar_fin()
                return {"paso": self.paso_actual, "accion": "RECARGAR",
                        "motivo": "OK_RECARGA", "pos": self.estado.pos,
                        "bateria": round(self.estado.bateria, 2),
                        "terminada": self.terminada,
                        "eventos": [e["tipo"] for e in eventos]}

        refugio_alcanzable = self._refugio_alcanzable()
        hay_sitio = any(s.pos == self.estado.pos and not s.recolectado
                        for s in self.world.sitios)

        decision = self.supervisor.evaluar(
            estado=self.estado, lectura=lectura, belief=self.belief,
            destino=destino,
            velocidad_difusa=self.traza_difusa.salidas["velocidad"],
            cautela=self.traza_difusa.salidas["cautela"],
            prob_rna=self.prob_rna, confirmacion_borde=confirmacion,
            refugio_alcanzable=refugio_alcanzable, hay_sitio_aqui=hay_sitio,
            ruta_factible=(self.plan.factible if self.plan else False),
            pendiente_destino=self._pendiente_transicion(destino),
            reserva_requerida=self.reserva_requerida())
        self.ultima_decision = decision
        if decision.vetos:
            self.metricas["vetos_duros"] += 1

        self.ejecutar_accion(decision)
        if decision.replanificar and not self.terminada:
            self.replanificar(decision.explicacion())
        self.actualizar_mundo(decision)

        self.metricas["tiempo_decision_s"] += time.time() - t0
        self.logger.registrar(
            self.paso_actual, self.estado, lectura, decision, self.prob_rna,
            self.traza_difusa, self.plan, self.world,
            extra={"en_transito": self._en_transito,
                   "reserva_requerida_pct": round(self.reserva_requerida(), 2),
                   "energia_hasta_refugio_pct": round(
                       min(999.0, self._energia_hasta_refugio()), 2),
                   "progreso_celda": round(self.estado.progreso, 3),
                   "sensores_invalidos": "|".join(invalidos),
                   "eventos": "|".join(e["tipo"] for e in eventos),
                   "ciencia_acumulada": round(self.estado.valor_ciencia, 3),
                   "riesgo_destino": round(
                       self.belief.riesgo_estimado(destino) if destino else 0.0, 3)})

        # Detección de estancamiento: si el rover no avanza y el plan no
        # ofrece destino, se declara formalmente la ausencia de ruta.
        if self.estado.pos == pos_previa and not self._en_transito \
                and decision.accion not in ("MUESTREAR", "RECARGAR", "MODO_SEGURO"):
            self._pasos_sin_progreso += 1
        else:
            self._pasos_sin_progreso = 0
        # Relajación intermedia: antes de declarar "sin ruta", se liberan las
        # zonas prohibidas que NO corresponden a peligros confirmados (p. ej.
        # vetos preventivos de la RNA o zonas de impacto ya expiradas). Evita
        # que el rover quede atrapado por su propia cautela acumulada.
        if self._pasos_sin_progreso == 6 and not self.terminada:
            liberadas = 0
            for f in range(self.belief.filas):
                for c in range(self.belief.cols):
                    if self.belief.prohibido[f, c] and not (
                            self.belief.confirmado[f, c] and
                            (self.belief.p_barr[f, c] >= 0.99 or
                             self.belief.p_obst[f, c] >= 0.99)):
                        self.belief.prohibido[f, c] = False
                        liberadas += 1
            if liberadas:
                self.logger.evento(self.paso_actual, "relajacion_zonas",
                                   f"Se liberaron {liberadas} celdas prohibidas no "
                                   f"confirmadas para reintentar la planificación.")
                self._replanificado_este_paso = False
                self.replanificar("Relajación de zonas prohibidas tras estancamiento")

        if self._pasos_sin_progreso >= 16 and not self.terminada:
            self.terminada = True
            pend = [s for s in self.world.sitios if not s.recolectado]
            en_refugio = self.estado.pos in self.world.refugios
            self.estado.estado = "EXITO" if (en_refugio and not pend) else "FRACASO"
            self.motivo_fin = ("No existe ruta factible hacia los objetivos "
                               "restantes; se reporta sin solución y el rover "
                               f"{'permanece a salvo en refugio' if en_refugio else 'queda en espera'}.")
            self.logger.evento(self.paso_actual, "sin_ruta", self.motivo_fin)

        self._evaluar_fin()
        return {"paso": self.paso_actual, "accion": decision.accion,
                "motivo": decision.motivo, "pos": self.estado.pos,
                "bateria": round(self.estado.bateria, 2),
                "terminada": self.terminada, "eventos": [e["tipo"] for e in eventos]}

    def _pendiente_transicion(self, destino: Optional[Celda]) -> float:
        """Pendiente estimada entre la celda actual y el destino (altimetría).

        Se usa la lectura ruidosa de diferencia de altura, no la verdad del
        terreno: el rover estima, no sabe.
        """
        if destino is None or not self.world.dentro(destino):
            return 0.0
        rasgos = getattr(self, "rasgos_actuales", None)
        if rasgos and destino in rasgos:
            dh = float(rasgos[destino][1])          # delta_altura_m observado
        else:
            dh = float(self.world.elev[destino] - self.world.elev[self.estado.pos])
            dh += float(self.rng.normal(0, 0.45))
        dl = CELDA_M * max(1, distancia_cheb(self.estado.pos, destino))
        return float(np.degrees(np.arctan2(dh, dl)))

    def _consumo_neto_por_celda(self) -> float:
        """Consumo esperado por celda descontando la recarga solar observada."""
        v = max(0.2, self.velocidad_media_obs)
        pend = self.world.pendiente_local(self.estado.pos)
        bruto = 0.05 + (0.45 + 0.03 * max(0.0, pend)) * v
        return max(0.04, (bruto - self.recarga_media_por_celda) / v)

    def _energia_hasta_refugio(self) -> float:
        """Energía (% de batería) estimada para alcanzar el refugio más cercano."""
        if not self.world.refugios:
            return float("inf")
        d = min(distancia_cheb(self.estado.pos, r) for r in self.world.refugios)
        return d * self._consumo_neto_por_celda()

    def reserva_requerida(self) -> float:
        """Reserva dinámica: energía hasta la zona segura + margen del 50 %."""
        e = self._energia_hasta_refugio()
        if not np.isfinite(e):
            return 100.0
        return float(clamp(e * 1.5 + 3.0, self.cfg.reserva_energia_pct, 60.0))

    def _refugio_alcanzable(self) -> bool:
        if not self.world.refugios:
            return False
        return self._energia_hasta_refugio() * 1.15 <= self.estado.bateria

    def _evaluar_fin(self) -> None:
        e = self.estado
        pendientes = [s for s in self.world.sitios if not s.recolectado]
        objetivo_cumplido = (e.valor_ciencia >= 0.5 * self.valor_ciencia_total
                             or not self.world.sitios)
        en_refugio = e.pos in self.world.refugios

        if e.bateria <= 0.0:
            self.terminada = True
            e.estado = "FRACASO"
            self.motivo_fin = "Batería agotada."
        elif e.desgaste >= 200:
            self.terminada = True
            e.estado = "FRACASO"
            self.motivo_fin = self.motivo_fin or "Daño estructural irreversible."
        elif en_refugio and not pendientes and self.world.sitios:
            self.terminada = True
            e.estado = "EXITO"
            self.motivo_fin = "Todas las muestras recolectadas y retorno seguro."
        elif objetivo_cumplido and en_refugio and not pendientes:
            self.terminada = True
            e.estado = "EXITO"
            self.motivo_fin = "Objetivos completados y retorno seguro."
        elif objetivo_cumplido and en_refugio and self.paso_actual > 12:
            self.terminada = True
            e.estado = "EXITO"
            self.motivo_fin = ("Objetivo científico mínimo alcanzado y rover a "
                               "salvo en refugio.")
        elif self.paso_actual >= self.cfg.max_pasos:
            self.terminada = True
            e.estado = "EXITO" if (objetivo_cumplido and en_refugio) else "FRACASO"
            self.motivo_fin = "Se alcanzó el número máximo de pasos."

    def ejecutar(self, max_pasos: Optional[int] = None,
                 callback=None) -> Dict[str, object]:
        n = max_pasos or self.cfg.max_pasos
        while not self.terminada and self.paso_actual < n:
            r = self.paso()
            if callback:
                callback(r)
        return self.resultados()

    # ------------------------------------------------------------------ #
    def resultados(self) -> Dict[str, object]:
        e = self.estado
        res = {
            "exito": e.estado == "EXITO",
            "estado_final": e.estado,
            "motivo_fin": self.motivo_fin,
            "pasos": self.paso_actual,
            "bateria_final": round(e.bateria, 2),
            "energia_restante_pct": round(e.bateria, 2),
            "distancia_m": round(e.distancia_recorrida_m, 1),
            "celdas_recorridas": len(self.ruta_ejecutada),
            "ciencia_recolectada": round(e.valor_ciencia, 3),
            "ciencia_pct": round(100.0 * e.valor_ciencia / self.valor_ciencia_total, 1),
            "muestras": len(e.muestras),
            "desgaste_final": round(e.desgaste, 1),
            # Métricas de eficiencia: separan "hacer más" de "gastar menos"
            "celdas_por_paso": round(len(self.ruta_ejecutada) /
                                     max(1, self.paso_actual), 3),
            "energia_consumida_pct": round(
                max(0.0, self.cfg.bateria_inicial - e.bateria), 2),
            "ciencia_por_pct_bateria": round(
                e.valor_ciencia / max(1.0, self.cfg.bateria_inicial - e.bateria), 4),
            "incidentes_velocidad": sum(
                1 for ev in self.logger.eventos
                if ev["tipo"] in ("frenado_insuficiente", "deslizamiento",
                                  "frenado_en_borde")),
            **{k: (round(v, 4) if isinstance(v, float) else v)
               for k, v in self.metricas.items()},
        }
        self.logger.resultados = res
        return res

    # Interfaz para el operador (Panel E) ------------------------------- #
    def inyectar_evento(self, tipo: str) -> str:
        self.eventos.inyectar(tipo)
        return f"Evento '{tipo}' encolado para el próximo paso."

    def colocar_barranco(self, celda: Celda) -> str:
        """RF-02 / CA-02: el operador crea un precipicio frente al rover."""
        if not self.world.dentro(celda):
            return "Celda fuera del mapa."
        self.world.clase[celda[0], celda[1]] = BARRANCO
        self.world.elev[celda[0], celda[1]] -= 9.0
        self.world.gradiente = self.world._calc_gradiente()
        self.belief.confirmado[celda[0], celda[1]] = False   # el rover no lo sabe aún
        self.logger.evento(self.paso_actual, "barranco_manual",
                           f"El operador colocó un precipicio en {celda}.")
        return f"Precipicio colocado en {celda}."

    def celda_frente(self) -> Celda:
        d = DIRECCIONES_8[self.estado.rumbo_idx]
        return (int(clamp(self.estado.pos[0] + d[0], 0, self.world.filas - 1)),
                int(clamp(self.estado.pos[1] + d[1], 0, self.world.cols - 1)))

## 9 · Evaluación experimental, ablaciones y causalidad

Seis políticas comparadas **con exactamente la misma semilla, el mismo mapa y los mismos
eventos**: dos baselines (ruta más corta ignorando riesgo; AG puro sobre riesgos estimados),
tres ablaciones (sin RNA, sin difuso, sin AG) y el sistema completo.

Sobre la ablación *sin difuso*: usa velocidad **fija 0,40**, que es la velocidad media medida
del controlador difuso. Igualar la media es lo que hace honesta la comparación: lo que se
elimina es la **asignación** inteligente de velocidad —rápido donde es seguro, lento donde no—
y no el presupuesto de velocidad.

Las **pruebas de causalidad** responden a la condición de auditoría del enunciado: demostrar
que alterar la salida de cada módulo cambia la decisión o el movimiento. Cada prueba modifica
**una** salida y mantiene el resto constante.

In [ ]:
"""
ARES-X · Módulo 9: Evaluación experimental, ablaciones y pruebas de causalidad.

Contenido:
  * POLITICAS            baselines + híbrido + ablaciones (RF-14)
  * ejecutar_experimento  >= 20 semillas por escenario (RF-15) con dispersión
  * pruebas_causalidad    demostración controlada de que CADA técnica influye
                          en la decisión (condición de auditoría, sección 4)

Nota metodológica: todas las políticas se comparan con EXACTAMENTE la misma
semilla, el mismo mapa y los mismos eventos. Sin condiciones idénticas no se
puede atribuir una mejora a la arquitectura.
"""

from __future__ import annotations

import time
from dataclasses import replace
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd


# ----------------------------------------------------------------------------
# Políticas comparadas
# ----------------------------------------------------------------------------
POLITICAS: Dict[str, dict] = {
    "ruta_corta": dict(usar_rna=False, usar_difuso=False, usar_ag=False,
                       baseline_ruta_corta=True,
                       _desc="BASELINE de control: camino más corto por distancia, "
                             "sin considerar el riesgo estimado."),
    "ag_puro": dict(usar_rna=False, usar_difuso=False, usar_ag=True,
                    _desc="AG sobre riesgos estimados fijos, sin RNA ni difuso."),
    "sin_rna": dict(usar_rna=False, usar_difuso=True, usar_ag=True,
                    _desc="ABLACIÓN: se elimina la percepción neuronal."),
    "sin_difuso": dict(usar_rna=True, usar_difuso=False, usar_ag=True,
                       _desc="ABLACIÓN: velocidad fija en lugar de control difuso."),
    "sin_ag": dict(usar_rna=True, usar_difuso=True, usar_ag=False,
                   _desc="ABLACIÓN: planificación A* en lugar del AG."),
    "hibrido": dict(usar_rna=True, usar_difuso=True, usar_ag=True,
                    _desc="SISTEMA COMPLETO: RNA + lógica difusa + AG + supervisor."),
}

METRICAS = ["exito", "pasos", "ciencia_pct", "muestras", "bateria_final",
            "distancia_m", "colisiones", "barrancos_evitados",
            "violaciones_seguridad", "replanificaciones", "pasos_modo_seguro",
            "tiempo_decision_s", "celdas_por_paso", "energia_consumida_pct",
            "ciencia_por_pct_bateria", "incidentes_velocidad"]


def config_politica(base: Config, politica: str) -> Config:
    cfg = Config.from_dict({**base.__dict__})
    flags = {k: v for k, v in POLITICAS[politica].items() if not k.startswith("_")}
    for k, v in flags.items():
        setattr(cfg, k, v)
    if "baseline_ruta_corta" not in flags:
        cfg.baseline_ruta_corta = False
    return cfg


def ejecutar_episodio(base: Config, politica: str, semilla: int,
                      rna: Optional[NeuralPerception]) -> dict:
    cfg = config_politica(base, politica)
    cfg.semilla = semilla
    m = Mission(cfg, rna)
    r = m.ejecutar()
    r["politica"] = politica
    r["semilla"] = semilla
    r["escenario"] = f"{base.planeta}/{base.modo}/{base.dificultad}"
    return r


# ----------------------------------------------------------------------------
def ejecutar_experimento(escenarios: List[Config], politicas: List[str],
                         n_semillas: int, rna: Optional[NeuralPerception],
                         semilla_base: int = 1000, verbose: bool = True
                         ) -> pd.DataFrame:
    """RF-15: repite cada (escenario, política) en n_semillas independientes."""
    filas: List[dict] = []
    t0 = time.time()
    total = len(escenarios) * len(politicas) * n_semillas
    hecho = 0
    for esc in escenarios:
        for pol in politicas:
            for i in range(n_semillas):
                s = semilla_base + i
                filas.append(ejecutar_episodio(esc, pol, s, rna))
                hecho += 1
            if verbose:
                print(f"  [{hecho:4d}/{total}] {esc.planeta}/{esc.dificultad} · "
                      f"{pol:<12s} ({time.time() - t0:5.1f}s)")
    df = pd.DataFrame(filas)
    if verbose:
        print(f"  Experimento completo en {time.time() - t0:.1f} s "
              f"({len(df)} episodios).")
    return df


def resumen_metricas(df: pd.DataFrame,
                     metricas: Optional[List[str]] = None) -> pd.DataFrame:
    """Promedio, mediana y dispersión por escenario y política."""
    metricas = metricas or METRICAS
    cols = [m for m in metricas if m in df.columns]
    g = df.groupby(["escenario", "politica"])[cols]
    agg = g.agg(["mean", "median", "std"])
    agg.columns = [f"{a}_{b}" for a, b in agg.columns]
    agg["n_episodios"] = df.groupby(["escenario", "politica"]).size()
    return agg.reset_index()


def tabla_comparativa(df: pd.DataFrame) -> pd.DataFrame:
    """Tabla compacta lista para el informe."""
    filas = []
    for (esc, pol), g in df.groupby(["escenario", "politica"]):
        filas.append({
            "escenario": esc, "politica": pol, "n": len(g),
            "exito_%": round(100 * g["exito"].mean(), 1),
            "ciencia_%_media": round(g["ciencia_pct"].mean(), 1),
            "ciencia_%_sd": round(g["ciencia_pct"].std(ddof=1), 1),
            "colisiones_media": round(g["colisiones"].mean(), 2),
            "violaciones_media": round(g["violaciones_seguridad"].mean(), 2),
            "bateria_final_media": round(g["bateria_final"].mean(), 1),
            "pasos_mediana": float(g["pasos"].median()),
            "incid_velocidad_media": round(g["incidentes_velocidad"].mean(), 2),
            "celdas_por_paso": round(g["celdas_por_paso"].mean(), 3),
            "ciencia_por_bateria": round(g["ciencia_por_pct_bateria"].mean(), 4),
            "replan_media": round(g["replanificaciones"].mean(), 1),
            "t_decision_s_media": round(g["tiempo_decision_s"].mean(), 2),
        })
    out = pd.DataFrame(filas)
    orden = {k: i for i, k in enumerate(POLITICAS)}
    out["_o"] = out["politica"].map(orden).fillna(99)
    return out.sort_values(["escenario", "_o"]).drop(columns="_o").reset_index(drop=True)


def intervalo_confianza(x: np.ndarray, conf: float = 0.95) -> Tuple[float, float]:
    """IC normal aproximado de la media (n>=20 por diseño experimental)."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    if n < 2:
        return (float(x.mean()) if n else 0.0, 0.0)
    z = 1.96 if conf == 0.95 else 2.576
    err = z * x.std(ddof=1) / np.sqrt(n)
    return (float(x.mean()), float(err))


def comparar_con_baseline(df: pd.DataFrame, metrica: str = "ciencia_pct",
                          referencia: str = "hibrido") -> pd.DataFrame:
    """Diferencia de medias frente al sistema completo, con IC al 95 %."""
    filas = []
    for esc, g in df.groupby("escenario"):
        ref = g[g["politica"] == referencia][metrica].to_numpy(dtype=float)
        m_ref, e_ref = intervalo_confianza(ref)
        for pol, gp in g.groupby("politica"):
            x = gp[metrica].to_numpy(dtype=float)
            m, e = intervalo_confianza(x)
            dif = m - m_ref
            err = float(np.sqrt(e ** 2 + e_ref ** 2))
            filas.append({
                "escenario": esc, "politica": pol, "n": len(x),
                f"{metrica}_media": round(m, 2),
                f"{metrica}_ic95": round(e, 2),
                "dif_vs_" + referencia: round(dif, 2),
                "ic95_dif": round(err, 2),
                "significativo_95": bool(abs(dif) > err) and pol != referencia,
            })
    return pd.DataFrame(filas)


# ----------------------------------------------------------------------------
# Pruebas de causalidad (condición de auditoría de la sección 4)
# ----------------------------------------------------------------------------
def _preparar_mision(rna, semilla=42, pasos=6, **kw) -> Mission:
    cfg = Config(semilla=semilla, bateria_inicial=60, eventos_aleatorios=False, **kw)
    m = Mission(cfg, rna)
    for _ in range(pasos):
        if m.terminada:
            break
        m.paso()
    return m


def causalidad_rna(rna: NeuralPerception, semilla: int = 42) -> dict:
    """Variar SOLO la salida de la RNA debe cambiar la decisión del supervisor."""
    resultados = {}
    for etiqueta, p in (("p_barranco_baja", np.array([0.90, 0.07, 0.03])),
                        ("p_barranco_alta", np.array([0.10, 0.08, 0.82]))):
        m = _preparar_mision(rna, semilla)
        destino = m._destino_actual() or m.celda_frente()
        d = m.supervisor.evaluar(
            estado=m.estado, lectura=m.observar(), belief=m.belief, destino=destino,
            velocidad_difusa=0.6, cautela=0.3, prob_rna=p,
            confirmacion_borde=None, refugio_alcanzable=True, hay_sitio_aqui=False,
            ruta_factible=True, pendiente_destino=4.0)
        resultados[etiqueta] = {"accion": d.accion, "motivo": d.motivo,
                                "velocidad": round(d.velocidad, 3)}
    a, b = resultados["p_barranco_baja"], resultados["p_barranco_alta"]
    return {"tecnica": "RNA", "variable_controlada": "p(barranco) del destino",
            "resultado": resultados,
            "cambia_decision": a["accion"] != b["accion"] or a["motivo"] != b["motivo"],
            "explicacion": (f"Con p=0.03 el supervisor decide '{a['accion']}' "
                            f"({a['motivo']}); con p=0.82 decide '{b['accion']}' "
                            f"({b['motivo']}). Ninguna otra entrada cambió.")}


def causalidad_difuso(rna: NeuralPerception, semilla: int = 42) -> dict:
    """CA-05: cambiar la distancia al obstáculo debe cambiar la velocidad."""
    m = _preparar_mision(rna, semilla)
    mt = margen_termico(m.estado.temp_interna, m.cfg.perfil.limite_termico_C)
    res = {}
    for etiqueta, dist in (("distancia_10m", 10.0), ("distancia_1m", 1.0)):
        tz = m.difuso.inferir(distancia_m=dist, bateria_pct=m.estado.bateria,
                              pendiente_deg=6.0, margen_termico_C=mt)
        d = m.supervisor.evaluar(
            estado=m.estado, lectura=m.observar(), belief=m.belief,
            destino=m._destino_actual() or m.celda_frente(),
            velocidad_difusa=tz.salidas["velocidad"], cautela=tz.salidas["cautela"],
            prob_rna=np.array([0.9, 0.06, 0.04]), confirmacion_borde=LIBRE,
            refugio_alcanzable=True, hay_sitio_aqui=False, ruta_factible=True,
            pendiente_destino=4.0)
        res[etiqueta] = {"velocidad_difusa": round(tz.salidas["velocidad"], 3),
                         "cautela": round(tz.salidas["cautela"], 3),
                         "reglas": tz.resumen_texto(3),
                         "accion": d.accion,
                         "velocidad_final": round(d.velocidad, 3)}
    a, b = res["distancia_10m"], res["distancia_1m"]
    return {"tecnica": "Lógica difusa", "variable_controlada": "distancia a riesgo",
            "resultado": res,
            "cambia_decision": abs(a["velocidad_final"] - b["velocidad_final"]) > 0.05
            or a["accion"] != b["accion"],
            "explicacion": (f"A 10 m la velocidad ejecutada es {a['velocidad_final']} "
                            f"({a['accion']}); a 1 m baja a {b['velocidad_final']} "
                            f"({b['accion']}).")}


def causalidad_ag(rna: NeuralPerception, semilla: int = 42) -> dict:
    """Cambiar los pesos del fitness debe cambiar la ruta elegida."""
    res = {}
    caminos = {}
    for etiqueta, pesos in (
        ("prioriza_ciencia", {"ciencia": 0.70, "distancia": 0.05, "energia": 0.05,
                              "riesgo": 0.15, "tiempo": 0.05}),
        ("prioriza_seguridad", {"ciencia": 0.05, "distancia": 0.15, "energia": 0.15,
                                "riesgo": 0.60, "tiempo": 0.05}),
    ):
        m = _preparar_mision(rna, semilla)
        m.cfg.ag_pesos = pesos
        m.ag.cfg.ag_pesos = pesos
        m._replanificado_este_paso = False
        m.replanificar("prueba de causalidad AG")
        p = m.plan
        caminos[etiqueta] = [tuple(c) for c in p.camino]
        res[etiqueta] = {
            "fitness": round(p.fitness, 4),
            "largo_ruta": len(p.camino),
            "ciencia": round(p.componentes.get("ciencia", 0), 3),
            "riesgo": round(p.componentes.get("riesgo", 0), 3),
            "sitios_alcanzados": list(p.sitios_alcanzados),
            "genes": p.cromosoma.genes_texto()[:90],
        }
    a, b = res["prioriza_ciencia"], res["prioriza_seguridad"]
    distinta = caminos["prioriza_ciencia"] != caminos["prioriza_seguridad"]
    return {"tecnica": "Algoritmo genético", "variable_controlada": "pesos del fitness",
            "resultado": res, "cambia_decision": distinta,
            "explicacion": (f"Priorizando ciencia la ruta visita "
                            f"{len(a['sitios_alcanzados'])} sitios con riesgo "
                            f"{a['riesgo']}; priorizando seguridad visita "
                            f"{len(b['sitios_alcanzados'])} con riesgo {b['riesgo']}. "
                            f"Las rutas {'difieren' if distinta else 'coinciden'}.")}


def causalidad_supervisor(rna: NeuralPerception, semilla: int = 42) -> dict:
    """El veto duro debe primar sobre el mejor fitness."""
    m = _preparar_mision(rna, semilla)
    destino = m._destino_actual() or m.celda_frente()
    base = m.supervisor.evaluar(
        estado=m.estado, lectura=m.observar(), belief=m.belief, destino=destino,
        velocidad_difusa=0.9, cautela=0.1, prob_rna=np.array([0.95, 0.03, 0.02]),
        confirmacion_borde=LIBRE, refugio_alcanzable=True, hay_sitio_aqui=False,
        ruta_factible=True, pendiente_destino=3.0)
    vetado = m.supervisor.evaluar(
        estado=m.estado, lectura=m.observar(), belief=m.belief, destino=destino,
        velocidad_difusa=0.9, cautela=0.1, prob_rna=np.array([0.95, 0.03, 0.02]),
        confirmacion_borde=BARRANCO, refugio_alcanzable=True, hay_sitio_aqui=False,
        ruta_factible=True, pendiente_destino=3.0)
    return {"tecnica": "Supervisor", "variable_controlada": "sensor de borde",
            "resultado": {"sin_veto": {"accion": base.accion, "motivo": base.motivo,
                                       "velocidad": round(base.velocidad, 3)},
                          "con_barranco_confirmado": {"accion": vetado.accion,
                                                      "motivo": vetado.motivo,
                                                      "velocidad": round(vetado.velocidad, 3)}},
            "cambia_decision": base.accion != vetado.accion,
            "explicacion": (f"Con la MISMA recomendación de IA (velocidad 0.9, "
                            f"p_libre 0.95) la decisión pasa de '{base.accion}' a "
                            f"'{vetado.accion}' ({vetado.motivo}) al confirmarse el "
                            f"precipicio: la restricción dura prima sobre el fitness.")}


def pruebas_causalidad(rna: NeuralPerception, semilla: int = 42
                       ) -> Tuple[pd.DataFrame, List[dict]]:
    pruebas = [causalidad_rna(rna, semilla), causalidad_difuso(rna, semilla),
               causalidad_ag(rna, semilla), causalidad_supervisor(rna, semilla)]
    df = pd.DataFrame([{"tecnica": p["tecnica"],
                        "variable_controlada": p["variable_controlada"],
                        "influye_en_la_decision": "SÍ" if p["cambia_decision"] else "NO",
                        "evidencia": p["explicacion"]} for p in pruebas])
    return df, pruebas


# ----------------------------------------------------------------------------
def verificar_determinismo(base: Config, rna, semilla: int = 42) -> dict:
    """Condición de borde: misma semilla y configuración -> traza equivalente."""
    r1 = Mission(Config.from_dict({**base.__dict__, "semilla": semilla}), rna).ejecutar()
    r2 = Mission(Config.from_dict({**base.__dict__, "semilla": semilla}), rna).ejecutar()
    iguales = all(r1.get(k) == r2.get(k) for k in
                  ["pasos", "estado_final", "ciencia_recolectada", "distancia_m",
                   "colisiones", "replanificaciones"])
    return {"determinista": bool(iguales), "corrida_1": r1, "corrida_2": r2}

## 10 · Centro de Control de Misión (paneles A–E)

Interfaz con `ipywidgets` y `matplotlib`. Panel A: mapa con ruta planificada (verde discontinua),
ruta realmente ejecutada (azul continua), rutas candidatas del AG (punteadas), peligros, zonas
prohibidas, refugios, sitios científicos y rover. Panel B: telemetría con unidades, límite y
estado de cada sensor. Panel C: explicación por algoritmo con valores concretos. Panel D:
historia temporal filtrable. Panel E: controles de simulación e inyección de eventos.

In [ ]:
"""
ARES-X · Módulo 10: Centro de Control de Misión (ipywidgets + matplotlib).

Paneles exigidos por la sección 8 del caso:
  Panel A  mapa con rutas planificada/ejecutada, peligros, objetivos y rover
  Panel B  telemetría con unidades, límites y estado de cada sensor
  Panel C  explicación por algoritmo con valores concretos
  Panel D  historia temporal filtrable y exportable
  Panel E  controles de simulación e inyección de eventos

Vistas secundarias: Laboratorio RNA, Laboratorio Difuso, Laboratorio Genético y
Comparación de políticas.
"""

from __future__ import annotations

from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Circle, Rectangle

import ipywidgets as W
from IPython.display import display, clear_output, HTML


# Paleta accesible y coherente en claro/oscuro
C_LIBRE = "#efe4d2"
C_OBST = "#6b5844"
C_BARR = "#1d1b1a"
C_RUTA_EJEC = "#1f6feb"
C_RUTA_PLAN = "#1a9850"
C_ROVER = "#d94801"
C_REFUGIO = "#2b8cbe"
C_CIENCIA = "#54a24b"
C_PROHIBIDO = "#d62728"
C_ALERTA = "#e6a700"


def _css() -> str:
    return """
    <style>
    .ax-wrap{font-family:ui-sans-serif,system-ui,-apple-system,sans-serif;}
    .ax-title{font-size:15px;font-weight:700;letter-spacing:.04em;margin:0 0 6px 0;}
    .ax-card{border:1px solid rgba(128,128,128,.35);border-radius:8px;padding:8px 10px;
             margin:0 0 8px 0;}
    .ax-kv{display:grid;grid-template-columns:auto 1fr auto;gap:2px 10px;font-size:12px;}
    .ax-k{opacity:.75;}
    .ax-v{font-variant-numeric:tabular-nums;font-weight:600;text-align:right;}
    .ax-ok{color:#2e7d32;} .ax-warn{color:#e6a700;} .ax-bad{color:#c62828;}
    .ax-tag{display:inline-block;padding:1px 7px;border-radius:10px;font-size:11px;
            font-weight:700;border:1px solid currentColor;}
    .ax-small{font-size:11px;opacity:.8;}
    table.ax-t{border-collapse:collapse;font-size:11.5px;width:100%;}
    table.ax-t th,table.ax-t td{border-bottom:1px solid rgba(128,128,128,.25);
            padding:3px 6px;text-align:left;}
    table.ax-t th{opacity:.7;font-weight:600;}
    </style>"""


def _tag(txt: str, clase: str = "ax-ok") -> str:
    return f'<span class="ax-tag {clase}">{txt}</span>'


def _estado_clase(estado: str) -> str:
    if estado in ("EXITO",):
        return "ax-ok"
    if estado in ("FRACASO", "MODO_SEGURO"):
        return "ax-bad"
    if estado in ("ALERTA", "REPLANIFICACION", "REFUGIO"):
        return "ax-warn"
    return ""


# ============================================================================
class Dashboard:
    """Centro de Control de Misión interactivo."""

    def __init__(self, cfg: Optional[Config] = None, rna=None,
                 figsize: Tuple[float, float] = (6.4, 6.0)):
        self.cfg_base = cfg or Config()
        self.rna = rna
        self.figsize = figsize
        self.mision: Optional[Mission] = None
        self.autoplay = False
        self._construir_widgets()
        self.reiniciar()

    # ---------------------------------------------------------------- #
    # Construcción de la interfaz
    # ---------------------------------------------------------------- #
    def _construir_widgets(self) -> None:
        c = self.cfg_base
        est = {"description_width": "110px"}
        ancho = W.Layout(width="250px")

        self.w_planeta = W.Dropdown(
            options=[(PERFILES[k].nombre, k) for k in PERFILES],
            value=c.planeta, description="Planeta:", style=est, layout=ancho)
        self.w_modo = W.Dropdown(options=[("Rover rodante", "rover"),
                                          ("Sonda atmosférica", "sonda")],
                                 value=c.modo, description="Modo:", style=est,
                                 layout=ancho)
        self.w_dificultad = W.Dropdown(options=["facil", "media", "dificil"],
                                       value=c.dificultad, description="Dificultad:",
                                       style=est, layout=ancho)
        self.w_semilla = W.BoundedIntText(value=c.semilla, min=0, max=10**6,
                                          description="Semilla:", style=est,
                                          layout=ancho)
        self.w_bateria = W.FloatSlider(value=c.bateria_inicial, min=1, max=100, step=1,
                                       description="Batería %:", style=est,
                                       layout=ancho, continuous_update=False)
        self.w_tam = W.IntSlider(value=c.filas, min=10, max=40, step=2,
                                 description="Lado mapa:", style=est, layout=ancho,
                                 continuous_update=False)
        self.w_sitios = W.IntSlider(value=c.n_sitios_ciencia, min=0, max=12,
                                    description="Sitios:", style=est, layout=ancho,
                                    continuous_update=False)
        self.w_prioridad = W.FloatSlider(value=c.prioridad_ciencia, min=0.1, max=2.0,
                                         step=0.1, description="Prior. ciencia:",
                                         style=est, layout=ancho,
                                         continuous_update=False)
        self.w_usar_rna = W.Checkbox(value=c.usar_rna, description="Usar RNA",
                                     indent=False)
        self.w_usar_difuso = W.Checkbox(value=c.usar_difuso, description="Usar difuso",
                                        indent=False)
        self.w_usar_ag = W.Checkbox(value=c.usar_ag, description="Usar AG", indent=False)
        self.w_eventos_rnd = W.Checkbox(value=c.eventos_aleatorios,
                                        description="Eventos aleatorios", indent=False)

        self.b_aplicar = W.Button(description="Aplicar y reiniciar", icon="refresh",
                                  button_style="primary")
        self.b_paso = W.Button(description="Un paso", icon="step-forward")
        self.b_correr = W.Button(description="Correr", icon="play",
                                 button_style="success")
        self.b_pausa = W.Button(description="Pausa", icon="pause", button_style="warning")
        self.b_reiniciar = W.Button(description="Reiniciar", icon="undo")
        self.w_n_pasos = W.BoundedIntText(value=25, min=1, max=400, description="pasos:",
                                          layout=W.Layout(width="140px"),
                                          style={"description_width": "45px"})
        self.b_csv = W.Button(description="Exportar CSV", icon="download")
        self.b_json = W.Button(description="Exportar JSON", icon="download")

        self.w_evento = W.Dropdown(options=EventManager.CATALOGO, value="sismo",
                                   description="Evento:", style=est,
                                   layout=W.Layout(width="230px"))
        self.b_evento = W.Button(description="Inyectar", icon="bolt",
                                 button_style="danger")
        self.b_barranco = W.Button(description="Barranco al frente", icon="exclamation",
                                   button_style="danger")
        self.b_fallo_lidar = W.Button(description="NaN en LIDAR", icon="bug")

        self.w_filtro = W.Text(value="", description="Filtrar:",
                               placeholder="VETO, AVANZAR, sismo...",
                               style=est, layout=W.Layout(width="260px"))
        self.w_n_hist = W.IntSlider(value=12, min=5, max=60, description="Filas:",
                                    style=est, layout=W.Layout(width="230px"),
                                    continuous_update=False)

        self.out_mapa = W.Output()
        self.out_telemetria = W.Output()
        self.out_explicacion = W.Output()
        self.out_historial = W.Output()
        self.out_mensajes = W.Output()

        # --- handlers -------------------------------------------------
        self.b_aplicar.on_click(lambda _: self.reiniciar(desde_widgets=True))
        self.b_reiniciar.on_click(lambda _: self.reiniciar(desde_widgets=True))
        self.b_paso.on_click(lambda _: self.avanzar(1))
        self.b_correr.on_click(lambda _: self.avanzar(int(self.w_n_pasos.value)))
        self.b_pausa.on_click(lambda _: self._pausar())
        self.b_csv.on_click(lambda _: self._exportar("csv"))
        self.b_json.on_click(lambda _: self._exportar("json"))
        self.b_evento.on_click(lambda _: self._inyectar(self.w_evento.value))
        self.b_barranco.on_click(lambda _: self._colocar_barranco())
        self.b_fallo_lidar.on_click(lambda _: self._fallo("lidar"))
        self.w_filtro.observe(lambda ch: self._pintar_historial(), names="value")
        self.w_n_hist.observe(lambda ch: self._pintar_historial(), names="value")
        self.w_planeta.observe(self._sincronizar_modo, names="value")

    def _sincronizar_modo(self, change) -> None:
        """CA-07: si el planeta no admite rover, la interfaz lo impide."""
        perfil = PERFILES[self.w_planeta.value]
        if not perfil.permite_rover:
            self.w_modo.value = "sonda"
            self.w_modo.disabled = True
            with self.out_mensajes:
                clear_output(wait=True)
                ok, msg = perfil.compatible("rover")
                display(HTML(f'<div class="ax-card"><b>Modo rover bloqueado.</b> '
                             f'<span class="ax-small">{msg}</span></div>'))
        else:
            self.w_modo.disabled = False
            with self.out_mensajes:
                clear_output(wait=True)

    # ---------------------------------------------------------------- #
    def _cfg_desde_widgets(self) -> Config:
        d = {**self.cfg_base.__dict__}
        d.update(dict(
            planeta=self.w_planeta.value, modo=self.w_modo.value,
            dificultad=self.w_dificultad.value, semilla=int(self.w_semilla.value),
            bateria_inicial=float(self.w_bateria.value),
            filas=int(self.w_tam.value), columnas=int(self.w_tam.value),
            n_sitios_ciencia=int(self.w_sitios.value),
            prioridad_ciencia=float(self.w_prioridad.value),
            usar_rna=bool(self.w_usar_rna.value),
            usar_difuso=bool(self.w_usar_difuso.value),
            usar_ag=bool(self.w_usar_ag.value),
            eventos_aleatorios=bool(self.w_eventos_rnd.value),
        ))
        return Config.from_dict(d)

    def reiniciar(self, desde_widgets: bool = False) -> None:
        cfg = self._cfg_desde_widgets() if desde_widgets else self.cfg_base
        ok, errores = cfg.validar()
        with self.out_mensajes:
            clear_output(wait=True)
            if not ok:
                display(HTML('<div class="ax-card ax-bad"><b>Configuración '
                             'rechazada:</b><ul class="ax-small">' +
                             "".join(f"<li>{e}</li>" for e in errores) +
                             "</ul></div>"))
                return
        self.autoplay = False
        self.mision = Mission(cfg, self.rna)
        self.actualizar()

    def avanzar(self, n: int = 1) -> None:
        if self.mision is None:
            return
        self.autoplay = True
        for _ in range(max(1, n)):
            if not self.autoplay or self.mision.terminada:
                break
            self.mision.paso()
        self.autoplay = False
        self.actualizar()

    def _pausar(self) -> None:
        self.autoplay = False

    def _inyectar(self, tipo: str) -> None:
        if self.mision:
            msg = self.mision.inyectar_evento(tipo)
            self.mision.paso()
            with self.out_mensajes:
                clear_output(wait=True)
                display(HTML(f'<div class="ax-card ax-small">{msg}</div>'))
            self.actualizar()

    def _colocar_barranco(self) -> None:
        if self.mision:
            celda = self.mision.celda_frente()
            msg = self.mision.colocar_barranco(celda)
            with self.out_mensajes:
                clear_output(wait=True)
                display(HTML(f'<div class="ax-card ax-small">{msg} '
                             f'El rover aún no lo sabe: debe detectarlo.</div>'))
            self.actualizar()

    def _fallo(self, sensor: str) -> None:
        if self.mision:
            self.mision.sensores.forzar_fallo(sensor, "nan")
            self.mision.paso()
            self.actualizar()

    def _exportar(self, tipo: str) -> None:
        if not self.mision:
            return
        if tipo == "csv":
            r = self.mision.logger.exportar_csv("ares_x_traza.csv")
        else:
            self.mision.resultados()
            r = self.mision.logger.exportar_json(
                "ares_x_mision.json",
                extra={"metricas_rna": (self.rna.metricas_test if self.rna else {}),
                       "resumen_mundo": self.mision.world.resumen()})
        with self.out_mensajes:
            clear_output(wait=True)
            display(HTML(f'<div class="ax-card ax-small">Archivo generado: '
                         f'<code>{r}</code></div>'))

    # ---------------------------------------------------------------- #
    # Panel A: mapa
    # ---------------------------------------------------------------- #
    def dibujar_mapa(self, ax, mostrar_candidatas: bool = True,
                     mostrar_creencia: bool = False) -> None:
        m = self.mision
        w = m.world
        base = w.elev.copy()
        ax.imshow(base, cmap="pink_r", alpha=0.55, origin="upper",
                  interpolation="nearest")

        if mostrar_creencia:
            ax.imshow(m.belief.mapa_riesgo(), cmap="Reds", alpha=0.45,
                      origin="upper", vmin=0, vmax=1, interpolation="nearest")
        else:
            cap = np.zeros((w.filas, w.cols, 4))
            cap[w.clase == OBSTACULO] = matplotlib.colors.to_rgba(C_OBST, 0.95)
            cap[w.clase == BARRANCO] = matplotlib.colors.to_rgba(C_BARR, 0.95)
            ax.imshow(cap, origin="upper", interpolation="nearest")

        # zonas prohibidas declaradas por el supervisor
        ff, cc = np.where(m.belief.prohibido)
        ax.scatter(cc, ff, marker="x", s=28, c=C_PROHIBIDO, linewidths=1.2,
                   label="zona prohibida", zorder=4)

        # zonas de riesgo dinámicas (meteorito, contacto ficticio)
        for z in w.zonas_riesgo:
            ax.add_patch(Circle((z["centro"][1], z["centro"][0]), z["radio"],
                                fill=False, ec=C_ALERTA, lw=1.8, ls="--", zorder=4))

        # refugios y sitios científicos
        if w.refugios:
            rf = np.array(w.refugios)
            ax.scatter(rf[:, 1], rf[:, 0], marker="s", s=120, c=C_REFUGIO,
                       edgecolors="white", linewidths=1.2, label="refugio", zorder=5)
        pend = [s for s in w.sitios if not s.recolectado]
        hecho = [s for s in w.sitios if s.recolectado]
        if pend:
            p = np.array([s.pos for s in pend])
            ax.scatter(p[:, 1], p[:, 0], marker="*", s=180, c=C_CIENCIA,
                       edgecolors="white", linewidths=0.8, label="sitio pendiente",
                       zorder=5)
        if hecho:
            p = np.array([s.pos for s in hecho])
            ax.scatter(p[:, 1], p[:, 0], marker="*", s=110, c="#9e9e9e",
                       edgecolors="white", linewidths=0.8, label="sitio recolectado",
                       zorder=5)
        if w.anomalia_bio:
            ax.scatter([w.anomalia_bio[1]], [w.anomalia_bio[0]], marker="P", s=130,
                       c="#9467bd", edgecolors="white",
                       label="señal anómala (no es vida)", zorder=6)
        if w.alienigena:
            ax.scatter([w.alienigena[1]], [w.alienigena[0]], marker="X", s=140,
                       c="#17becf", edgecolors="white",
                       label="contacto ficticio", zorder=6)

        # rutas candidatas del AG (punteadas)
        if mostrar_candidatas and m.candidatas:
            for i, cand in enumerate(m.candidatas[1:3], start=2):
                if len(cand.camino) > 1:
                    arr = np.array(cand.camino)
                    ax.plot(arr[:, 1], arr[:, 0], ls=":", lw=1.3, alpha=0.85,
                            color="#7fbf7b", label=f"candidata {i}", zorder=3)

        # ruta planificada vigente y ruta realmente ejecutada (RF-10)
        if len(m.ruta_planificada) > 1:
            arr = np.array(m.ruta_planificada)
            ax.plot(arr[:, 1], arr[:, 0], ls="--", lw=2.0, color=C_RUTA_PLAN,
                    label="ruta planificada", zorder=3)
        if len(m.ruta_ejecutada) > 1:
            arr = np.array(m.ruta_ejecutada)
            ax.plot(arr[:, 1], arr[:, 0], ls="-", lw=2.4, color=C_RUTA_EJEC,
                    alpha=0.95, label="ruta ejecutada", zorder=4)

        # rover
        ax.scatter([m.estado.pos[1]], [m.estado.pos[0]], marker="^", s=240,
                   c=C_ROVER, edgecolors="white", linewidths=1.5, label="ARES-X",
                   zorder=7)

        sol = f"SOL {m.paso_actual:03d}"
        ax.set_title(f"ARES-X | CENTRO DE CONTROL DE MISIÓN · "
                     f"{w.perfil.nombre.upper()} • {sol} • "
                     f"{'NOCHE' if w.es_noche() else 'DÍA'}", fontsize=10.5,
                     fontweight="bold")
        ax.set_xlabel(f"columna  (1 celda = {CELDA_M:.0f} m simulados)", fontsize=8)
        ax.set_ylabel("fila", fontsize=8)
        ax.set_xticks(range(0, w.cols, max(1, w.cols // 10)))
        ax.set_yticks(range(0, w.filas, max(1, w.filas // 10)))
        ax.tick_params(labelsize=7)
        ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=7.2,
                  frameon=False)
        ax.grid(color="gray", alpha=0.15, lw=0.4)

    def _pintar_mapa(self) -> None:
        with self.out_mapa:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=self.figsize)
            self.dibujar_mapa(ax)
            plt.tight_layout()
            plt.show()
            plt.close(fig)

    # ---------------------------------------------------------------- #
    # Panel B: telemetría
    # ---------------------------------------------------------------- #
    def _html_telemetria(self) -> str:
        m = self.mision
        filas = m.logger.filas
        if not filas:
            return '<div class="ax-card">Sin lecturas aún. Pulse «Un paso».</div>'
        f = filas[-1]
        e, w = m.estado, m.world
        lo, hi = m.cfg.perfil.limite_termico_C
        mt = margen_termico(e.temp_interna, (lo, hi))

        def kv(k, v, unidad="", clase="", limite=""):
            return (f'<div class="ax-k">{k}</div>'
                    f'<div class="ax-small">{limite}</div>'
                    f'<div class="ax-v {clase}">{v}{unidad}</div>')

        bat_cl = "ax-ok" if e.bateria > 30 else ("ax-warn" if e.bateria > 12 else "ax-bad")
        t_cl = "ax-ok" if mt > 25 else ("ax-warn" if mt > 8 else "ax-bad")
        riesgo = f.get("riesgo_destino", 0.0)
        r_cl = "ax-ok" if riesgo < 0.35 else ("ax-warn" if riesgo < 0.6 else "ax-bad")

        sensores_mal = [k for k, v in (m.logger.filas[-1].get("sensores_invalidos") or
                                       "").split("|") if False] if False else \
            [s for s in (f.get("sensores_invalidos") or "").split("|") if s]

        html = [_css(), '<div class="ax-wrap">',
                '<div class="ax-card"><div class="ax-title">PANEL B · TELEMETRÍA</div>',
                f'<div style="margin-bottom:6px;">'
                f'{_tag(e.estado, _estado_clase(e.estado))} '
                f'<span class="ax-small">paso {m.paso_actual} · '
                f'pos ({e.pos[0]},{e.pos[1]}) · rumbo {e.rumbo_idx*45}°</span></div>',
                '<div class="ax-kv">',
                kv("Batería", f"{e.bateria:.1f}", " %", bat_cl,
                   f"reserva {m.cfg.reserva_energia_pct:.0f}%"),
                kv("Potencia solar", f"{f.get('sensor_potencia_solar_W', 0):.1f}", " W",
                   "", f"irr. {w.irradiancia():.2f}×Tierra"),
                kv("Temp. interior", f"{e.temp_interna:.1f}", " °C", t_cl,
                   f"HW [{lo:.0f},{hi:.0f}]"),
                kv("Temp. exterior", f"{f.get('sensor_temp_exterior_C', 0):.1f}", " °C",
                   "", f"ambiente {w.perfil.nombre}"),
                kv("Margen térmico", f"{mt:.1f}", " °C", t_cl, "entrada difusa"),
                kv("Prox. frontal", f"{f.get('sensor_prox_frontal_m', 0):.1f}", " m",
                   "", "rango 0–20"),
                kv("Inclinación", f"{f.get('sensor_inclinacion_deg', 0):.1f}", " °", "",
                   f"máx vuelco {m.cfg.pendiente_max_deg:.0f}°"),
                kv("Riesgo destino", f"{riesgo:.2f}", "", r_cl, "0–1 estimado"),
                kv("Aceleración", f"{f.get('sensor_aceleracion', 0):.2f}", " m/s²", "",
                   f"sismo {w.sismo:.2f}"),
                kv("Radiación", f"{f.get('sensor_radiacion_mSv_h', 0):.3f}", " mSv/h",
                   "", "escala ilustrativa"),
                kv("Presión", f"{f.get('sensor_presion_bar', 0):.4g}", " bar", "",
                   "compatibilidad"),
                kv("O₂ ambiental", f"{f.get('sensor_o2_pct', 0):.2f}", " % vol", "",
                   "composición, no respiración"),
                kv("CO₂ ambiental", f"{f.get('sensor_co2_pct', 0):.2f}", " % vol", "",
                   "composición"),
                kv("Partículas", f"{f.get('sensor_particulas', 0):.2f}", "", "",
                   f"visib. {f.get('sensor_visibilidad_m', 0):.0f} m"),
                kv("Odometría", f"{f.get('sensor_odometria_m', 0):.1f}", " m", "",
                   f"real {e.distancia_recorrida_m:.1f} m (deriva)"),
                kv("Estado mecánico", f"{100 - min(100, e.desgaste):.0f}", " %",
                   "ax-ok" if e.desgaste < 60 else "ax-warn", "desgaste acumulado"),
                kv("Latencia enlace", f"{f.get('sensor_latencia_enlace_s', 0):.2f}",
                   " s", "", "autonomía local si falla"),
                kv("Ciencia", f"{f.get('ciencia_acumulada', 0):.2f}", "", "",
                   f"{len(e.muestras)} muestras"),
                '</div>']
        if sensores_mal:
            html.append(f'<div class="ax-small ax-bad" style="margin-top:6px;">'
                        f'⚠ Sensores inválidos: {", ".join(sensores_mal)} '
                        f'→ degradación segura.</div>')
        if w.anomalia_bio:
            html.append('<div class="ax-small" style="margin-top:6px;">'
                        '🜚 Señal anómala registrada. <b>No constituye detección de '
                        'vida</b>; requiere análisis independiente.</div>')
        html.append("</div></div>")
        return "".join(html)

    def _pintar_telemetria(self) -> None:
        with self.out_telemetria:
            clear_output(wait=True)
            display(HTML(self._html_telemetria()))

    # ---------------------------------------------------------------- #
    # Panel C: explicación por algoritmo
    # ---------------------------------------------------------------- #
    def _html_explicacion(self) -> str:
        m = self.mision
        d = m.ultima_decision
        if d is None:
            return '<div class="ax-card">Sin decisiones aún.</div>'
        p = m.prob_rna
        tz = m.traza_difusa
        plan = m.plan

        bloques = [_css(), '<div class="ax-wrap"><div class="ax-card">',
                   '<div class="ax-title">PANEL C · EXPLICACIÓN POR ALGORITMO</div>']

        # RNA
        if p is not None:
            pred = NOMBRES_CLASES[int(np.argmax(p))]
            bloques.append(
                f'<div class="ax-small"><b>RNA</b> — predicción <b>{pred}</b> '
                f'(confianza {float(np.max(p)):.2f})<br>'
                f'libre {p[LIBRE]:.2f} · obstáculo {p[OBSTACULO]:.2f} · '
                f'barranco {p[BARRANCO]:.2f} · umbral de veto '
                f'{m.cfg.umbral_rna_barranco:.2f}</div>')
        else:
            bloques.append('<div class="ax-small"><b>RNA</b> — desactivada '
                           '(ablación): la creencia usa sólo el sensor de borde.</div>')

        # Difuso
        if tz is not None:
            v = tz.salidas.get("velocidad", 0.0)
            ct = tz.salidas.get("cautela", 0.0)
            bloques.append(
                f'<div class="ax-small" style="margin-top:6px;"><b>DIFUSO</b> — '
                f'velocidad <b>{v:.3f}</b>, cautela <b>{ct:.3f}</b> '
                f'({tz.n_reglas_activas} reglas activas'
                f'{", SALIDA DE RESPALDO" if tz.fallback else ""})<br>'
                f'{tz.resumen_texto(3)}</div>')
            if tz.membresias:
                items = []
                for var, grados in tz.membresias.items():
                    act = [f"{k} {g:.2f}" for k, g in grados.items()
                           if g > 0.01 and not k.startswith("NO ")]
                    if act:
                        items.append(f"{var}: " + ", ".join(act))
                bloques.append('<div class="ax-small" style="opacity:.75;">μ → ' +
                               " | ".join(items) + "</div>")

        # AG
        if plan is not None and m.cfg.usar_ag:
            comp = plan.componentes
            bloques.append(
                f'<div class="ax-small" style="margin-top:6px;"><b>AG</b> — '
                f'{len(m.candidatas)} rutas candidatas; mejor fitness '
                f'<b>{plan.fitness:+.3f}</b> '
                f'({"factible" if plan.factible else "INVIABLE"})<br>'
                f'ciencia {comp.get("aporte_ciencia", 0):+.3f} · '
                f'distancia {comp.get("aporte_distancia", 0):+.3f} · '
                f'energía {comp.get("aporte_energia", 0):+.3f} · '
                f'riesgo {comp.get("aporte_riesgo", 0):+.3f} · '
                f'tiempo {comp.get("aporte_tiempo", 0):+.3f} · '
                f'penalización −{comp.get("penalizacion", 0):.3f}<br>'
                f'genes: <code>{plan.cromosoma.genes_texto()[:110]}</code></div>')
        elif not m.cfg.usar_ag:
            bloques.append('<div class="ax-small" style="margin-top:6px;"><b>AG</b> — '
                           'desactivado (ablación): planificación A*.</div>')

        # Supervisor
        clase = "ax-bad" if d.vetos else "ax-ok"
        bloques.append(
            f'<div style="margin-top:8px;"><b class="ax-small">ÁRBITRO</b> '
            f'{_tag(d.accion, clase)} '
            f'<span class="ax-small">velocidad ejecutada '
            f'<b>{d.velocidad:.3f}</b></span><br>'
            f'<span class="ax-small">{d.explicacion()}</span></div>')
        if d.permitidos:
            bloques.append('<div class="ax-small" style="opacity:.75;">'
                           'condiciones verificadas: ' + ", ".join(d.permitidos) +
                           "</div>")
        bloques.append("</div></div>")
        return "".join(bloques)

    def _pintar_explicacion(self) -> None:
        with self.out_explicacion:
            clear_output(wait=True)
            display(HTML(self._html_explicacion()))

    # ---------------------------------------------------------------- #
    # Panel D: historial
    # ---------------------------------------------------------------- #
    def _pintar_historial(self) -> None:
        m = self.mision
        with self.out_historial:
            clear_output(wait=True)
            if m is None or not m.logger.filas:
                display(HTML('<div class="ax-card">Sin historial.</div>'))
                return
            df = m.logger.df()
            cols = [c for c in ["paso", "estado_mision", "accion", "motivo",
                                "velocidad", "bateria_pct", "rna_p_barranco",
                                "difuso_velocidad", "ag_fitness", "eventos"]
                    if c in df.columns]
            vista = df[cols].copy()
            filtro = (self.w_filtro.value or "").strip().lower()
            if filtro:
                mask = vista.apply(
                    lambda r: filtro in " ".join(str(x).lower() for x in r.values),
                    axis=1)
                vista = vista[mask]
            n = int(self.w_n_hist.value)
            display(HTML(_css() +
                         '<div class="ax-wrap"><div class="ax-card">'
                         '<div class="ax-title">PANEL D · HISTORIA TEMPORAL</div>' +
                         vista.tail(n).to_html(index=False, classes="ax-t",
                                               border=0, float_format="%.3f") +
                         f'<div class="ax-small">{len(vista)} de {len(df)} pasos '
                         f'tras el filtro. Eventos registrados: '
                         f'{len(m.logger.eventos)}.</div></div></div>'))
            if m.logger.eventos:
                ev = m.logger.df_eventos().tail(6)
                display(HTML(_css() + '<div class="ax-wrap"><div class="ax-card">'
                             '<div class="ax-title">EVENTOS RECIENTES</div>' +
                             ev.to_html(index=False, classes="ax-t", border=0) +
                             "</div></div>"))

    # ---------------------------------------------------------------- #
    def actualizar(self) -> None:
        self._pintar_mapa()
        self._pintar_telemetria()
        self._pintar_explicacion()
        self._pintar_historial()

    # ---------------------------------------------------------------- #
    def panel_control(self) -> W.Widget:
        cfg_box = W.VBox([
            W.HTML(_css() + '<div class="ax-title">CONFIGURACIÓN (RF-01)</div>'),
            self.w_planeta, self.w_modo, self.w_dificultad, self.w_semilla,
            self.w_tam, self.w_sitios, self.w_bateria, self.w_prioridad,
            W.HTML('<div class="ax-small">Ablaciones (RF-14):</div>'),
            W.HBox([self.w_usar_rna, self.w_usar_difuso, self.w_usar_ag]),
            self.w_eventos_rnd, self.b_aplicar,
        ])
        ctrl_box = W.VBox([
            W.HTML(_css() + '<div class="ax-title">PANEL E · CONTROLES Y EVENTOS</div>'),
            W.HBox([self.b_paso, self.b_correr, self.w_n_pasos, self.b_pausa,
                    self.b_reiniciar]),
            W.HBox([self.w_evento, self.b_evento]),
            W.HBox([self.b_barranco, self.b_fallo_lidar]),
            W.HBox([self.b_csv, self.b_json]),
            self.out_mensajes,
        ])
        izquierda = W.VBox([self.out_mapa], layout=W.Layout(width="56%"))
        derecha = W.VBox([self.out_telemetria, self.out_explicacion],
                         layout=W.Layout(width="44%"))
        return W.VBox([
            W.HBox([cfg_box, ctrl_box]),
            W.HBox([izquierda, derecha]),
            W.HBox([self.w_filtro, self.w_n_hist]),
            self.out_historial,
        ])

## 11 · Laboratorios de inspección y comparador de políticas

Las cuatro vistas secundarias exigidas: Laboratorio RNA (rasgos → normalización → capas →
probabilidades, historial de entrenamiento con selector de época, matriz de confusión,
calibración y baselines), Laboratorio Difuso (curvas de membresía con el punto de entrada
marcado, reglas disparadas con su grado y justificación, agregación y centroide, activación y
desactivación de reglas, superficie de control), Laboratorio Genético (población por generación,
aptitud desglosada, torneos, cruces y mutaciones concretas, convergencia, y replanificación con
pesos modificados) y Comparación de políticas.

In [ ]:
"""
ARES-X · Módulo 11: Vistas secundarias (laboratorios de inspección).

  LaboratorioRNA        entrada -> normalización -> capas -> probabilidades,
                        historial de entrenamiento, matriz de confusión,
                        calibración y baselines.
  LaboratorioDifuso     curvas de membresía, reglas disparadas, agregación y
                        centroide; activar/desactivar reglas; superficie.
  LaboratorioGenetico   población por generación, aptitud desglosada, torneos,
                        cruces, mutaciones y convergencia.
  ComparadorPoliticas   híbrido vs baselines vs ablaciones con igual semilla.
"""

from __future__ import annotations

from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output, HTML


# ============================================================================
class LaboratorioRNA:
    """Inspección completa del módulo de percepción neuronal."""

    def __init__(self, rna, dataset: Optional[Dataset] = None):
        self.rna = rna
        self.ds = dataset
        self.sliders: Dict[str, W.FloatSlider] = {}
        for nombre in RASGOS:
            lo, hi = RANGOS_RASGOS[nombre]
            self.sliders[nombre] = W.FloatSlider(
                value=float(np.mean([lo, hi])), min=lo, max=hi,
                step=(hi - lo) / 100.0, description=nombre,
                style={"description_width": "135px"},
                layout=W.Layout(width="330px"), continuous_update=False,
                readout_format=".2f")
        self.w_preset = W.Dropdown(
            options=["(manual)", "celda libre típica", "obstáculo frontal",
                     "borde de barranco", "lectura ambigua con polvo"],
            value="celda libre típica", description="Caso:",
            style={"description_width": "60px"}, layout=W.Layout(width="330px"))
        self.w_epoca = W.IntSlider(value=0, min=0, max=1, description="Época:",
                                   continuous_update=False,
                                   layout=W.Layout(width="420px"))
        if rna is not None and rna.historial["epoca"]:
            self.w_epoca.max = len(rna.historial["epoca"]) - 1
            self.w_epoca.value = self.w_epoca.max
        self.out_pred = W.Output()
        self.out_hist = W.Output()
        self.out_diag = W.Output()

        for s in self.sliders.values():
            s.observe(lambda ch: self._predecir(), names="value")
        self.w_preset.observe(self._aplicar_preset, names="value")
        self.w_epoca.observe(lambda ch: self._pintar_historial(), names="value")
        self._aplicar_preset(None)
        self._pintar_historial()
        self._pintar_diagnostico()

    # -- presets ------------------------------------------------------------
    def _aplicar_preset(self, change) -> None:
        p = self.w_preset.value
        valores = {
            "celda libre típica": dict(prox_frontal_m=18.5, delta_altura_m=0.2,
                                       pendiente_deg=6.0, contraste_borde=1.2,
                                       vibracion=0.3, textura=0.4, reflectancia=0.55,
                                       delta_temp_C=0.3, visibilidad=0.95,
                                       dispersion_lidar=0.5),
            "obstáculo frontal": dict(prox_frontal_m=4.5, delta_altura_m=1.8,
                                      pendiente_deg=24.0, contraste_borde=4.0,
                                      vibracion=0.7, textura=0.85, reflectancia=0.6,
                                      delta_temp_C=0.5, visibilidad=0.92,
                                      dispersion_lidar=1.0),
            "borde de barranco": dict(prox_frontal_m=19.2, delta_altura_m=-8.5,
                                      pendiente_deg=58.0, contraste_borde=9.5,
                                      vibracion=0.5, textura=0.45, reflectancia=0.2,
                                      delta_temp_C=-4.0, visibilidad=0.9,
                                      dispersion_lidar=2.1),
            "lectura ambigua con polvo": dict(prox_frontal_m=11.0,
                                              delta_altura_m=-2.0, pendiente_deg=28.0,
                                              contraste_borde=3.5, vibracion=0.6,
                                              textura=0.55, reflectancia=0.45,
                                              delta_temp_C=-1.0, visibilidad=0.25,
                                              dispersion_lidar=1.4),
        }.get(p)
        if valores:
            for k, v in valores.items():
                self.sliders[k].value = float(v)
        self._predecir()

    def vector(self) -> np.ndarray:
        return np.array([self.sliders[n].value for n in RASGOS], dtype=float)

    # -- predicción ---------------------------------------------------------
    def _predecir(self) -> None:
        with self.out_pred:
            clear_output(wait=True)
            if self.rna is None or not self.rna.entrenado:
                display(HTML('<div class="ax-card">La RNA no está entrenada.</div>'))
                return
            x = self.vector()
            info = self.rna.explicar(x)
            p = np.array([info["probabilidades"][c] for c in NOMBRES_CLASES])
            fig, axes = plt.subplots(1, 2, figsize=(9.2, 2.9),
                                     gridspec_kw={"width_ratios": [1, 1.25]})
            colores = ["#54a24b", "#6b5844", "#1d1b1a"]
            axes[0].barh(NOMBRES_CLASES, p, color=colores)
            axes[0].set_xlim(0, 1)
            axes[0].set_title(f"Softmax → {info['prediccion']} "
                              f"(conf. {info['confianza']:.2f})", fontsize=10)
            axes[0].axvline(self.rna_umbral(), ls="--", c="#d62728", lw=1,
                            label="umbral de veto")
            axes[0].legend(fontsize=7, frameon=False)
            axes[0].tick_params(labelsize=8)
            z = np.array([info["normalizado"][n] for n in RASGOS])
            axes[1].bar(range(len(RASGOS)), z, color="#1f6feb")
            axes[1].set_xticks(range(len(RASGOS)))
            axes[1].set_xticklabels(RASGOS, rotation=55, ha="right", fontsize=6.6)
            axes[1].set_title("Vector normalizado (entrada efectiva)", fontsize=10)
            axes[1].set_ylim(-0.3, 1.3)
            axes[1].tick_params(labelsize=7)
            plt.tight_layout()
            plt.show()
            plt.close(fig)

            capas = info.get("capas_ocultas", [])
            html = [_css(), '<div class="ax-wrap"><div class="ax-card">',
                    '<div class="ax-title">PROPAGACIÓN</div>',
                    '<table class="ax-t"><tr><th>Capa</th><th>Neuronas</th>'
                    '<th>Activas (ReLU&gt;0)</th><th>Activación media</th>'
                    '<th>Máx.</th></tr>']
            for i, c in enumerate(capas, start=1):
                html.append(f"<tr><td>oculta {i}</td><td>{c['n_neuronas']}</td>"
                            f"<td>{c['activas']}</td><td>{c['media']:.3f}</td>"
                            f"<td>{c['max']:.3f}</td></tr>")
            html.append("</table>")
            veto = p[2] >= self.rna_umbral()
            html.append(
                f'<div class="ax-small" style="margin-top:6px;">Consecuencia '
                f'operativa: p(barranco)={p[2]:.2f} → '
                f'{_tag("VETO PREVENTIVO + replanificación", "ax-bad") if veto else _tag("avance permitido (sujeto al resto de restricciones)", "ax-ok")}'
                f'. La predicción alimenta el mapa de creencias y el riesgo de ruta '
                f'que evalúa el AG.</div></div></div>')
            display(HTML("".join(html)))

    @staticmethod
    def rna_umbral() -> float:
        return Config().umbral_rna_barranco

    # -- historial de entrenamiento ----------------------------------------
    def _pintar_historial(self) -> None:
        with self.out_hist:
            clear_output(wait=True)
            if self.rna is None or not self.rna.historial["epoca"]:
                display(HTML('<div class="ax-card">Sin historial.</div>'))
                return
            h = self.rna.historial
            i = int(clamp(self.w_epoca.value, 0, len(h["epoca"]) - 1))
            fig, axes = plt.subplots(1, 2, figsize=(9.2, 2.9))
            axes[0].plot(h["epoca"], h["perdida"], color="#d62728")
            axes[0].axvline(h["epoca"][i], ls="--", c="gray", lw=1)
            axes[0].set_title("Pérdida de entrenamiento", fontsize=10)
            axes[0].set_xlabel("época", fontsize=8)
            axes[1].plot(h["epoca"], h["acc_train"], label="exactitud train")
            axes[1].plot(h["epoca"], h["acc_valid"], label="exactitud valid")
            axes[1].plot(h["epoca"], h["recall_barranco_valid"], ls=":",
                         label="recall barranco (valid)")
            axes[1].axvline(h["epoca"][i], ls="--", c="gray", lw=1)
            axes[1].set_ylim(0, 1.02)
            axes[1].legend(fontsize=7, frameon=False)
            axes[1].set_title("Exactitud y recall crítico", fontsize=10)
            axes[1].set_xlabel("época", fontsize=8)
            for a in axes:
                a.tick_params(labelsize=8)
                a.grid(alpha=0.25)
            plt.tight_layout()
            plt.show()
            plt.close(fig)
            display(HTML(
                _css() + f'<div class="ax-wrap"><div class="ax-small">'
                f'Época {h["epoca"][i]}: pérdida {h["perdida"][i]:.4f} · '
                f'train {h["acc_train"][i]:.3f} · valid {h["acc_valid"][i]:.3f} · '
                f'recall barranco {h["recall_barranco_valid"][i]:.3f}. '
                f'La validación usa MAPAS distintos de los de entrenamiento.'
                f'</div></div>'))

    # -- diagnóstico sobre test --------------------------------------------
    def _pintar_diagnostico(self) -> None:
        with self.out_diag:
            clear_output(wait=True)
            if self.rna is None or self.rna.matriz_test is None:
                display(HTML('<div class="ax-card">Sin evaluación de test.</div>'))
                return
            M = self.rna.matriz_test
            fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.2))
            im = axes[0].imshow(M, cmap="Blues")
            for i in range(3):
                for j in range(3):
                    axes[0].text(j, i, str(M[i, j]), ha="center", va="center",
                                 color="white" if M[i, j] > M.max() / 2 else "black",
                                 fontsize=10, fontweight="bold")
            axes[0].set_xticks(range(3))
            axes[0].set_xticklabels(NOMBRES_CLASES, rotation=20, fontsize=8)
            axes[0].set_yticks(range(3))
            axes[0].set_yticklabels(NOMBRES_CLASES, fontsize=8)
            axes[0].set_xlabel("predicho", fontsize=8)
            axes[0].set_ylabel("real", fontsize=8)
            axes[0].set_title("Matriz de confusión (TEST)", fontsize=10)

            if self.ds is not None:
                Xte, yte = self.ds.subset("test")
                probas = self.rna.predecir(Xte)
                ece, xs, accs, cnts = error_calibracion(yte, probas)
                axes[1].plot([0, 1], [0, 1], ls="--", c="gray", lw=1,
                             label="calibración ideal")
                mask = ~np.isnan(accs)
                axes[1].plot(xs[mask], accs[mask], "o-", color="#1f6feb",
                             label=f"observado (ECE={ece:.3f})")
                axes[1].set_xlabel("confianza media", fontsize=8)
                axes[1].set_ylabel("exactitud", fontsize=8)
                axes[1].set_title("Curva de calibración", fontsize=10)
                axes[1].legend(fontsize=7, frameon=False)
                axes[1].grid(alpha=0.25)
            else:
                axes[1].axis("off")
            for a in axes:
                a.tick_params(labelsize=8)
            plt.tight_layout()
            plt.show()
            plt.close(fig)

            m = self.rna.metricas_test
            b = self.rna.baselines
            filas = [("RNA (MLP 32-16)", m["exactitud"], m["macro_f1"],
                      m["recall_barranco"], m["barranco_como_libre"])]
            for k, v in b.items():
                filas.append((f"baseline {k}", v["exactitud"], v["macro_f1"],
                              v["recall_barranco"], v["barranco_como_libre"]))
            html = [_css(), '<div class="ax-wrap"><div class="ax-card">',
                    '<div class="ax-title">MÉTRICAS SOBRE TEST (mapas no vistos)</div>',
                    '<table class="ax-t"><tr><th>Modelo</th><th>Exactitud</th>'
                    '<th>macro-F1</th><th>Recall barranco</th>'
                    '<th>Barranco→libre (FN crítico)</th></tr>']
            for n, a, f1, rb, fn in filas:
                html.append(f"<tr><td>{n}</td><td>{a:.3f}</td><td>{f1:.3f}</td>"
                            f"<td>{rb:.3f}</td><td>{fn}</td></tr>")
            html.append(f'</table><div class="ax-small">ECE = '
                        f'{self.rna.ece_test:.3f} · parámetros entrenables = '
                        f'{self.rna.n_parametros} · partición: '
                        f'{self.rna.dataset_resumen}</div></div></div>')
            display(HTML("".join(html)))

    def vista(self) -> W.Widget:
        izq = W.VBox([self.w_preset] + [self.sliders[n] for n in RASGOS])
        return W.VBox([
            W.HTML(_css() + '<div class="ax-title">LABORATORIO RNA</div>'
                   '<div class="ax-small">Modifique los rasgos de entrada y observe '
                   'cómo cambian las probabilidades y la consecuencia operativa '
                   '(CA-06).</div>'),
            W.HBox([izq, self.out_pred]),
            W.HTML('<hr>'), self.w_epoca, self.out_hist,
            W.HTML('<hr>'), self.out_diag,
        ])


# ============================================================================
class LaboratorioDifuso:
    """Inspección de membresías, reglas, agregación y centroide."""

    def __init__(self, controlador: Optional[FuzzyController] = None):
        self.fc = controlador or FuzzyController()
        est = {"description_width": "140px"}
        lay = W.Layout(width="360px")
        self.w_dist = W.FloatSlider(value=10.0, min=0, max=20, step=0.25,
                                    description="distancia (m)", style=est,
                                    layout=lay, continuous_update=False)
        self.w_bat = W.FloatSlider(value=60.0, min=0, max=100, step=1,
                                   description="batería (%)", style=est, layout=lay,
                                   continuous_update=False)
        self.w_pend = W.FloatSlider(value=8.0, min=0, max=45, step=0.5,
                                    description="pendiente (°)", style=est, layout=lay,
                                    continuous_update=False)
        self.w_mt = W.FloatSlider(value=45.0, min=-20, max=80, step=1,
                                  description="margen térmico (°C)", style=est,
                                  layout=lay, continuous_update=False)
        self.chk_reglas = [W.Checkbox(value=True, description=f"R{r.id:02d}",
                                      indent=False, layout=W.Layout(width="78px"))
                           for r in self.fc.reglas]
        self.w_sup_x = W.Dropdown(options=list(self.fc.entradas), value="distancia_m",
                                  description="eje X:", layout=W.Layout(width="230px"))
        self.w_sup_y = W.Dropdown(options=list(self.fc.entradas), value="bateria_pct",
                                  description="eje Y:", layout=W.Layout(width="230px"))
        self.b_superficie = W.Button(description="Calcular superficie", icon="cube")
        self.out_curvas = W.Output()
        self.out_reglas = W.Output()
        self.out_salida = W.Output()
        self.out_sup = W.Output()

        for w in (self.w_dist, self.w_bat, self.w_pend, self.w_mt):
            w.observe(lambda ch: self.actualizar(), names="value")
        for c in self.chk_reglas:
            c.observe(lambda ch: self.actualizar(), names="value")
        self.b_superficie.on_click(lambda _: self._pintar_superficie())
        self.actualizar()

    def _traza(self):
        habilitadas = {r.id for r, c in zip(self.fc.reglas, self.chk_reglas) if c.value}
        self.fc.reglas_habilitadas = (None if len(habilitadas) == len(self.fc.reglas)
                                      else habilitadas)
        return self.fc.inferir(distancia_m=self.w_dist.value,
                               bateria_pct=self.w_bat.value,
                               pendiente_deg=self.w_pend.value,
                               margen_termico_C=self.w_mt.value)

    def actualizar(self) -> None:
        tz = self._traza()
        self._pintar_curvas(tz)
        self._pintar_reglas(tz)
        self._pintar_salida(tz)

    def _pintar_curvas(self, tz) -> None:
        with self.out_curvas:
            clear_output(wait=True)
            vars_ = list(self.fc.entradas)
            fig, axes = plt.subplots(1, 4, figsize=(13.0, 2.5))
            for ax, nombre in zip(axes, vars_):
                var = self.fc.entradas[nombre]
                u, curvas = var.curvas()
                for k, y in curvas.items():
                    ax.plot(u, y, lw=1.6, label=k)
                valor = tz.entradas[nombre]
                ax.axvline(valor, color="#d62728", lw=1.6)
                for k, g in tz.membresias[nombre].items():
                    if k.startswith("NO ") or g <= 0.01:
                        continue
                    ax.plot([valor], [g], "o", color="#d62728", ms=5)
                    ax.annotate(f"{k} {g:.2f}", (valor, g), fontsize=6.5,
                                xytext=(4, 3), textcoords="offset points")
                ax.set_title(f"{nombre} = {valor:.2f} {var.unidad}", fontsize=9)
                ax.set_ylim(-0.05, 1.12)
                ax.tick_params(labelsize=7)
                ax.legend(fontsize=6.2, frameon=False, loc="upper right")
                ax.grid(alpha=0.2)
            plt.tight_layout()
            plt.show()
            plt.close(fig)

    def _pintar_reglas(self, tz) -> None:
        with self.out_reglas:
            clear_output(wait=True)
            filas = sorted(tz.activaciones, key=lambda a: -a["grado"])
            html = [_css(), '<div class="ax-wrap"><div class="ax-card">',
                    f'<div class="ax-title">BASE DE REGLAS · '
                    f'{tz.n_reglas_activas} activas de {len(self.fc.reglas)}</div>',
                    '<table class="ax-t"><tr><th>Grado</th><th>Regla</th>'
                    '<th>Antecedentes (μ)</th><th>Justificación</th></tr>']
            for a in filas:
                g = a["grado"]
                col = ("ax-ok" if g > 0.5 else ("ax-warn" if g > 0.01 else ""))
                ants = ", ".join(f"{k}={v:.2f}" for k, v in a["antecedentes"].items())
                html.append(f'<tr><td class="ax-v {col}">{g:.3f}</td>'
                            f'<td class="ax-small">{a["texto"]}</td>'
                            f'<td class="ax-small">{ants}</td>'
                            f'<td class="ax-small" style="opacity:.75;">'
                            f'{a["justificacion"]}</td></tr>')
            html.append("</table>")
            if tz.fallback:
                html.append('<div class="ax-small ax-bad">Ninguna regla se activó: '
                            'se aplica la SALIDA DE RESPALDO documentada '
                            '(velocidad 0, cautela 1), es decir, detención '
                            'conservadora en lugar de una velocidad indefinida.</div>')
            html.append("</div></div>")
            display(HTML("".join(html)))

    def _pintar_salida(self, tz) -> None:
        with self.out_salida:
            clear_output(wait=True)
            fig, axes = plt.subplots(1, 2, figsize=(9.2, 2.7))
            for ax, salida in zip(axes, ["velocidad", "cautela"]):
                var = self.fc.salidas[salida]
                u, curvas = var.curvas()
                for k, y in curvas.items():
                    ax.plot(u, y, lw=1.0, ls="--", alpha=0.6, label=k)
                agr = tz.agregados.get(salida)
                if agr is not None:
                    ax.fill_between(u, 0, agr, color="#1f6feb", alpha=0.35,
                                    label="agregado (max-min)")
                cen = tz.salidas[salida]
                ax.axvline(cen, color="#d62728", lw=2.0)
                ax.annotate(f"centroide = {cen:.3f}", (cen, 1.02), fontsize=8,
                            ha="center", color="#d62728")
                ax.set_title(f"Desfusificación de «{salida}»", fontsize=10)
                ax.set_ylim(-0.05, 1.18)
                ax.tick_params(labelsize=7)
                ax.legend(fontsize=6.2, frameon=False, loc="upper left")
                ax.grid(alpha=0.2)
            plt.tight_layout()
            plt.show()
            plt.close(fig)

    def _pintar_superficie(self) -> None:
        with self.out_sup:
            clear_output(wait=True)
            vx, vy = self.w_sup_x.value, self.w_sup_y.value
            if vx == vy:
                display(HTML('<div class="ax-card">Elija dos variables distintas.</div>'))
                return
            fijos = {"distancia_m": self.w_dist.value, "bateria_pct": self.w_bat.value,
                     "pendiente_deg": self.w_pend.value, "margen_termico_C": self.w_mt.value}
            xs, ys, Z = self.fc.superficie(vx, vy, "velocidad", fijos, n=22)
            fig, ax = plt.subplots(figsize=(5.2, 3.6))
            cf = ax.contourf(xs, ys, Z, levels=14, cmap="viridis")
            plt.colorbar(cf, ax=ax, label="velocidad")
            ax.set_xlabel(vx, fontsize=8)
            ax.set_ylabel(vy, fontsize=8)
            ax.set_title("Superficie de control", fontsize=10)
            ax.tick_params(labelsize=7)
            plt.tight_layout()
            plt.show()
            plt.close(fig)

    def vista(self) -> W.Widget:
        return W.VBox([
            W.HTML(_css() + '<div class="ax-title">LABORATORIO DIFUSO</div>'
                   '<div class="ax-small">Cambie la distancia de 10 m a 1 m (CA-05) y '
                   'observe membresías, reglas disparadas y el centroide. Desactive '
                   'reglas para provocar la condición «ninguna regla activa».</div>'),
            W.HBox([W.VBox([self.w_dist, self.w_bat, self.w_pend, self.w_mt]),
                    W.VBox([W.HTML('<div class="ax-small">Reglas habilitadas:</div>'),
                            W.GridBox(self.chk_reglas,
                                      layout=W.Layout(
                                          grid_template_columns="repeat(7, 80px)"))])]),
            self.out_curvas, self.out_salida, self.out_reglas,
            W.HTML("<hr>"),
            W.HBox([self.w_sup_x, self.w_sup_y, self.b_superficie]), self.out_sup,
        ])


# ============================================================================
class LaboratorioGenetico:
    """Inspección de población, operadores y convergencia del AG."""

    def __init__(self, dashboard: Dashboard):
        self.dash = dashboard
        self.w_gen = W.IntSlider(value=0, min=0, max=1, description="Generación:",
                                 continuous_update=False,
                                 layout=W.Layout(width="460px"))
        self.w_evento = W.Dropdown(options=[("(sin eventos)", -1)], value=-1,
                                   description="Operación:",
                                   layout=W.Layout(width="460px"))
        self.pesos = {}
        for k, v in Config().ag_pesos.items():
            self.pesos[k] = W.FloatSlider(value=v, min=0.0, max=1.0, step=0.05,
                                          description=f"w_{k}",
                                          style={"description_width": "95px"},
                                          layout=W.Layout(width="290px"),
                                          continuous_update=False)
        self.b_replan = W.Button(description="Replanificar con estos pesos",
                                 icon="dna", button_style="primary")
        self.out_curva = W.Output()
        self.out_pobla = W.Output()
        self.out_ops = W.Output()
        self.out_rutas = W.Output()

        self.w_gen.observe(lambda ch: self._pintar_poblacion(), names="value")
        self.w_evento.observe(lambda ch: self._pintar_operacion(), names="value")
        self.b_replan.on_click(lambda _: self._replanificar())
        self.actualizar()

    def _traza(self):
        return self.dash.mision.traza_ag if self.dash.mision else None

    def _replanificar(self) -> None:
        m = self.dash.mision
        if m is None:
            return
        pesos = {k: float(w.value) for k, w in self.pesos.items()}
        m.cfg.ag_pesos = pesos
        m.ag.cfg.ag_pesos = pesos
        m._replanificado_este_paso = False
        m.replanificar("Replanificación manual desde el Laboratorio Genético")
        self.actualizar()
        self.dash.actualizar()

    def actualizar(self) -> None:
        tz = self._traza()
        if tz is not None and tz.mejor_por_gen:
            self.w_gen.max = max(0, len(tz.mejor_por_gen) - 1)
            self.w_gen.value = min(self.w_gen.value, self.w_gen.max)
            ops = [(f"gen {e['gen']} · {e['op']}", i)
                   for i, e in enumerate(tz.eventos)]
            self.w_evento.options = ops or [("(sin eventos)", -1)]
            self.w_evento.value = ops[0][1] if ops else -1
        self._pintar_curva()
        self._pintar_poblacion()
        self._pintar_operacion()
        self._pintar_rutas()

    def _pintar_curva(self) -> None:
        with self.out_curva:
            clear_output(wait=True)
            tz = self._traza()
            if tz is None or not tz.mejor_por_gen:
                display(HTML('<div class="ax-card">El AG no se ha ejecutado '
                             '(¿ablación sin AG?).</div>'))
                return
            g = range(len(tz.mejor_por_gen))
            fig, axes = plt.subplots(1, 2, figsize=(9.6, 2.9))
            axes[0].plot(g, tz.mejor_por_gen, label="mejor", lw=2)
            axes[0].plot(g, tz.media_por_gen, label="media", lw=1.2)
            axes[0].plot(g, tz.peor_por_gen, label="peor", lw=0.9, ls=":")
            axes[0].axvline(self.w_gen.value, ls="--", c="gray", lw=1)
            axes[0].set_title("Convergencia del fitness", fontsize=10)
            axes[0].set_xlabel("generación", fontsize=8)
            axes[0].legend(fontsize=7, frameon=False)
            axes[1].plot(g, tz.diversidad_por_gen, color="#9467bd")
            axes[1].set_title("Diversidad (desv. típica del fitness)", fontsize=10)
            axes[1].set_xlabel("generación", fontsize=8)
            for a in axes:
                a.grid(alpha=0.25)
                a.tick_params(labelsize=8)
            plt.tight_layout()
            plt.show()
            plt.close(fig)
            display(HTML(
                _css() + f'<div class="ax-wrap"><div class="ax-small">'
                f'Generaciones ejecutadas: {tz.generaciones_ejecutadas} · '
                f'cruces: {tz.n_cruces} · mutaciones: {tz.n_mutaciones} · '
                f'parada temprana: {"SÍ" if tz.parada_temprana else "no"} · '
                f'tiempo: {tz.segundos:.3f} s</div></div>'))

    def _pintar_poblacion(self) -> None:
        with self.out_pobla:
            clear_output(wait=True)
            tz = self._traza()
            if tz is None or not tz.snapshots:
                return
            i = int(clamp(self.w_gen.value, 0, len(tz.snapshots) - 1))
            snap = tz.snapshots[i]
            html = [_css(), '<div class="ax-wrap"><div class="ax-card">',
                    f'<div class="ax-title">GENERACIÓN {i} · MEJORES CROMOSOMAS '
                    f'(CA-04)</div>',
                    '<table class="ax-t"><tr><th>#</th><th>Fitness</th>'
                    '<th>Ciencia</th><th>Dist.</th><th>Energía</th><th>Riesgo</th>'
                    '<th>Tiempo</th><th>Penal.</th><th>Factible</th>'
                    '<th>Genes</th></tr>']
            for k, r in enumerate(snap):
                c = r.componentes
                html.append(
                    f'<tr><td>{k+1}</td><td class="ax-v">{r.fitness:+.4f}</td>'
                    f'<td>{c.get("aporte_ciencia",0):+.3f}</td>'
                    f'<td>{c.get("aporte_distancia",0):+.3f}</td>'
                    f'<td>{c.get("aporte_energia",0):+.3f}</td>'
                    f'<td>{c.get("aporte_riesgo",0):+.3f}</td>'
                    f'<td>{c.get("aporte_tiempo",0):+.3f}</td>'
                    f'<td>−{c.get("penalizacion",0):.3f}</td>'
                    f'<td>{"sí" if r.factible else "NO"}</td>'
                    f'<td class="ax-small"><code>{r.cromosoma.genes_texto()[:84]}'
                    f'</code></td></tr>')
            html.append('</table><div class="ax-small">El fitness es la suma de los '
                        'aportes menos la penalización: '
                        'fitness = w₁·ciencia − w₂·distancia − w₃·energía − '
                        'w₄·riesgo − w₅·tiempo − P(inviabilidad).</div>')
            if tz.torneos:
                t = [x for x in tz.torneos if x["gen"] == i][:3]
                if t:
                    html.append('<div class="ax-small" style="margin-top:6px;">'
                                '<b>Torneos de esta generación:</b> ' +
                                " · ".join(
                                    f"{[c[0] for c in x['t1']['competidores']]}"
                                    f"→gana {x['t1']['ganador']}" for x in t) +
                                "</div>")
            html.append("</div></div>")
            display(HTML("".join(html)))

    def _pintar_operacion(self) -> None:
        with self.out_ops:
            clear_output(wait=True)
            tz = self._traza()
            if tz is None or not tz.eventos or self.w_evento.value is None \
                    or self.w_evento.value < 0:
                return
            e = tz.eventos[int(self.w_evento.value)]
            if e["op"] == "cruce":
                cuerpo = (f'<div class="ax-small"><b>Cruce</b> ({e.get("corte","")}) '
                          f'en la generación {e["gen"]}:<br>'
                          f'padre&nbsp; <code>{e["padre"]}</code><br>'
                          f'madre&nbsp; <code>{e["madre"]}</code><br>'
                          f'<b>hijo</b>&nbsp;&nbsp; <code>{e["hijo"]}</code></div>')
            else:
                cuerpo = (f'<div class="ax-small"><b>Mutación</b> en la generación '
                          f'{e["gen"]} · operadores: {", ".join(e["operadores"])}<br>'
                          f'antes&nbsp;&nbsp; <code>{e["antes"]}</code><br>'
                          f'después <code>{e["despues"]}</code></div>')
            display(HTML(_css() + '<div class="ax-wrap"><div class="ax-card">'
                         '<div class="ax-title">OPERACIÓN GENÉTICA</div>' +
                         cuerpo + "</div></div>"))

    def _pintar_rutas(self) -> None:
        with self.out_rutas:
            clear_output(wait=True)
            m = self.dash.mision
            if m is None or not m.candidatas:
                return
            fig, ax = plt.subplots(figsize=(5.6, 5.2))
            self.dash.dibujar_mapa(ax, mostrar_candidatas=True,
                                   mostrar_creencia=True)
            ax.set_title("Rutas candidatas sobre el MAPA DE RIESGO ESTIMADO",
                         fontsize=9.5, fontweight="bold")
            plt.tight_layout()
            plt.show()
            plt.close(fig)
            html = [_css(), '<div class="ax-wrap"><div class="ax-card">',
                    '<div class="ax-title">RUTAS CANDIDATAS INSPECCIONABLES '
                    '(RF-07)</div>', '<table class="ax-t"><tr><th>Ruta</th>'
                    '<th>Fitness</th><th>Celdas</th><th>Ciencia</th><th>Riesgo máx</th>'
                    '<th>Energía est.</th><th>Masa</th><th>Potencia</th>'
                    '<th>Estado</th><th>Motivos</th></tr>']
            for i, c in enumerate(m.candidatas, start=1):
                html.append(
                    f'<tr><td>{i}{" (elegida)" if c is m.plan else ""}</td>'
                    f'<td class="ax-v">{c.fitness:+.4f}</td><td>{len(c.camino)}</td>'
                    f'<td>{c.componentes.get("ciencia",0):.3f}</td>'
                    f'<td>{c.riesgo_max:.2f}</td><td>{c.energia_estimada:.1f} %</td>'
                    f'<td>{c.masa_usada:.1f}/{m.cfg.masa_max_kg:.0f} kg</td>'
                    f'<td>{c.potencia_usada:.0f}/{m.cfg.potencia_max_W:.0f} W</td>'
                    f'<td>{"factible" if c.factible else "INVIABLE"}</td>'
                    f'<td class="ax-small">{", ".join(c.motivos[:2]) or "—"}</td></tr>')
            html.append('</table><div class="ax-small">La masa y la potencia '
                        'corresponden al subproblema de MOCHILA (selección de '
                        'instrumentos); el orden de visitas, al de VIAJERO.</div>'
                        "</div></div>")
            display(HTML("".join(html)))

    def vista(self) -> W.Widget:
        return W.VBox([
            W.HTML(_css() + '<div class="ax-title">LABORATORIO GENÉTICO</div>'
                   '<div class="ax-small">Seleccione una generación para ver sus '
                   'mejores cromosomas y una operación de cruce o mutación (CA-04). '
                   'Cambie los pesos del fitness y vuelva a planificar para comprobar '
                   'que el AG influye causalmente en la ruta.</div>'),
            self.w_gen, self.out_curva, self.out_pobla,
            self.w_evento, self.out_ops,
            W.HTML("<hr>"),
            W.HBox([W.VBox(list(self.pesos.values())), self.b_replan]),
            self.out_rutas,
        ])


# ============================================================================
class ComparadorPoliticas:
    """Comparación de políticas con igual semilla (RF-14, RF-15, CA-08)."""

    def __init__(self, rna, cfg_base: Optional[Config] = None):
        self.rna = rna
        self.cfg = cfg_base or Config(bateria_inicial=60, max_pasos=180,
                                      ag_poblacion=28, ag_generaciones=22)
        self.w_semillas = W.IntSlider(value=20, min=3, max=40,
                                      description="Semillas:",
                                      layout=W.Layout(width="330px"),
                                      continuous_update=False)
        self.w_dificultad = W.Dropdown(options=["facil", "media", "dificil"],
                                       value="media", description="Dificultad:",
                                       layout=W.Layout(width="230px"))
        self.w_politicas = W.SelectMultiple(
            options=list(POLITICAS), value=("ruta_corta", "sin_rna", "sin_ag",
                                            "hibrido"),
            description="Políticas:", rows=6, layout=W.Layout(width="330px"))
        self.b_correr = W.Button(description="Ejecutar comparación", icon="play",
                                 button_style="primary")
        self.b_causal = W.Button(description="Pruebas de causalidad", icon="check")
        self.out = W.Output()
        self.df: Optional[pd.DataFrame] = None
        self.b_correr.on_click(lambda _: self._correr())
        self.b_causal.on_click(lambda _: self._causalidad())

    def _correr(self) -> None:
        with self.out:
            clear_output(wait=True)
            print("Ejecutando… (cada política usa las MISMAS semillas y mapas)")
            cfg = Config.from_dict({**self.cfg.__dict__,
                                    "dificultad": self.w_dificultad.value})
            df = ejecutar_experimento([cfg], list(self.w_politicas.value),
                                      int(self.w_semillas.value), self.rna,
                                      verbose=False)
            self.df = df
            clear_output(wait=True)
            tabla = tabla_comparativa(df)
            display(HTML(_css() + '<div class="ax-wrap"><div class="ax-card">'
                         '<div class="ax-title">COMPARACIÓN DE POLÍTICAS</div>' +
                         tabla.to_html(index=False, classes="ax-t", border=0) +
                         "</div></div>"))
            comp = comparar_con_baseline(df, "ciencia_pct", "hibrido")
            display(HTML(_css() + '<div class="ax-wrap"><div class="ax-card">'
                         '<div class="ax-title">DIFERENCIA FRENTE AL HÍBRIDO '
                         '(IC 95 %)</div>' +
                         comp.to_html(index=False, classes="ax-t", border=0) +
                         '<div class="ax-small">La columna '
                         '<code>significativo_95</code> marca las diferencias cuyo '
                         'intervalo de confianza no incluye el cero.</div>'
                         "</div></div>"))
            self._graficar(df)

    def _graficar(self, df: pd.DataFrame) -> None:
        g = df.groupby("politica")
        pols = [p for p in POLITICAS if p in g.groups]
        fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.1))
        datos = [
            ("Éxito de misión (%)", [100 * g.get_group(p)["exito"].mean()
                                     for p in pols], None),
            ("Ciencia recolectada (%)", [g.get_group(p)["ciencia_pct"].mean()
                                         for p in pols],
             [g.get_group(p)["ciencia_pct"].std(ddof=1) for p in pols]),
            ("Violaciones de seguridad", [g.get_group(p)["violaciones_seguridad"].mean()
                                          for p in pols],
             [g.get_group(p)["violaciones_seguridad"].std(ddof=1) for p in pols]),
        ]
        for ax, (titulo, valores, err) in zip(axes, datos):
            colores = ["#1f6feb" if p == "hibrido" else "#9aa0a6" for p in pols]
            ax.bar(pols, valores, yerr=err, capsize=3, color=colores)
            ax.set_title(titulo, fontsize=10)
            ax.tick_params(axis="x", rotation=35, labelsize=7.5)
            ax.tick_params(axis="y", labelsize=8)
            ax.grid(alpha=0.25, axis="y")
        plt.tight_layout()
        plt.show()
        plt.close(fig)

    def _causalidad(self) -> None:
        with self.out:
            clear_output(wait=True)
            print("Ejecutando pruebas controladas…")
            df, detalle = pruebas_causalidad(self.rna)
            clear_output(wait=True)
            display(HTML(_css() + '<div class="ax-wrap"><div class="ax-card">'
                         '<div class="ax-title">PRUEBAS DE CAUSALIDAD</div>'
                         '<div class="ax-small">Cada prueba modifica UNA salida y '
                         'mantiene el resto constante. Demuestra que no existe una '
                         '«función secreta» que resuelva la misión ignorando las '
                         'tres técnicas.</div>' +
                         df.to_html(index=False, classes="ax-t", border=0) +
                         "</div></div>"))
            for p in detalle:
                display(HTML(_css() + f'<div class="ax-wrap"><div class="ax-card">'
                             f'<b class="ax-small">{p["tecnica"]}</b>'
                             f'<pre class="ax-small">'
                             f'{pd.DataFrame(p["resultado"]).to_string()}</pre>'
                             f"</div></div>"))

    def vista(self) -> W.Widget:
        return W.VBox([
            W.HTML(_css() + '<div class="ax-title">COMPARACIÓN DE POLÍTICAS</div>'
                   '<div class="ax-small">Ejecuta el sistema completo frente a los '
                   'baselines y las ablaciones sobre las mismas semillas y mapas.'
                   '</div>'),
            W.HBox([W.VBox([self.w_semillas, self.w_dificultad]),
                    self.w_politicas,
                    W.VBox([self.b_correr, self.b_causal])]),
            self.out,
        ])


# ============================================================================
def centro_de_control(cfg: Config, rna, dataset=None) -> W.Widget:
    """Arma la interfaz completa con las vistas principal y secundarias."""
    dash = Dashboard(cfg, rna)
    lab_rna = LaboratorioRNA(rna, dataset)
    lab_fuz = LaboratorioDifuso()
    lab_gen = LaboratorioGenetico(dash)
    comp = ComparadorPoliticas(rna, cfg)

    tabs = W.Tab(children=[dash.panel_control(), lab_rna.vista(), lab_fuz.vista(),
                           lab_gen.vista(), comp.vista()])
    for i, t in enumerate(["Control de misión", "Laboratorio RNA",
                           "Laboratorio Difuso", "Laboratorio Genético",
                           "Comparación de políticas"]):
        tabs.set_title(i, t)

    # El laboratorio genético se refresca cuando avanza la misión
    _avanzar = dash.avanzar

    def avanzar_y_refrescar(n=1):
        _avanzar(n)
        lab_gen.actualizar()
    dash.avanzar = avanzar_y_refrescar
    dash.b_paso._click_handlers.callbacks.clear()
    dash.b_correr._click_handlers.callbacks.clear()
    dash.b_paso.on_click(lambda _: avanzar_y_refrescar(1))
    dash.b_correr.on_click(lambda _: avanzar_y_refrescar(int(dash.w_n_pasos.value)))

    cabecera = W.HTML(_css() + f"""
    <div class="ax-wrap"><div class="ax-card">
      <div class="ax-title">ARES-X · CENTRO DE CONTROL DE MISIÓN</div>
      <div class="ax-small">Rover autónomo de exploración planetaria ·
      RNA + lógica difusa + algoritmos genéticos · versión {Config().__class__ and ''}
      simulador v1.0.0<br>
      <b>Simulación ficticia</b> con parámetros astronómicos de referencia. No
      representa una misión real ni certifica condiciones operacionales de
      ingeniería espacial. El detector de biofirmas sólo emite señales anómalas:
      nunca confirma vida.</div>
    </div></div>""")
    return W.VBox([cabecera, tabs]), dash, lab_rna, lab_fuz, lab_gen, comp

## 12 · Pruebas obligatorias

Las 16 condiciones de borde de la sección 12 del enunciado y los 8 casos de aceptación de la
sección 13, como verificaciones automáticas con evidencia textual. Ninguna prueba debe provocar
una excepción no controlada: eso forma parte del criterio.

In [ ]:
"""
ARES-X · Módulo 12: Pruebas obligatorias.

  * 16 condiciones de borde de la sección 12 del caso.
  * 8 casos de aceptación CA-01 .. CA-08 de la sección 13.

Cada prueba devuelve (id, descripcion, resultado, evidencia). Ninguna debe
provocar una excepción no controlada: ése es, en sí mismo, parte del criterio.
"""

from __future__ import annotations

import math
import traceback
from typing import Callable, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd


Resultado = Tuple[str, str, bool, str]


def _cfg(**kw) -> Config:
    base = dict(semilla=42, bateria_inicial=60, max_pasos=120,
                ag_poblacion=20, ag_generaciones=15, eventos_aleatorios=False)
    base.update(kw)
    return Config(**base)


# ============================================================================
# CONDICIONES DE BORDE (sección 12)
# ============================================================================
def cb01_bateria_cero(rna) -> Resultado:
    m = Mission(_cfg(bateria_inicial=0.5), rna)
    m.estado.bateria = 0.0
    # Lejos de cualquier refugio: sin recarga posible
    m.world.refugios = [(m.world.filas - 1, m.world.cols - 1)]
    pos0 = m.estado.pos
    m.paso()
    ok = (m.estado.pos == pos0 and m.estado.bateria >= 0.0 and
          m.estado.estado in ("FRACASO", "MODO_SEGURO", "ALERTA"))
    return ("CB-01", "Batería = 0 %: no mover, fin o modo seguro, nunca negativa",
            ok, f"estado={m.estado.estado}, batería={m.estado.bateria:.2f} %, "
                f"posición sin cambios={m.estado.pos == pos0}")


def cb02_bateria_1_refugio_lejano(rna) -> Resultado:
    m = Mission(_cfg(bateria_inicial=1.0, filas=30, columnas=30, n_refugios=1), rna)
    # Alejar artificialmente el refugio
    m.world.refugios = [(m.world.filas - 1, m.world.cols - 1)]
    m.estado.pos = (0, 0)
    m.estado.bateria = 1.0
    m.belief.confirmar((0, 0), LIBRE)
    r = m.paso()
    alcanzable = m._refugio_alcanzable()
    ok = (not alcanzable) and r["motivo"] in ("REFUGIO_INALCANZABLE", "FIN_FRACASO",
                                              "VETO_ENERGIA")
    return ("CB-02", "Batería 1 % y refugio lejano: declarar inalcanzable",
            ok, f"refugio_alcanzable={alcanzable}, motivo emitido='{r['motivo']}' "
                f"(no se promete un rescate imposible)")


def cb03_sensor_invalido(rna) -> Resultado:
    fc = FuzzyController()
    salidas = {}
    for etiqueta, v in (("cero", 0.0), ("nan", float("nan")),
                        ("inf", float("inf"))):
        tz = fc.inferir(distancia_m=v, bateria_pct=70, pendiente_deg=5,
                        margen_termico_C=50)
        salidas[etiqueta] = round(tz.salidas["velocidad"], 3)
    ok = all(es_valido(v) and v < 0.25 for v in salidas.values())
    return ("CB-03", "Distancia = 0 / NaN / infinito: validación y prioridad al frenado",
            ok, f"velocidad resultante {salidas} (todas finitas y conservadoras)")


def cb04_todos_sensores_fallan(rna) -> Resultado:
    m = Mission(_cfg(), rna)
    for s in ("proximidad", "lidar", "inclinometro"):
        m.sensores.forzar_fallo(s, "nan")
    pos0 = m.estado.pos
    r = m.paso()
    ok = (r["motivo"] == "VETO_SENSOR" and m.estado.pos == pos0 and
          m.estado.estado == "MODO_SEGURO")
    return ("CB-04", "Todos los sensores de navegación fallan: no continuar a ciegas",
            ok, f"estado={m.estado.estado}, motivo={r['motivo']}, "
                f"degradación registrada en bitácora")


def cb05_barranco_bajo_siguiente_casilla(rna) -> Resultado:
    m = Mission(_cfg(), rna)
    for _ in range(4):
        m.paso()
    destino = m._destino_actual() or m.celda_frente()
    m.colocar_barranco(destino)
    pos0 = m.estado.pos
    motivos = []
    for _ in range(3):
        r = m.paso()
        motivos.append(r["motivo"])
        if r["motivo"] in ("VETO_BARRANCO", "VETO_PENDIENTE", "VETO_RNA_BARRANCO"):
            break
    entro = m.estado.pos == destino
    ok = (not entro) and any(x in motivos for x in
                             ("VETO_BARRANCO", "VETO_PENDIENTE", "VETO_RNA_BARRANCO"))
    return ("CB-05", "Barranco bajo la siguiente casilla: detener antes de entrar",
            ok, f"motivos={motivos}, ¿entró al precipicio?={entro}, "
                f"zona prohibida creada={bool(m.belief.prohibido[destino])}")


def cb06_sin_ruta(rna) -> Resultado:
    m = Mission(_cfg(filas=12, columnas=12), rna)
    # Encerrar el rover con obstáculos CONFIRMADOS (la relajación de zonas
    # prohibidas no puede liberarlos: son peligros verificados, no sospechas).
    for df_ in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if df_ or dc:
                q = (m.estado.pos[0] + df_, m.estado.pos[1] + dc)
                if m.belief.dentro(q):
                    m.world.clase[q[0], q[1]] = OBSTACULO
                    m.belief.confirmar(q, OBSTACULO)
                    m.belief.prohibir(q)
    sin_excepcion = True
    try:
        for _ in range(20):
            if m.terminada:
                break
            m.paso()
    except Exception:
        sin_excepcion = False
    ok = sin_excepcion and (m.terminada or m._pasos_sin_progreso > 0)
    return ("CB-06", "No existe ruta al objetivo: reportar sin solución, sin excepción",
            ok, f"terminada={m.terminada}, motivo='{m.motivo_fin[:70]}', "
                f"sin excepción={sin_excepcion}")


def cb07_mutacion_invalida(rna) -> Resultado:
    cfg = _cfg()
    w = World(cfg)
    b = BeliefMap(w.filas, w.cols)
    gp = GeneticPlanner(cfg)
    # Cromosoma con waypoints fuera de rango y orden corrupto
    crom = Cromosoma(orden=list(range(len(w.sitios))), n_visitas=len(w.sitios),
                     instrumentos=[1] * 8,
                     waypoints=[(-5, -9), (999, 999), (w.filas + 3, -2)])
    sin_excepcion = True
    try:
        ruta = gp.evaluar(crom, b, w.inicio, w.sitios, w.refugios, 60.0)
        finito = math.isfinite(ruta.fitness)
    except Exception:
        sin_excepcion, finito = False, False
        ruta = None
    ok = sin_excepcion and finito
    return ("CB-07", "Mutación produce coordenada inválida: reparar o penalizar",
            ok, f"sin excepción={sin_excepcion}, fitness finito={finito}, "
                f"fitness={ruta.fitness:.3f} y penalización registrada"
                if ruta else "excepción")


def cb08_ag_sin_mejora(rna) -> Resultado:
    cfg = _cfg(ag_poblacion=16, ag_generaciones=60)
    w = World(cfg)
    b = BeliefMap(w.filas, w.cols)
    gp = GeneticPlanner(cfg)
    mejor, _, tz = gp.planificar(b, w.inicio, w.sitios, w.refugios, 60.0,
                                 semilla=1, paciencia=10)
    ok = (tz.generaciones_ejecutadas <= cfg.ag_generaciones and
          math.isfinite(mejor.fitness))
    return ("CB-08", "AG sin mejora en 10 generaciones: parada temprana opcional",
            ok, f"generaciones ejecutadas={tz.generaciones_ejecutadas}/"
                f"{cfg.ag_generaciones}, parada temprana={tz.parada_temprana}, "
                f"mejor factible conservado (fitness {mejor.fitness:+.3f})")


def cb09_rna_baja_confianza(rna) -> Resultado:
    m = Mission(_cfg(), rna)
    m.paso()
    destino = m._destino_actual() or m.celda_frente()
    lectura = m.observar()
    d_alta = m.supervisor.evaluar(
        estado=m.estado, lectura=lectura, belief=m.belief, destino=destino,
        velocidad_difusa=0.8, cautela=0.2, prob_rna=np.array([0.95, 0.03, 0.02]),
        confirmacion_borde=LIBRE, refugio_alcanzable=True, hay_sitio_aqui=False,
        ruta_factible=True, pendiente_destino=4.0)
    d_baja = m.supervisor.evaluar(
        estado=m.estado, lectura=lectura, belief=m.belief, destino=destino,
        velocidad_difusa=0.8, cautela=0.2, prob_rna=np.array([0.40, 0.33, 0.27]),
        confirmacion_borde=LIBRE, refugio_alcanzable=True, hay_sitio_aqui=False,
        ruta_factible=True, pendiente_destino=4.0)
    ok = d_baja.velocidad < d_alta.velocidad
    return ("CB-09", "RNA con baja confianza: fusión con sensor y modo conservador",
            ok, f"confianza 0.95 → velocidad {d_alta.velocidad:.3f}; "
                f"confianza 0.40 → velocidad {d_baja.velocidad:.3f} "
                f"({', '.join(d_baja.permitidos) or 'sin marcas'})")


def cb10_difuso_sin_reglas(rna) -> Resultado:
    fc = FuzzyController()
    fc.reglas_habilitadas = set()          # ninguna regla habilitada
    tz = fc.inferir(distancia_m=15, bateria_pct=80, pendiente_deg=5,
                    margen_termico_C=50)
    ok = (tz.n_reglas_activas == 0 and tz.fallback and
          tz.salidas["velocidad"] == 0.0 and tz.salidas["cautela"] == 1.0)
    fc.reglas_habilitadas = None
    return ("CB-10", "Reglas difusas sin activación: salida de respaldo documentada",
            ok, f"reglas activas={tz.n_reglas_activas}, fallback={tz.fallback}, "
                f"velocidad={tz.salidas['velocidad']:.2f}, "
                f"cautela={tz.salidas['cautela']:.2f}")


def cb11_temperatura_fuera_de_tolerancia(rna) -> Resultado:
    m = Mission(_cfg(), rna)
    lo, hi = m.cfg.perfil.limite_termico_C
    m.estado.temp_interna = hi + 40.0
    pos0 = m.estado.pos
    r = m.paso()
    ok = (r["motivo"] == "VETO_TERMICO" and m.estado.pos == pos0)
    return ("CB-11", "Temperatura fuera de tolerancia: abortar o aislar",
            ok, f"T interna={hi + 40:.0f} °C vs límite {hi:.0f} °C → "
                f"motivo='{r['motivo']}', estado={m.estado.estado}")


def cb12_rover_en_jupiter(rna) -> Resultado:
    cfg_ok, errores = Config(planeta="jupiter", modo="rover").validar()
    excepcion = False
    try:
        Mission(Config(planeta="jupiter", modo="rover"), rna)
    except ValueError:
        excepcion = True
    sonda_ok, _ = Config(planeta="jupiter", modo="sonda").validar()
    ok = (not cfg_ok) and excepcion and sonda_ok
    return ("CB-12", "Modo rover en Júpiter: la interfaz impide la simulación de rodaje",
            ok, f"validación rover={cfg_ok} (rechazada), Mission lanza ValueError="
                f"{excepcion}, modo sonda aceptado={sonda_ok}. "
                f"Motivo: {errores[0][:110] if errores else ''}")


def cb13_eventos_simultaneos(rna) -> Resultado:
    m = Mission(_cfg(bateria_inicial=9.0), rna)
    m.inyectar_evento("meteorito")
    m.inyectar_evento("sismo")
    sin_excepcion = True
    motivos = []
    try:
        for _ in range(3):
            motivos.append(m.paso()["motivo"])
    except Exception:
        sin_excepcion = False
    # La prioridad de seguridad debe resolverse de forma determinista
    m2 = Mission(_cfg(bateria_inicial=9.0), rna)
    m2.inyectar_evento("meteorito")
    m2.inyectar_evento("sismo")
    motivos2 = [m2.paso()["motivo"] for _ in range(3)]
    ok = sin_excepcion and motivos == motivos2
    return ("CB-13", "Eventos simultáneos (meteorito + batería baja): prioridad "
                     "determinista",
            ok, f"motivos corrida 1={motivos}; corrida 2={motivos2}; "
                f"idénticos={motivos == motivos2}")


def cb14_semilla_repetida(rna) -> Resultado:
    r = verificar_determinismo(_cfg(), rna, semilla=77)
    return ("CB-14", "Misma semilla y configuración: traza equivalente",
            bool(r["determinista"]),
            f"pasos {r['corrida_1']['pasos']} vs {r['corrida_2']['pasos']}; "
            f"ciencia {r['corrida_1']['ciencia_recolectada']} vs "
            f"{r['corrida_2']['ciencia_recolectada']}; estado "
            f"{r['corrida_1']['estado_final']} vs {r['corrida_2']['estado_final']}")


def cb15_config_imposible(rna) -> Resultado:
    casos = {
        "batería NaN": Config(bateria_inicial=float("nan")),
        "mapa 3x3": Config(filas=3, columnas=3),
        "batería 150 %": Config(bateria_inicial=150.0),
        "planeta inexistente": Config(planeta="planeta_x"),
        "población AG = 1": Config(ag_poblacion=1),
    }
    detalles = []
    todas_rechazadas = True
    for nombre, c in casos.items():
        ok_c, errs = c.validar()
        todas_rechazadas &= (not ok_c)
        detalles.append(f"{nombre} → {'RECHAZADA' if not ok_c else 'ACEPTADA'}"
                        f" ({errs[0][:48] if errs else ''})")
    return ("CB-15", "Configuración imposible o NaN: rechazo con mensaje legible",
            todas_rechazadas, " | ".join(detalles))


def cb16_mapa_sin_ciencia(rna) -> Resultado:
    sin_excepcion = True
    try:
        m = Mission(_cfg(n_sitios_ciencia=0, max_pasos=40), rna)
        r = m.ejecutar()
        division = es_valido(r["ciencia_pct"])
    except Exception:
        sin_excepcion, division, r = False, False, {}
    ok = sin_excepcion and division
    return ("CB-16", "Mapa sin sitios científicos: misión de solo retorno, sin "
                     "división por cero",
            ok, f"sin excepción={sin_excepcion}, ciencia_pct="
                f"{r.get('ciencia_pct')} (finito), estado={r.get('estado_final')}")


CONDICIONES_BORDE: List[Callable] = [
    cb01_bateria_cero, cb02_bateria_1_refugio_lejano, cb03_sensor_invalido,
    cb04_todos_sensores_fallan, cb05_barranco_bajo_siguiente_casilla, cb06_sin_ruta,
    cb07_mutacion_invalida, cb08_ag_sin_mejora, cb09_rna_baja_confianza,
    cb10_difuso_sin_reglas, cb11_temperatura_fuera_de_tolerancia,
    cb12_rover_en_jupiter, cb13_eventos_simultaneos, cb14_semilla_repetida,
    cb15_config_imposible, cb16_mapa_sin_ciencia,
]


# ============================================================================
# CASOS DE ACEPTACIÓN (sección 13)
# ============================================================================
def ca01_navegacion_normal(rna) -> Resultado:
    m = Mission(_cfg(planeta="marte", semilla=42), rna)
    for _ in range(14):
        if m.terminada:
            break
        m.paso()
    df = m.logger.df()
    ok = (len(df) >= 10 and len(m.ruta_ejecutada) > 1 and
          "bateria_pct" in df.columns and df["explicacion"].notna().all())
    return ("CA-01", "Navegación normal: ≥10 pasos con telemetría e historial "
                     "consistentes",
            ok, f"{len(df)} pasos registrados, {len(df.columns)} columnas por paso, "
                f"{len(m.ruta_ejecutada)} celdas recorridas")


def ca02_obstaculo_imprevisto(rna) -> Resultado:
    res = cb05_barranco_bajo_siguiente_casilla(rna)
    m = Mission(_cfg(), rna)
    for _ in range(4):
        m.paso()
    destino = m._destino_actual() or m.celda_frente()
    m.colocar_barranco(destino)
    replan_antes = m.metricas["replanificaciones"]
    for _ in range(3):
        m.paso()
    tiene_rna = m.prob_rna is not None
    tiene_difuso = m.traza_difusa is not None and m.traza_difusa.n_reglas_activas >= 0
    replanifico = m.metricas["replanificaciones"] > replan_antes
    ok = res[2] and tiene_rna and tiene_difuso and replanifico
    return ("CA-02", "Obstáculo imprevisto: frenado, explicación de las 3 técnicas y "
                     "replanificación",
            ok, f"detenido antes del peligro={res[2]}, confianza RNA disponible="
                f"{tiene_rna}, reglas difusas evaluadas={tiene_difuso}, "
                f"replanificó={replanifico}")


def ca03_energia_critica(rna) -> Resultado:
    # De noche no hay captación solar: con 12 % la energía es estrictamente
    # decreciente y el refugio pasa a ser la prioridad real.
    m = Mission(_cfg(bateria_inicial=12.0, n_refugios=1, hora_inicial=22.0), rna)
    for _ in range(25):
        if m.terminada:
            break
        m.paso()
    df = m.logger.df()
    motivos = set(df["motivo"]) if len(df) else set()
    refugios = set(m.world.refugios)
    paso_refugio = next((i for i, p in enumerate(m.ruta_ejecutada)
                         if tuple(p) in refugios), None)
    alcanzo_refugio = paso_refugio is not None
    declaro_imposible = bool({"REFUGIO_INALCANZABLE", "VETO_ENERGIA"} & motivos)
    # Contraste: con batería holgada el rover NO debería correr al refugio igual
    m2 = Mission(_cfg(bateria_inicial=85.0, n_refugios=1, hora_inicial=22.0), rna)
    for _ in range(25):
        if m2.terminada:
            break
        m2.paso()
    ref2 = set(m2.world.refugios)
    paso_refugio2 = next((i for i, p in enumerate(m2.ruta_ejecutada)
                          if tuple(p) in ref2), None)
    ok = alcanzo_refugio or declaro_imposible
    return ("CA-03", "Batería inicial 12 %: priorizar refugio alcanzable o explicar "
                     "que no existe alternativa",
            ok,
            f"con 12 % alcanzó el refugio en la celda {paso_refugio} de su ruta "
            f"(batería {12.0:.0f} % → {m.estado.bateria:.1f} %); con 85 % lo alcanzó "
            f"en la celda {paso_refugio2}; motivos={sorted(motivos)[:5]}; "
            f"declaración de inalcanzable={declaro_imposible}")


def ca04_diagnostico_genetico(rna) -> Resultado:
    m = Mission(_cfg(ag_generaciones=25, ag_poblacion=24), rna)
    m.paso()
    tz = m.traza_ag
    gen = min(5, len(tz.snapshots) - 1) if tz and tz.snapshots else -1
    ok = False
    detalle = "AG no ejecutado"
    if tz and gen >= 0:
        snap = tz.snapshots[gen]
        hay_op = len(tz.eventos) > 0
        ok = len(snap) >= 3 and hay_op
        detalle = (f"generación {gen}: {len(snap)} cromosomas con fitness "
                   f"{[round(r.fitness, 3) for r in snap]}; "
                   f"{tz.n_cruces} cruces y {tz.n_mutaciones} mutaciones "
                   f"inspeccionables ({len(tz.eventos)} registradas en detalle)")
    return ("CA-04", "Generación 5: ≥3 cromosomas, su fitness y una operación de "
                     "cruce o mutación", ok, detalle)


def ca05_diagnostico_difuso(rna) -> Resultado:
    fc = FuzzyController()
    a = fc.inferir(distancia_m=10, bateria_pct=60, pendiente_deg=8,
                   margen_termico_C=50)
    b = fc.inferir(distancia_m=1, bateria_pct=60, pendiente_deg=8,
                   margen_termico_C=50)
    ok = (a.salidas["velocidad"] > b.salidas["velocidad"] + 0.1 and
          a.n_reglas_activas > 0 and b.n_reglas_activas > 0)
    return ("CA-05", "Distancia 10 m → 1 m: cambian curvas, reglas y velocidad",
            ok, f"10 m → velocidad {a.salidas['velocidad']:.3f} "
                f"({a.n_reglas_activas} reglas: {a.resumen_texto(2)}); "
                f"1 m → velocidad {b.salidas['velocidad']:.3f} "
                f"({b.n_reglas_activas} reglas: {b.resumen_texto(2)})")


def ca06_diagnostico_neuronal(rna) -> Resultado:
    if rna is None or not rna.entrenado:
        return ("CA-06", "Cambiar rasgos actualiza probabilidades; test y recall "
                         "consultables", False, "RNA no entrenada")
    libre = np.array([18.5, 0.2, 6.0, 1.2, 0.3, 0.4, 0.55, 0.3, 0.95, 0.5])
    barranco = np.array([19.2, -8.5, 58.0, 9.5, 0.5, 0.45, 0.2, -4.0, 0.9, 2.1])
    p1 = rna.predecir_una(libre)[2]
    p2 = rna.predecir_una(barranco)[2]
    ok = (int(np.argmax(p1)) == LIBRE and int(np.argmax(p2)) == BARRANCO and
          "recall_barranco" in rna.metricas_test)
    return ("CA-06", "Modificar rasgos actualiza probabilidades; test y recall "
                     "consultables",
            ok, f"rasgos de celda libre → p={np.round(p1, 3).tolist()}; rasgos de "
                f"borde de barranco → p={np.round(p2, 3).tolist()}; "
                f"recall_barranco(test)={rna.metricas_test['recall_barranco']:.3f}, "
                f"macro-F1={rna.metricas_test['macro_f1']:.3f}")


def ca07_planeta_invalido(rna) -> Resultado:
    return ("CA-07",) + cb12_rover_en_jupiter(rna)[1:]


def ca08_experimento_controlado(rna) -> Resultado:
    cfg = _cfg(max_pasos=90, ag_poblacion=18, ag_generaciones=12)
    r_full = ejecutar_episodio(cfg, "hibrido", 55, rna)
    r_abl = ejecutar_episodio(cfg, "sin_rna", 55, rna)
    m = Mission(Config.from_dict({**cfg.__dict__, "semilla": 55}), rna)
    m.ejecutar(60)
    m.resultados()
    csv = m.logger.exportar_csv("/tmp/ares_x_prueba.csv")
    js = m.logger.exportar_json("/tmp/ares_x_prueba.json")
    import os
    ok = (os.path.getsize(csv) > 500 and os.path.getsize(js) > 500 and
          r_full["semilla"] == r_abl["semilla"])
    return ("CA-08", "Misma semilla: comparar sistema completo con una técnica "
                     "desactivada y exportar",
            ok, f"semilla 55 — híbrido: ciencia {r_full['ciencia_pct']} %, "
                f"colisiones {r_full['colisiones']}; sin RNA: ciencia "
                f"{r_abl['ciencia_pct']} %, colisiones {r_abl['colisiones']}. "
                f"CSV {os.path.getsize(csv)} B y JSON {os.path.getsize(js)} B "
                f"generados")


CASOS_ACEPTACION: List[Callable] = [
    ca01_navegacion_normal, ca02_obstaculo_imprevisto, ca03_energia_critica,
    ca04_diagnostico_genetico, ca05_diagnostico_difuso, ca06_diagnostico_neuronal,
    ca07_planeta_invalido, ca08_experimento_controlado,
]


# ============================================================================
def ejecutar_pruebas(rna, cuales: str = "todas", verbose: bool = True
                     ) -> pd.DataFrame:
    """Ejecuta la batería y devuelve una tabla de resultados."""
    funciones: List[Callable] = []
    if cuales in ("todas", "borde"):
        funciones += CONDICIONES_BORDE
    if cuales in ("todas", "aceptacion"):
        funciones += CASOS_ACEPTACION

    filas = []
    for f in funciones:
        try:
            idp, desc, ok, evid = f(rna)
        except Exception as exc:               # ninguna prueba debe reventar
            idp = getattr(f, "__name__", "?").split("_")[0].upper()
            desc = f.__name__
            ok = False
            evid = f"EXCEPCIÓN: {type(exc).__name__}: {exc}"
            if verbose:
                traceback.print_exc()
        filas.append({"id": idp, "prueba": desc,
                      "resultado": "PASA" if ok else "FALLA", "evidencia": evid})
        if verbose:
            marca = "✔" if ok else "✘"
            print(f"  {marca} {idp}  {desc[:62]}")
    df = pd.DataFrame(filas)
    if verbose:
        n_ok = int((df["resultado"] == "PASA").sum())
        print(f"\n  Resultado: {n_ok}/{len(df)} pruebas superadas.")
    return df

---
# Parte II · Ejecución

## 12.1 · Entrenamiento de la red neuronal

Se generan los 24 mapas independientes, se verifica la ausencia de fuga entre particiones y se
entrena el MLP. En Colab gratuito con CPU esto toma unos 10–20 segundos.

In [ ]:
import time
_t0 = time.time()

CFG_BASE = Config(planeta="marte", modo="rover", dificultad="media",
                  semilla=42, bateria_inicial=60.0, filas=22, columnas=22,
                  n_sitios_ciencia=6, max_pasos=240)

print("Generando dataset sintético (24 mapas independientes)…")
DATASET = generar_dataset(CFG_BASE, n_mapas=24, muestras_por_mapa=260, semilla=20260)
print(f"  {len(DATASET.X)} muestras × {DATASET.X.shape[1]} rasgos\n")

import pandas as pd
display(pd.DataFrame(DATASET.resumen()).T)

fuga = verificar_sin_fuga(DATASET)
print(f"\n¿Sin fuga entre particiones?  {fuga['sin_fuga']}")
for p, mapas in fuga["mapas_por_particion"].items():
    print(f"  {p:<6} mapas {mapas}")

print("\nEntrenando MLP (32, 16) en CPU…")
RNA = NeuralPerception(capas=(32, 16), semilla=42, epocas=140).entrenar(DATASET)
print(f"\nTiempo total de preparación: {time.time() - _t0:.1f} s")

### Lectura de las métricas

Las métricas se calculan sobre **mapas que la red nunca vio**. Conviene mirar en este orden:

* **recall de barrancos**: es el error que mata al rover. Un falso negativo barranco→libre es
  cualitativamente peor que confundir libre con obstáculo.
* **macro-F1** frente a los dos baselines: si la red no supera claramente a las reglas fijas de
  umbral, no está aportando nada que no se consiga con dos `if`.
* **ECE**: el supervisor usa la confianza para activar el modo conservador. Una red
  sobreconfiada haría que esa salvaguarda nunca se dispare.

In [ ]:
import pandas as pd
import numpy as np

m, b = RNA.metricas_test, RNA.baselines
tabla = pd.DataFrame([
    {"modelo": "RNA (MLP 32-16)", **{k: m[k] for k in
     ["exactitud", "macro_f1", "recall_barranco", "recall_obstaculo", "recall_libre"]},
     "barranco→libre": m["barranco_como_libre"]},
    {"modelo": "baseline mayoritario", **{k: b["mayoritario"][k] for k in
     ["exactitud", "macro_f1", "recall_barranco", "recall_obstaculo", "recall_libre"]},
     "barranco→libre": b["mayoritario"]["barranco_como_libre"]},
    {"modelo": "baseline reglas fijas", **{k: b["reglas_fijas"][k] for k in
     ["exactitud", "macro_f1", "recall_barranco", "recall_obstaculo", "recall_libre"]},
     "barranco→libre": b["reglas_fijas"]["barranco_como_libre"]},
]).round(3)
display(tabla)
print(f"Error de calibración esperado (ECE) sobre test: {RNA.ece_test:.4f}")
print(f"Parámetros entrenables: {RNA.n_parametros}")
print("\nMatriz de confusión (filas = real, columnas = predicho):")
display(pd.DataFrame(RNA.matriz_test, index=NOMBRES_CLASES, columns=NOMBRES_CLASES))

## 12.2 · Una misión completa, paso a paso

Antes de abrir la interfaz, una ejecución no interactiva para ver la bitácora en crudo.

In [ ]:
mision_demo = Mission(CFG_BASE, RNA)
print("Mundo generado:", mision_demo.world.resumen())
print("Inicio:", mision_demo.estado.pos, "· refugios:", mision_demo.world.refugios)
print("Sitios científicos:", [(s.id, s.pos, round(s.valor, 2), s.requiere)
                              for s in mision_demo.world.sitios])
print()

resultado_demo = mision_demo.ejecutar()
for k, v in resultado_demo.items():
    print(f"  {k:<26} {v}")

In [ ]:
# Bitácora: las columnas que exige RF-08, paso a paso
df_demo = mision_demo.logger.df()
cols = [c for c in ["paso", "estado_mision", "accion", "motivo", "velocidad",
                    "bateria_pct", "rna_prediccion", "rna_confianza",
                    "rna_p_barranco", "difuso_velocidad", "difuso_cautela",
                    "difuso_reglas_activas", "ag_fitness", "ag_ciencia",
                    "reserva_requerida_pct", "eventos"] if c in df_demo.columns]
print(f"{len(df_demo)} pasos × {len(df_demo.columns)} columnas registradas")
display(df_demo[cols].head(18))
print("\nEventos de la misión:")
display(mision_demo.logger.df_eventos().head(12))

In [ ]:
# Mapa final: ruta planificada vs. ruta ejecutada (RF-10)
import matplotlib.pyplot as plt
_dash_demo = Dashboard(CFG_BASE, RNA)
_dash_demo.mision = mision_demo
fig, ax = plt.subplots(figsize=(7.2, 6.6))
_dash_demo.dibujar_mapa(ax)
plt.tight_layout(); plt.show()

---
## 13 · Centro de Control de Misión (interfaz interactiva)

Ejecute la celda siguiente para abrir la interfaz completa. Las cinco pestañas son:

| Pestaña | Qué permite hacer |
|---|---|
| **Control de misión** | Elegir planeta, modo, dificultad, semilla, batería y prioridad científica; iniciar, pausar, avanzar paso a paso, reiniciar; inyectar sismo, meteorito, tormenta, noche, fallo de sensor, pérdida de enlace, señal anómala, contacto ficticio o ráfaga; colocar un barranco frente al rover; exportar CSV y JSON |
| **Laboratorio RNA** | Mover los diez rasgos de entrada y ver probabilidades, activaciones por capa y la consecuencia operativa; recorrer el historial de entrenamiento por época; revisar matriz de confusión y calibración |
| **Laboratorio Difuso** | Cambiar las cuatro entradas y ver membresías, reglas disparadas con su grado, agregación y centroide; desactivar reglas para provocar la condición «ninguna regla activa»; calcular superficies de control |
| **Laboratorio Genético** | Elegir generación y ver sus mejores cromosomas con el fitness desglosado; inspeccionar un cruce o una mutación concretos; cambiar los pesos del fitness y replanificar |
| **Comparación de políticas** | Ejecutar el híbrido frente a baselines y ablaciones con las mismas semillas; lanzar las pruebas de causalidad |

**Sugerencia de recorrido para la defensa oral (6–8 min)**

1. *Control de misión* → «Correr» 25 pasos y comentar los paneles B y C.
2. «Barranco al frente» → un paso → mostrar en el Panel C cómo la RNA sube `p(barranco)`,
   el difuso baja la velocidad y el supervisor emite `VETO_BARRANCO` y replanifica.
3. *Laboratorio Difuso* → distancia de 10 m a 1 m (CA-05).
4. *Laboratorio Genético* → generación 5, tres cromosomas y una mutación (CA-04).
5. *Laboratorio RNA* → preset «borde de barranco» (CA-06).
6. Planeta **Júpiter** → el modo rover queda bloqueado con explicación científica (CA-07).

In [ ]:
INTERFAZ, DASH, LAB_RNA, LAB_DIFUSO, LAB_AG, COMPARADOR = centro_de_control(
    CFG_BASE, RNA, DATASET)
display(INTERFAZ)

---
## 14 · Pruebas obligatorias

Las 16 condiciones de borde de la sección 12 del enunciado y los 8 casos de aceptación de la
sección 13. Cada fila incluye la evidencia concreta que respalda el resultado.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 190)

print("CONDICIONES DE BORDE")
tabla_borde = ejecutar_pruebas(RNA, cuales="borde", verbose=True)
print("\nCASOS DE ACEPTACIÓN")
tabla_aceptacion = ejecutar_pruebas(RNA, cuales="aceptacion", verbose=True)

TABLA_PRUEBAS = pd.concat([tabla_borde, tabla_aceptacion], ignore_index=True)
n_ok = int((TABLA_PRUEBAS["resultado"] == "PASA").sum())
print(f"\nTOTAL: {n_ok}/{len(TABLA_PRUEBAS)} pruebas superadas")
display(TABLA_PRUEBAS)

---
## 15 · Pruebas de causalidad

El enunciado impone una **condición de auditoría**: la política final no puede depender de una
«función secreta» que resuelva el problema ignorando la RNA, el controlador difuso o el AG; hay
que demostrar causalmente que alterar una salida cambia la decisión o el movimiento.

Cada prueba modifica **una sola** salida y mantiene todo lo demás constante.

In [ ]:
TABLA_CAUSAL, DETALLE_CAUSAL = pruebas_causalidad(RNA, semilla=42)
display(TABLA_CAUSAL)

for p in DETALLE_CAUSAL:
    print(f"\n{'='*78}\n{p['tecnica'].upper()}  —  variable controlada: {p['variable_controlada']}")
    print(f"{'='*78}")
    display(pd.DataFrame(p["resultado"]))
    print(p["explicacion"])

In [ ]:
# Determinismo (RF-12): misma semilla y configuración -> traza equivalente
det = verificar_determinismo(CFG_BASE, RNA, semilla=42)
print("¿Determinista?", det["determinista"])
display(pd.DataFrame([det["corrida_1"], det["corrida_2"]],
                     index=["corrida 1", "corrida 2"])[
    ["pasos", "estado_final", "ciencia_recolectada", "distancia_m",
     "colisiones", "replanificaciones"]])

---
## 16 · Experimento con 20 semillas, baselines y ablaciones

RF-14 y RF-15: **20 semillas independientes por escenario**, seis políticas y **condiciones
idénticas** entre políticas. Sin condiciones idénticas no se puede atribuir ninguna mejora a la
arquitectura.

Dos escenarios: Marte dificultad media con 60 % de batería y Marte dificultad difícil con 45 %.
En Colab gratuito esto toma del orden de 5–8 minutos. Reduzca `N_SEMILLAS` si desea una
exploración rápida, pero **el informe debe reportar 20**.

In [ ]:
N_SEMILLAS = 20          # RF-15 exige >= 20

ESCENARIOS = [
    Config.from_dict({**CFG_BASE.__dict__, "dificultad": "media",
                      "bateria_inicial": 60.0, "max_pasos": 300,
                      "ag_poblacion": 28, "ag_generaciones": 22}),
    Config.from_dict({**CFG_BASE.__dict__, "dificultad": "dificil",
                      "bateria_inicial": 45.0, "max_pasos": 300,
                      "ag_poblacion": 28, "ag_generaciones": 22}),
]

print(f"Ejecutando {len(ESCENARIOS)} escenarios × {len(POLITICAS)} políticas × "
      f"{N_SEMILLAS} semillas = {len(ESCENARIOS)*len(POLITICAS)*N_SEMILLAS} episodios\n")
for k, v in POLITICAS.items():
    print(f"  {k:<12} {v['_desc']}")
print()

DF_EXPERIMENTO = ejecutar_experimento(ESCENARIOS, list(POLITICAS), N_SEMILLAS,
                                      RNA, verbose=True)

In [ ]:
TABLA_COMPARATIVA = tabla_comparativa(DF_EXPERIMENTO)
display(TABLA_COMPARATIVA)

print("\nMétricas con media, mediana y dispersión (RF-15):")
display(resumen_metricas(DF_EXPERIMENTO,
                         ["exito", "ciencia_pct", "colisiones",
                          "violaciones_seguridad", "bateria_final",
                          "replanificaciones"]).round(3))

In [ ]:
# Diferencias frente al sistema completo, con intervalo de confianza al 95 %
for metrica in ["colisiones", "ciencia_pct", "replanificaciones"]:
    print(f"\n--- {metrica} ---")
    display(comparar_con_baseline(DF_EXPERIMENTO, metrica, "hibrido"))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

pols = [p for p in POLITICAS if p in set(DF_EXPERIMENTO["politica"])]
escenarios = sorted(DF_EXPERIMENTO["escenario"].unique())
metricas_graf = [("exito", "Éxito de misión", 100.0),
                 ("ciencia_pct", "Ciencia recolectada (%)", 1.0),
                 ("colisiones", "Colisiones por misión", 1.0),
                 ("violaciones_seguridad", "Violaciones de seguridad", 1.0)]

fig, axes = plt.subplots(len(escenarios), len(metricas_graf),
                         figsize=(15, 3.4 * len(escenarios)))
axes = np.atleast_2d(axes)
for i, esc in enumerate(escenarios):
    sub = DF_EXPERIMENTO[DF_EXPERIMENTO["escenario"] == esc]
    for j, (col, titulo, factor) in enumerate(metricas_graf):
        ax = axes[i, j]
        medias = [sub[sub.politica == p][col].mean() * factor for p in pols]
        errs = [sub[sub.politica == p][col].std(ddof=1) / np.sqrt(N_SEMILLAS) * 1.96 * factor
                for p in pols]
        colores = ["#1f6feb" if p == "hibrido" else "#9aa0a6" for p in pols]
        ax.bar(pols, medias, yerr=errs, capsize=3, color=colores)
        ax.set_title(f"{titulo}\n{esc}", fontsize=9)
        ax.tick_params(axis="x", rotation=38, labelsize=7.5)
        ax.tick_params(axis="y", labelsize=8)
plt.tight_layout(); plt.show()
print("Barras: media sobre 20 semillas. Error: intervalo de confianza al 95 %.")
print("La política completa aparece en azul.")

---
## 17 · Exportación de trazas y configuración (RF-12)

In [ ]:
mision_demo.resultados()
ruta_csv = mision_demo.logger.exportar_csv("ares_x_traza.csv")
ruta_json = mision_demo.logger.exportar_json(
    "ares_x_mision.json",
    extra={"metricas_rna": RNA.metricas_test,
           "baselines_rna": {k: {kk: vv for kk, vv in v.items()
                                 if not kk.startswith("_")}
                             for k, v in RNA.baselines.items()},
           "resumen_mundo": mision_demo.world.resumen(),
           "particion_dataset": RNA.dataset_resumen})

DF_EXPERIMENTO.to_csv("ares_x_experimento_20_semillas.csv", index=False)
TABLA_COMPARATIVA.to_csv("ares_x_tabla_comparativa.csv", index=False)
TABLA_PRUEBAS.to_csv("ares_x_pruebas.csv", index=False)

import os
for f in ["ares_x_traza.csv", "ares_x_mision.json",
          "ares_x_experimento_20_semillas.csv", "ares_x_tabla_comparativa.csv",
          "ares_x_pruebas.csv"]:
    print(f"  {f:<42} {os.path.getsize(f):>9,} bytes")

print("\nEn Colab, descárguelos con:")
print("  from google.colab import files; files.download('ares_x_mision.json')")

---
## 18 · Conclusiones, límites y supuestos

### Qué quedó demostrado

* **Las tres técnicas se ejecutan de verdad y están conectadas.** La RNA se entrena sobre datos
  sintéticos con partición por mapas y alimenta el mapa de creencias; ese mapa es el que evalúa
  el AG; el AG entrega la ruta; el controlador difuso fija la velocidad; el supervisor arbitra.
  Las pruebas de la sección 15 muestran, con todo lo demás constante, que alterar cada salida
  cambia la decisión.
* **El supervisor prima sobre el fitness.** Con la misma recomendación de IA, confirmar un
  precipicio con el sensor de borde convierte un `AVANZAR` en `REPLANIFICAR`.
* **Reproducibilidad.** Misma semilla y configuración producen la misma traza, también entre
  sesiones distintas.

### Límites honestos de este trabajo

* Los parámetros operacionales son **hipótesis pedagógicas**. Nada aquí es una especificación
  de ingeniería y nada debe atribuirse a la NASA salvo los valores marcados `[NASA]`.
* El mundo es una cuadrícula 2D con celdas de 5 m y ocho direcciones de movimiento. No hay
  dinámica de ruedas, suspensión, tracción diferencial ni mecánica de contacto.
* El modo sonda es una **analogía** sobre la misma estructura de datos (filas = capas de
  altitud), no un modelo de dinámica atmosférica.
* La RNA clasifica celdas a partir de rasgos resumidos, no de imágenes reales. Su desempeño
  depende del generador de datos: es un ejercicio de percepción simulada, no de visión por
  computadora.
* El decodificador del AG es voraz y subóptimo por diseño; una ruta "mejor" podría existir y no
  ser encontrada.
* La comparación entre políticas se hace sobre dos escenarios de Marte. Generalizar a otros
  planetas o dificultades exigiría repetir el experimento.
* El detector de biofirmas **sólo emite señales anómalas**. Ninguna salida de este sistema
  constituye evidencia de vida, y los eventos de contacto son ficción narrativa sin valor
  científico.

### Registro de supuestos

La celda siguiente imprime el registro completo de supuestos de simulación y las fuentes NASA
utilizadas como referencia de contraste.

In [ ]:
print("SUPUESTOS DE SIMULACIÓN (hipótesis pedagógicas ajustables)")
print("=" * 78)
for k, v in SUPUESTOS_SIMULACION.items():
    print(f"  {k:<22} {v}")

print("\n\nVALORES DE REFERENCIA POR PLANETA  [NASA]")
print("=" * 78)
import pandas as pd
display(pd.DataFrame([{
    "planeta": p.nombre, "tipo": p.tipo, "¿rover?": "sí" if p.permite_rover else "NO",
    "T media °C": p.temp_media_C, "presión bar": p.presion_bar,
    "gravedad m/s²": p.gravedad_ms2, "irradiancia (×Tierra)": p.irradiancia_rel,
    "límite térmico HW °C [SUPUESTO]": f"{p.limite_termico_C[0]:.0f} a {p.limite_termico_C[1]:.0f}",
} for p in PERFILES.values()]))

print("\nFUENTES NASA DE CONTRASTE")
for f in FUENTES_NASA:
    print("  " + f)
print(f"\nVersión del simulador: {VERSION} ({FECHA_BUILD})")